# Weekend Shape Baseline Improvement

This is the single working notebook for improving the frozen weekend-shape baseline. Keep future residual diagnostics, predictor screens, category shrinkage checks, interval work, and promotion decisions here rather than creating new notebooks.

The original baseline-creation notebook remains separate: `weekend_shape_top_down_disaggregation.ipynb`.

Immediate conclusion from the continuous predictor screens: **do not promote the current continuous residual models into the weekend-shape point forecast yet.**

The after-Saturday screen showed that almost all apparent point-forecast gain came from a simple intercept/residual-bias correction. The best available continuous model barely improved over `intercept_only`, and no tested variable cleared the 5% Sunday MAE promotion hurdle.

This notebook follows the improvement sequence:

1. freeze the current five-year calibrated log-ratio baseline as the benchmark;
2. create a diagnostic residual table with `e_a`, `e_b`, `r_sun_sat`, and absolute residuals;
3. aggregate residuals weekly before ACF/Ljung-Box checks;
4. test category-aware shrinkage corrections before richer continuous variables;
5. test group-specific residual distributions for intervals;
6. promote nothing unless it beats both the frozen baseline and intercept-only in rolling-origin validation.

The shape target remains log-ratio residual correction, not independent Friday/Saturday/Sunday dollar models:

\\[
e_a=\\log(Fri/Sat)-\\widehat{\\log(Fri/Sat)}_0,
\\quad
e_b=\\log(Sun/Sat)-\\widehat{\\log(Sun/Sat)}_0
\\]

and corrected shape predictions convert back to coherent shares.

Current modelling direction:

The modelling question is no longer which external predictor explains weekend shape. The working decomposition is now:

\[
\text{weekend-shape residual}=\text{regime effect}+\text{calendar/corridor effect}+\text{movie-mix effect}+\text{residual noise}.
\]

The unresolved decision is narrow: should the correction layer use all prior history, 2022+ only, or a hybrid shrinkage approach? The notebook now prioritizes regime/training-window diagnostics before further model testing:

1. compare 2015-2019 versus 2022+ corridor residuals;
2. inspect corridor-year persistence with sample-size overlays;
3. diagnose whether corridor effects survive movie-type conditioning;
4. quantify release-composition shifts by corridor and regime;
5. compare all-prior, last-5-year, 2022+-only, and hybrid correction estimates;
6. evaluate operational point performance separately for `e_b` and `r_sun_sat`;
7. treat ACF/Ljung-Box as a residual diagnostic, not a standalone promotion rule.

Current prior before this diagnostic pass: `r_sun_sat` is closest to production readiness, `e_b` likely needs hybrid shrinkage, and `e_a` should not be a production point correction until it improves operational dollar MAE.


In [ ]:
import os
import tempfile
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "pm-box-office-matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(tempfile.gettempdir(), "pm-box-office-cache"))

repo_root = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
DIAGNOSTICS_DIR = repo_root / "data" / "diagnostics"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from scipy.stats import chi2
from IPython.display import display

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

EXCLUDED_RELEASE_YEARS = {2020, 2021}
MIN_TRAIN_MOVIES = 50
MIN_GROUP_N = 20
DOC_CONCERT_MIN_N = 15
ROLLING_START_YEAR = 2014
SHARE_COLS = ["s_fri", "s_sat", "s_sun"]
PRED_SHARE_COLS = ["pred_friday_share", "pred_saturday_share", "pred_sunday_share"]
TARGETS = ["e_a", "e_b", "r_sun_sat"]

RESIDUAL_DIAGNOSTIC_CACHE_PATH = DIAGNOSTICS_DIR / "weekend_shape_residual_diagnostic_table.csv"
USE_RESIDUAL_DIAGNOSTIC_CACHE = os.environ.get("PM_WEEKEND_SHAPE_USE_CACHE", "1") != "0"
FORCE_REBUILD_RESIDUAL_DIAGNOSTIC = os.environ.get("PM_WEEKEND_SHAPE_FORCE_REBUILD", "0") == "1"

np.random.seed(17)

## Load Inputs and Prior Screen Conclusion

In [ ]:
shape_base_path = DIAGNOSTICS_DIR / "weekend_shape_base.csv"
after_sat_screen_summary_path = DIAGNOSTICS_DIR / "after_saturday_sunday_hold_predictor_screening_summary.csv"

if not shape_base_path.exists():
    raise FileNotFoundError(f"Missing {shape_base_path}. Run weekend_shape_top_down_disaggregation.ipynb first.")

shape = pd.read_csv(shape_base_path)
for col in ["opening_date", "opening_weekend_start"]:
    shape[col] = pd.to_datetime(shape[col], errors="coerce")

numeric_cols = [
    "release_year", "release_month", "release_week_of_year", "opening_weekend_gross_usd",
    "friday_gross_usd", "saturday_gross_usd", "sunday_gross_usd", "opening_weekend_theaters",
    "latest_estimate_mid_usd", "estimate_count", "estimate_source_count", "rt_fresh_share",
    "rt_prerelease_review_count", "wiki_views_cume_to_m1", "log_fri_sat", "log_sun_sat",
]
for col in numeric_cols:
    if col in shape.columns:
        shape[col] = pd.to_numeric(shape[col], errors="coerce")

bool_cols = [
    "is_franchise", "is_large_release", "is_action", "is_horror", "is_family_animation",
    "is_doc_concert", "is_holiday_corridor", "is_very_large_estimate", "is_fan_driven",
    "is_holiday_family", "is_other_wide",
]
for col in bool_cols:
    if col in shape.columns:
        shape[col] = shape[col].fillna(False).astype(bool)

shape = shape.loc[
    shape["opening_weekend_gross_usd"].gt(0)
    & shape["friday_gross_usd"].gt(0)
    & shape["saturday_gross_usd"].gt(0)
    & shape["sunday_gross_usd"].gt(0)
    & ~shape["release_year"].isin(EXCLUDED_RELEASE_YEARS)
].copy()
shape["release_year"] = shape["release_year"].astype(int)
shape = shape.sort_values("opening_weekend_start").reset_index(drop=True)

if "segment_current_broad" not in shape.columns:
    shape["segment_current_broad"] = shape["broad_segment"]
if "segment_ip_doc" not in shape.columns:
    shape["segment_ip_doc"] = np.select(
        [shape["is_doc_concert"], shape["is_franchise"]],
        ["doc/concert", "franchise/IP"],
        default="non-franchise commercial wide",
    )

if after_sat_screen_summary_path.exists():
    prior_screen = pd.read_csv(after_sat_screen_summary_path)
    display(prior_screen[[
        "residual_model", "baseline_log_mae", "corrected_log_mae", "log_mae_improvement_pct",
        "baseline_sunday_mae_usd", "new_sunday_mae_usd", "sunday_mae_improvement_pct",
        "passes_5pct_hurdle",
    ]].head(10))
else:
    prior_screen = pd.DataFrame()
    print("Prior after-Saturday continuous screen summary not found; continuing with this notebook's diagnostics.")

load_summary = pd.DataFrame({
    "rows": [len(shape)],
    "first_opening": [shape["opening_weekend_start"].min()],
    "last_opening": [shape["opening_weekend_start"].max()],
    "first_year": [shape["release_year"].min()],
    "last_year": [shape["release_year"].max()],
})
display(load_summary)

## Frozen Baseline Functions

This recreates the frozen shape policy for diagnostic residuals: last-five-theatrical-year broad direct median plus grid-calibrated log-ratio intercepts. The code is copied in compact form from the baseline notebook so this notebook can be rerun from `weekend_shape_base.csv`.

In [ ]:
def normalize_profile(values):
    values = np.asarray(values, dtype=float)
    total = np.nansum(values)
    if not np.isfinite(total) or total <= 0:
        return np.full_like(values, np.nan, dtype=float)
    return values / total


def shares_from_ab(a, b):
    exp_a = np.exp(a)
    exp_b = np.exp(b)
    denom = 1.0 + exp_a + exp_b
    return pd.Series({
        "pred_friday_share": exp_a / denom,
        "pred_saturday_share": 1.0 / denom,
        "pred_sunday_share": exp_b / denom,
    })


def direct_median_profile(df):
    return pd.Series(dict(zip(PRED_SHARE_COLS, normalize_profile(df[SHARE_COLS].median().values))))


def predicted_logratios(pred):
    return pd.DataFrame({
        "pred_log_fri_sat": np.log(pred["pred_friday_share"] / pred["pred_saturday_share"]),
        "pred_log_sun_sat": np.log(pred["pred_sunday_share"] / pred["pred_saturday_share"]),
    }, index=pred.index)


def add_shape_errors(actual, pred, model):
    out = actual.reset_index(drop=True).copy()
    pred = pred.reset_index(drop=True).copy()
    out[PRED_SHARE_COLS] = pred[PRED_SHARE_COLS]
    for col in pred.columns:
        if col not in out.columns:
            out[col] = pred[col]
    out["model"] = model
    out["pred_log_fri_sat"] = np.log(out["pred_friday_share"] / out["pred_saturday_share"])
    out["pred_log_sun_sat"] = np.log(out["pred_sunday_share"] / out["pred_saturday_share"])
    out["e_a"] = out["log_fri_sat"] - out["pred_log_fri_sat"]
    out["e_b"] = out["log_sun_sat"] - out["pred_log_sun_sat"]
    out["pred_friday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["pred_friday_share"]
    out["pred_saturday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["pred_saturday_share"]
    out["pred_sunday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["pred_sunday_share"]
    out["friday_dollar_error_shape_only"] = out["friday_gross_usd"] - out["pred_friday_gross_shape_only"]
    out["saturday_dollar_error_shape_only"] = out["saturday_gross_usd"] - out["pred_saturday_gross_shape_only"]
    out["sunday_dollar_error_shape_only"] = out["sunday_gross_usd"] - out["pred_sunday_gross_shape_only"]
    out["avg_abs_daily_dollar_error_shape_only"] = out[[
        "friday_dollar_error_shape_only", "saturday_dollar_error_shape_only", "sunday_dollar_error_shape_only",
    ]].abs().mean(axis=1)
    return out


def last_theatrical_years(train, forecast_year, n_years=5):
    years = sorted(
        y for y in train["release_year"].dropna().astype(int).unique()
        if y < forecast_year and y not in EXCLUDED_RELEASE_YEARS
    )
    return years[-n_years:]


def segment_profile_lookup(train, segment_col="segment_current_broad"):
    profiles = {}
    for group, gdf in train.groupby(segment_col, dropna=False, observed=False):
        min_n = DOC_CONCERT_MIN_N if group == "doc/concert" else MIN_GROUP_N
        if len(gdf) >= min_n:
            profiles[group] = direct_median_profile(gdf).to_dict()
    return profiles


def predict_weekend_shape_prior_v1(test, train, forecast_year):
    years = last_theatrical_years(train, forecast_year, n_years=5)
    recent = train.loc[train["release_year"].isin(years)].copy()
    if len(recent) < MIN_TRAIN_MOVIES:
        recent = train.copy()
    global_profile = direct_median_profile(recent).to_dict()
    profiles = segment_profile_lookup(recent, "segment_current_broad")
    return pd.DataFrame(
        [profiles.get(row["segment_current_broad"], global_profile) for _, row in test.iterrows()],
        index=test.index,
    )


def apply_logratio_constants(pred, c_a, c_b):
    ratios = predicted_logratios(pred)
    calibrated = pd.DataFrame(index=pred.index)
    calibrated["calibrated_pred_log_fri_sat"] = ratios["pred_log_fri_sat"] + c_a
    calibrated["calibrated_pred_log_sun_sat"] = ratios["pred_log_sun_sat"] + c_b
    shares = calibrated.apply(lambda row: shares_from_ab(row["calibrated_pred_log_fri_sat"], row["calibrated_pred_log_sun_sat"]), axis=1)
    calibrated[PRED_SHARE_COLS] = shares[PRED_SHARE_COLS]
    calibrated["calibration_c_a"] = c_a
    calibrated["calibration_c_b"] = c_b
    return calibrated


def prior_oos_shape_predictions(train):
    frames = []
    for validation_year in sorted(train["release_year"].dropna().astype(int).unique()):
        inner_train = train.loc[train["release_year"] < validation_year].copy()
        inner_test = train.loc[train["release_year"].eq(validation_year)].copy()
        if len(inner_train) < MIN_TRAIN_MOVIES or inner_test.empty:
            continue
        pred = predict_weekend_shape_prior_v1(inner_test, inner_train, validation_year)
        out = add_shape_errors(inner_test, pred, "WeekendShapePrior_v1_raw")
        out["validation_year"] = validation_year
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def evaluate_calibrated_prior_oos(prior_oos, c_a, c_b):
    pred = prior_oos[PRED_SHARE_COLS].copy()
    calibrated = apply_logratio_constants(pred, c_a, c_b)
    eval_df = add_shape_errors(prior_oos, calibrated, "calibration_grid")
    return pd.concat([
        eval_df["friday_dollar_error_shape_only"].abs(),
        eval_df["saturday_dollar_error_shape_only"].abs(),
        eval_df["sunday_dollar_error_shape_only"].abs(),
    ], ignore_index=True).mean()


def tune_shape_logratio_grid(train):
    prior_oos = prior_oos_shape_predictions(train)
    if prior_oos.empty:
        return {"calibration_c_a": 0.0, "calibration_c_b": 0.0, "calibration_objective_value": np.nan, "calibration_residual_n": 0}
    c_a_grid = np.round(np.arange(-0.05, 0.3001, 0.025), 3)
    c_b_grid = np.round(np.arange(-0.05, 0.1501, 0.025), 3)
    best = {"calibration_objective_value": np.inf}
    for c_a in c_a_grid:
        for c_b in c_b_grid:
            score = evaluate_calibrated_prior_oos(prior_oos, c_a, c_b)
            if score < best["calibration_objective_value"]:
                best = {"calibration_c_a": c_a, "calibration_c_b": c_b, "calibration_objective_value": score, "calibration_residual_n": len(prior_oos)}
    return best


def rolling_shape_prior_v1(df, start_year=ROLLING_START_YEAR):
    frames = []
    for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= start_year):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        base_pred = predict_weekend_shape_prior_v1(test, train, test_year)
        calibration = tune_shape_logratio_grid(train)
        pred = apply_logratio_constants(base_pred, calibration["calibration_c_a"], calibration["calibration_c_b"])
        pred["calibration_objective_value"] = calibration["calibration_objective_value"]
        pred["calibration_residual_n"] = calibration["calibration_residual_n"]
        out = add_shape_errors(test, pred, "WeekendShapePrior_v1")
        out["fold"] = f"train_before_{test_year}__test_{test_year}"
        out["train_n"] = len(train)
        out["test_year"] = test_year
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## Build Diagnostic Residual Table

The after-Saturday residual uses the clean target `r_sun_sat = log(Sun_actual / Sun_afterSat_base)`.

In [ ]:
def median_logratio_by_segment(test, train, numerator_col, denominator_col, segment_col="segment_ip_doc"):
    train = train.copy()
    ratio = np.log(train[numerator_col] / train[denominator_col]).replace([np.inf, -np.inf], np.nan)
    global_value = ratio.dropna().median()
    lookup = {}
    for group, idx in train.groupby(segment_col, dropna=False, observed=False).groups.items():
        values = ratio.loc[idx].dropna()
        min_n = DOC_CONCERT_MIN_N if group == "doc/concert" else MIN_GROUP_N
        if len(values) >= min_n:
            lookup[group] = values.median()
    return test[segment_col].map(lookup).fillna(global_value)


def predict_after_saturday_base(test, train):
    sun_log = median_logratio_by_segment(test, train, "sunday_gross_usd", "saturday_gross_usd", "segment_ip_doc")
    pred = test[["release_run_id", "title", "opening_weekend_start", "release_year"]].copy()
    pred["pred_sun_sat_log_ratio_raw"] = sun_log.values
    pred["pred_sunday_gross_raw"] = test["saturday_gross_usd"].values * np.exp(sun_log.values)
    return pred


def add_after_sat_actuals(test, pred):
    cols = ["release_run_id", "friday_gross_usd", "saturday_gross_usd", "sunday_gross_usd", "opening_weekend_gross_usd", "segment_ip_doc"]
    return pred.merge(test[cols], on="release_run_id", how="left")


def target_r_sun_sat(df, pred_col="pred_sunday_gross_base"):
    actual = pd.to_numeric(df["sunday_gross_usd"], errors="coerce")
    predicted = pd.to_numeric(df[pred_col], errors="coerce")
    out = pd.Series(np.nan, index=df.index, dtype="float64")
    mask = actual.gt(0) & predicted.gt(0)
    out.loc[mask] = np.log(actual.loc[mask] / predicted.loc[mask])
    return out.replace([np.inf, -np.inf], np.nan)


def prior_oos_after_sat(train):
    frames = []
    for validation_year in sorted(train["release_year"].dropna().astype(int).unique()):
        inner_train = train.loc[train["release_year"] < validation_year].copy()
        inner_test = train.loc[train["release_year"].eq(validation_year)].copy()
        if len(inner_train) < MIN_TRAIN_MOVIES or inner_test.empty:
            continue
        pred = add_after_sat_actuals(inner_test, predict_after_saturday_base(inner_test, inner_train))
        pred["pred_sunday_gross_base"] = pred["pred_sunday_gross_raw"]
        pred["r_sun_sat"] = target_r_sun_sat(pred)
        frames.append(pred)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def after_sat_log_bias(train):
    prior = prior_oos_after_sat(train)
    residual = prior["r_sun_sat"].dropna() if not prior.empty else pd.Series(dtype="float64")
    return residual.mean() if len(residual) else 0.0


def rolling_after_sat_base(df, start_year=ROLLING_START_YEAR):
    frames = []
    for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= start_year):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        c = after_sat_log_bias(train)
        pred = add_after_sat_actuals(test, predict_after_saturday_base(test, train))
        pred["pred_sunday_gross_base"] = pred["pred_sunday_gross_raw"] * np.exp(c)
        pred["pred_sun_sat_log_ratio_base"] = pred["pred_sun_sat_log_ratio_raw"] + c
        pred["r_sun_sat"] = target_r_sun_sat(pred)
        pred["test_year"] = test_year
        pred["fold"] = f"train_before_{test_year}__test_{test_year}"
        frames.append(pred)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def residual_cache_is_fresh(cache_path=RESIDUAL_DIAGNOSTIC_CACHE_PATH, source_path=shape_base_path):
    if not cache_path.exists():
        return False
    if not source_path.exists():
        return True
    return cache_path.stat().st_mtime >= source_path.stat().st_mtime


def finalize_residual_table_dates(df):
    out = df.copy()
    for col in ["opening_date", "opening_weekend_start"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce")
    for col in ["release_year", "release_month", "release_week", "test_year"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


if USE_RESIDUAL_DIAGNOSTIC_CACHE and not FORCE_REBUILD_RESIDUAL_DIAGNOSTIC and residual_cache_is_fresh():
    residual_table = pd.read_csv(RESIDUAL_DIAGNOSTIC_CACHE_PATH)
    residual_table = finalize_residual_table_dates(residual_table)
    print(f"Loaded cached residual diagnostic table: {RESIDUAL_DIAGNOSTIC_CACHE_PATH}")
else:
    shape_oos = rolling_shape_prior_v1(shape)
    after_sat_oos = rolling_after_sat_base(shape)

    residual_table = shape_oos.merge(
        after_sat_oos[["release_run_id", "pred_sunday_gross_base", "pred_sun_sat_log_ratio_base", "r_sun_sat"]],
        on="release_run_id",
        how="left",
    )
    residual_table["abs_e_a"] = residual_table["e_a"].abs()
    residual_table["abs_e_b"] = residual_table["e_b"].abs()
    residual_table["abs_r_sun_sat"] = residual_table["r_sun_sat"].abs()
    residual_table["log_own_expected_ow"] = np.log(residual_table["latest_estimate_mid_usd"].where(residual_table["latest_estimate_mid_usd"].gt(0)))
    residual_table["log_theaters"] = np.log(residual_table["opening_weekend_theaters"].where(residual_table["opening_weekend_theaters"].gt(0)))
    residual_table["release_month"] = residual_table["opening_weekend_start"].dt.month
    residual_table["release_week"] = residual_table["opening_weekend_start"].dt.isocalendar().week.astype("float")
    residual_table = finalize_residual_table_dates(residual_table)
    RESIDUAL_DIAGNOSTIC_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    residual_table.to_csv(RESIDUAL_DIAGNOSTIC_CACHE_PATH, index=False)
    print(f"Rebuilt and cached residual diagnostic table: {RESIDUAL_DIAGNOSTIC_CACHE_PATH}")

residual_summary = pd.DataFrame({
    "target": TARGETS,
    "n": [residual_table[t].notna().sum() for t in TARGETS],
    "mean": [residual_table[t].mean() for t in TARGETS],
    "mae": [residual_table[t].abs().mean() for t in TARGETS],
    "rmse": [np.sqrt(np.nanmean(np.square(residual_table[t]))) for t in TARGETS],
})
display(residual_summary)
display(residual_table[["release_run_id", "title", "opening_weekend_start", "e_a", "e_b", "r_sun_sat", "abs_e_a", "abs_e_b", "abs_r_sun_sat"]].head())

## Residual Diagnostic Plots

Graph first, then model. These panels look for repeated structure by time, fitted value, scale, theaters, and calendar buckets.

In [ ]:
def diagnostic_panel(df, target, fitted_col, title):
    plot = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[target, "opening_weekend_start"]).copy()
    fig, axes = plt.subplots(3, 3, figsize=(16, 12), constrained_layout=True)
    axes = axes.ravel()

    ordered = plot.sort_values("opening_weekend_start")
    axes[0].scatter(ordered["opening_weekend_start"], ordered[target], s=18, alpha=0.55, color="#4c78a8")
    axes[0].plot(ordered["opening_weekend_start"], ordered[target].rolling(35, min_periods=10).mean(), color="#e45756", linewidth=2)
    axes[0].axhline(0, color="#111111", linestyle="--", linewidth=1)
    axes[0].set_title("Residual vs release date")

    axes[1].hist(plot[target].dropna(), bins=40, color="#4c78a8", alpha=0.85, edgecolor="white")
    axes[1].axvline(0, color="#111111", linestyle="--", linewidth=1)
    axes[1].set_title("Residual histogram")

    qq = stats.probplot(plot[target].dropna(), dist="norm")
    axes[2].scatter(qq[0][0], qq[0][1], s=16, alpha=0.6, color="#4c78a8")
    slope, intercept = qq[1][0], qq[1][1]
    xline = np.asarray([np.nanmin(qq[0][0]), np.nanmax(qq[0][0])])
    axes[2].plot(xline, intercept + slope * xline, color="#111111", linestyle="--", linewidth=1)
    axes[2].set_title("Q-Q plot")

    for ax, xcol, label in [
        (axes[3], fitted_col, "fitted log-ratio"),
        (axes[4], "log_own_expected_ow", "log own expected OW"),
        (axes[5], "log_theaters", "log theaters"),
    ]:
        sub = plot[[xcol, target]].dropna()
        ax.scatter(sub[xcol], sub[target], s=18, alpha=0.55, color="#4c78a8")
        if len(sub) >= 40 and sub[xcol].nunique() > 4:
            bins = pd.qcut(sub[xcol].rank(method="first"), q=10, duplicates="drop")
            smooth = sub.groupby(bins, observed=True).agg(x=(xcol, "mean"), y=(target, "mean"))
            ax.plot(smooth["x"], smooth["y"], color="#e45756", linewidth=2)
        ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
        ax.set_title(f"Residual vs {label}")

    sub = plot[[fitted_col, target]].dropna()
    axes[6].scatter(sub[fitted_col], sub[target].abs(), s=18, alpha=0.55, color="#54a24b")
    axes[6].set_title("Absolute residual vs fitted")
    axes[6].set_xlabel("fitted log-ratio")
    axes[6].set_ylabel(f"abs({target})")

    year_data = [g[target].dropna() for _, g in plot.groupby("release_year", observed=True)]
    year_labels = [str(int(y)) for y in sorted(plot["release_year"].dropna().unique())]
    axes[7].boxplot(year_data, tick_labels=year_labels, showfliers=False)
    axes[7].axhline(0, color="#111111", linestyle="--", linewidth=1)
    axes[7].tick_params(axis="x", rotation=45)
    axes[7].set_title("Residual by year")

    month_data = [g[target].dropna() for _, g in plot.groupby("release_month", observed=True)]
    month_labels = [str(int(m)) for m in sorted(plot["release_month"].dropna().unique())]
    axes[8].boxplot(month_data, tick_labels=month_labels, showfliers=False)
    axes[8].axhline(0, color="#111111", linestyle="--", linewidth=1)
    axes[8].set_title("Residual by release month")

    fig.suptitle(title, fontsize=14)
    plt.show()

diagnostic_panel(residual_table, "e_a", "pred_log_fri_sat", "Shape residual e_a = log(Fri/Sat) miss")
diagnostic_panel(residual_table, "e_b", "pred_log_sun_sat", "Shape residual e_b = log(Sun/Sat) miss")
diagnostic_panel(residual_table, "r_sun_sat", "pred_sun_sat_log_ratio_base", "After-Saturday residual r_sun_sat")

## Calendar and Regime Diagnostics

The current evidence points toward deterministic calendar/regime structure before richer continuous predictors. These plots check whether the month/corridor effects are stable across years and whether they affect point residuals, volatility, or both.

In [ ]:
def assign_release_corridor(df):
    out = df.copy()
    week = pd.to_numeric(out["release_week"], errors="coerce") if "release_week" in out.columns else out["opening_weekend_start"].dt.isocalendar().week.astype(float)
    month = pd.to_numeric(out["release_month"], errors="coerce")
    thanksgiving = month.eq(11) & week.between(46, 48)
    christmas_new_year = month.eq(12) | week.isin([1, 52, 53])
    out["release_corridor"] = np.select(
        [
            christmas_new_year,
            thanksgiving,
            month.between(1, 3),
            month.between(4, 5),
            month.between(6, 8),
            month.between(9, 11),
        ],
        [
            "Christmas/New Year",
            "Thanksgiving",
            "Jan-Mar",
            "Apr-May",
            "summer",
            "weak fall",
        ],
        default="ordinary non-holiday",
    )
    out["corridor_fan_driven"] = out["release_corridor"].astype(str) + " | fan=" + out["is_fan_driven"].fillna(False).astype(int).astype(str)
    out["corridor_family"] = out["release_corridor"].astype(str) + " | family=" + out["is_family_animation"].fillna(False).astype(int).astype(str)
    out["corridor_holiday"] = out["release_corridor"].astype(str) + " | holiday=" + out["is_holiday_corridor"].fillna(False).astype(int).astype(str)
    if "log_own_expected_ow" in out.columns:
        valid = out["log_own_expected_ow"].replace([np.inf, -np.inf], np.nan).dropna()
        if valid.nunique() >= 4:
            out["expected_ow_tier"] = pd.qcut(out["log_own_expected_ow"].rank(method="first"), q=4, labels=["Q1", "Q2", "Q3", "Q4"])
            out["corridor_expected_ow_tier"] = out["release_corridor"].astype(str) + " | expected=" + out["expected_ow_tier"].astype(str)
    return out

residual_table = assign_release_corridor(residual_table)
calendar_targets = ["e_a", "e_b", "r_sun_sat", "abs_e_a", "abs_e_b", "abs_r_sun_sat"]


def month_year_heatmap(df, target):
    heat = df.dropna(subset=[target, "release_year", "release_month"]).copy()
    table = heat.pivot_table(index="release_month", columns="release_year", values=target, aggfunc="mean", observed=True)
    fig, ax = plt.subplots(figsize=(14, 4.8))
    vmax = np.nanmax(np.abs(table.values)) if target.startswith("abs_") is False else np.nanmax(table.values)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1
    cmap = "magma" if target.startswith("abs_") else "coolwarm"
    vmin = 0 if target.startswith("abs_") else -vmax
    im = ax.imshow(table.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_yticks(range(len(table.index)), [str(int(m)) for m in table.index])
    ax.set_xticks(range(len(table.columns)), [str(int(y)) for y in table.columns], rotation=45, ha="right")
    ax.set_title(f"Month x year mean {target}")
    ax.set_xlabel("release year")
    ax.set_ylabel("release month")
    fig.colorbar(im, ax=ax, label=target)
    plt.show()
    return table

month_year_tables = []
for target in calendar_targets:
    table = month_year_heatmap(residual_table, target)
    month_year_tables.append(table.stack().rename(target).reset_index())


def seasonal_subseries(df, target):
    data = df.dropna(subset=[target, "release_year", "release_month"]).copy()
    months = sorted(data["release_month"].dropna().astype(int).unique())
    fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=True, constrained_layout=True)
    axes = axes.ravel()
    for ax, month in zip(axes, months):
        g = data.loc[data["release_month"].eq(month)].groupby("release_year", observed=True)[target].mean().reset_index()
        ax.plot(g["release_year"], g[target], marker="o", linewidth=1.4, markersize=3)
        ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
        ax.set_title(f"Month {month}")
        ax.set_xlabel("year")
    for ax in axes[len(months):]:
        ax.axis("off")
    axes[0].set_ylabel(target)
    fig.suptitle(f"Seasonal subseries: {target} by month across years", fontsize=14)
    plt.show()

for target in ["e_a", "e_b", "r_sun_sat"]:
    seasonal_subseries(residual_table, target)


def week_curve(df, target):
    data = df.dropna(subset=[target, "release_week", "release_year"]).copy()
    latest_year = int(data["release_year"].max())
    specs = {
        "2010-2019": data["release_year"].between(2010, 2019),
        "2022+": data["release_year"].ge(2022),
        f"last5y_to_{latest_year}": data["release_year"].between(latest_year - 4, latest_year),
    }
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for label, mask in specs.items():
        g = data.loc[mask].groupby("release_week", observed=True)[target].mean().reindex(range(1, 54))
        ax.plot(g.index, g.values, marker="o", markersize=3, linewidth=1.4, label=label)
    ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
    ax.set_title(f"Calendar-week residual curve: {target}")
    ax.set_xlabel("ISO release week")
    ax.set_ylabel(target)
    ax.legend(frameon=False)
    plt.show()

for target in ["e_a", "e_b", "r_sun_sat"]:
    week_curve(residual_table, target)


def corridor_boxplots(df, target):
    order = ["Jan-Mar", "Apr-May", "summer", "weak fall", "Thanksgiving", "Christmas/New Year", "ordinary non-holiday"]
    data = []
    labels = []
    for corridor in order:
        values = df.loc[df["release_corridor"].eq(corridor), target].dropna()
        if len(values):
            data.append(values)
            labels.append(f"{corridor}\n(n={len(values)})")
    fig, ax = plt.subplots(figsize=(max(9, 1.25 * len(labels)), 4.8))
    ax.boxplot(data, tick_labels=labels, showfliers=True)
    ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
    ax.set_title(f"{target} by release corridor")
    ax.set_ylabel(target)
    plt.xticks(rotation=20, ha="right")
    plt.show()

for target in calendar_targets:
    corridor_boxplots(residual_table, target)

fig, ax = plt.subplots(figsize=(7, 6))
for corridor, g in residual_table.dropna(subset=["e_a", "e_b"]).groupby("release_corridor", observed=True):
    ax.scatter(g["e_a"], g["e_b"], s=24, alpha=0.6, label=corridor)
ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
ax.axvline(0, color="#111111", linestyle="--", linewidth=1)
ax.set_title("Joint residual cloud: e_a vs e_b by corridor")
ax.set_xlabel("e_a")
ax.set_ylabel("e_b")
ax.legend(frameon=False, fontsize=8, loc="best")
plt.show()

# W-shape confounding check: residualize e_a by corridor, then replot against expected OW.
corridor_means = residual_table.groupby("release_corridor", observed=True)["e_a"].transform("mean")
residual_table["e_a_after_corridor_mean"] = residual_table["e_a"] - corridor_means

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for ax, ycol, title in [
    (axes[0], "e_a", "Raw e_a vs expected OW"),
    (axes[1], "e_a_after_corridor_mean", "e_a after corridor mean removal"),
]:
    plot = residual_table[["log_own_expected_ow", ycol]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(plot["log_own_expected_ow"], plot[ycol], s=22, alpha=0.55, color="#4c78a8")
    if len(plot) >= 40:
        bins = pd.qcut(plot["log_own_expected_ow"].rank(method="first"), q=8, duplicates="drop")
        smooth = plot.groupby(bins, observed=True).agg(x=("log_own_expected_ow", "mean"), y=(ycol, "mean"))
        ax.plot(smooth["x"], smooth["y"], color="#e45756", linewidth=2)
    ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("log own expected OW")
    ax.set_ylabel(ycol)
plt.show()

month_year_residual_summary = pd.concat(month_year_tables, ignore_index=True)
display(month_year_residual_summary.head())

## Weekly ACF and Ljung-Box Checks

Movie rows are irregular and multiple films can open in the same weekend. ACF is run only after aggregating to weekly Friday-start residual series.

In [ ]:
def acf_pairwise(series, max_lag):
    values = pd.Series(series, dtype="float64")
    rows = []
    for lag in range(1, max_lag + 1):
        paired = pd.concat([values, values.shift(lag)], axis=1).dropna()
        if len(paired) < 8:
            rho = np.nan
        else:
            rho = paired.iloc[:, 0].corr(paired.iloc[:, 1])
        rows.append({"lag": lag, "acf": rho, "pair_n": len(paired)})
    return pd.DataFrame(rows)


def ljung_box_from_acf(acf_df, lags):
    rows = []
    for lag in lags:
        sub = acf_df.loc[acf_df["lag"].le(lag)].dropna(subset=["acf"])
        if sub.empty:
            rows.append({"lag": lag, "ljung_box_q": np.nan, "p_value": np.nan, "acf_terms": 0})
            continue
        n = sub["pair_n"].max()
        q = n * (n + 2) * np.sum((sub["acf"] ** 2) / np.maximum(n - sub["lag"], 1))
        p = chi2.sf(q, df=len(sub))
        rows.append({"lag": lag, "ljung_box_q": q, "p_value": p, "acf_terms": len(sub)})
    return pd.DataFrame(rows)


def weekly_series(df, target, subset_name="all", mask=None):
    data = df.loc[mask].copy() if mask is not None else df.copy()
    data = data.dropna(subset=["opening_weekend_start", target]).copy()
    if data.empty:
        return pd.Series(dtype="float64", name=target)
    weekly = data.groupby("opening_weekend_start", observed=True)[target].mean().sort_index()
    full_index = pd.date_range(weekly.index.min(), weekly.index.max(), freq="W-FRI")
    weekly = weekly.reindex(full_index)
    weekly.name = f"{target}_{subset_name}"
    return weekly

acf_lags_to_test = [1, 2, 3, 4, 5, 6, 7, 8, 13, 26, 52, 104]
subset_specs = {
    "all_eligible": pd.Series(True, index=residual_table.index),
    "pre_2020_2010_2019": residual_table["release_year"].between(2010, 2019),
    "post_2022": residual_table["release_year"].ge(2022),
    "non_holiday": ~residual_table["is_holiday_corridor"].fillna(False),
    "holiday_corridor": residual_table["is_holiday_corridor"].fillna(False),
}

acf_rows = []
ljung_rows = []
for target in TARGETS:
    for subset_name, mask in subset_specs.items():
        for transform_name, transform in [("mean", lambda s: s), ("volatility", lambda s: s.abs())]:
            series = transform(weekly_series(residual_table, target, subset_name, mask))
            acf_df = acf_pairwise(series, max_lag=104)
            acf_df["target"] = target
            acf_df["subset"] = subset_name
            acf_df["transform"] = transform_name
            acf_df["week_n"] = series.notna().sum()
            acf_rows.append(acf_df)
            lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
            lb["target"] = target
            lb["subset"] = subset_name
            lb["transform"] = transform_name
            lb["week_n"] = series.notna().sum()
            ljung_rows.append(lb)

acf_table = pd.concat(acf_rows, ignore_index=True)
ljung_box_table = pd.concat(ljung_rows, ignore_index=True)

display(ljung_box_table.sort_values(["target", "subset", "transform", "lag"]).head(40))

fig, axes = plt.subplots(3, 2, figsize=(13, 10), constrained_layout=True)
for row_idx, target in enumerate(TARGETS):
    for col_idx, subset_name in enumerate(["all_eligible", "post_2022"]):
        ax = axes[row_idx, col_idx]
        plot = acf_table.loc[
            acf_table["target"].eq(target)
            & acf_table["subset"].eq(subset_name)
            & acf_table["transform"].eq("mean")
            & acf_table["lag"].le(52)
        ]
        ax.bar(plot["lag"], plot["acf"], color="#4c78a8", alpha=0.85)
        ax.axhline(0, color="#111111", linewidth=1)
        ax.set_title(f"{target}: weekly ACF, {subset_name}")
        ax.set_xlabel("lag weeks")
        ax.set_ylabel("ACF")
plt.show()

## Rolling Intercept Correction and Corrected ACF

Full-sample residual means are not hard-coded. This section tests a rolling-origin intercept correction and reruns weekly ACF/Ljung-Box on the corrected residuals.

In [ ]:
def rolling_intercept_correction(df, targets=TARGETS, min_train=50):
    frames = []
    for test_year in sorted(df["release_year"].dropna().astype(int).unique()):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < min_train or test.empty:
            continue
        out = test.copy()
        out["intercept_train_n"] = len(train)
        for target in targets:
            intercept = train[target].dropna().mean()
            out[f"pred_intercept_{target}"] = intercept
            out[f"{target}_intercept_corrected"] = out[target] - intercept
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

intercept_corrected = rolling_intercept_correction(residual_table)
intercept_summary_rows = []
for target in TARGETS:
    base_mae = intercept_corrected[target].abs().mean()
    corrected_mae = intercept_corrected[f"{target}_intercept_corrected"].abs().mean()
    intercept_summary_rows.append({
        "target": target,
        "n": intercept_corrected[target].notna().sum(),
        "baseline_mae": base_mae,
        "intercept_corrected_mae": corrected_mae,
        "mae_improvement_pct": (base_mae - corrected_mae) / base_mae,
        "corrected_bias": intercept_corrected[f"{target}_intercept_corrected"].mean(),
    })
intercept_correction_summary = pd.DataFrame(intercept_summary_rows)
display(intercept_correction_summary)

intercept_acf_rows = []
intercept_ljung_rows = []
for target in TARGETS:
    corrected_col = f"{target}_intercept_corrected"
    series = weekly_series(intercept_corrected, corrected_col, "intercept_corrected")
    acf_df = acf_pairwise(series, max_lag=104)
    acf_df["target"] = target
    acf_df["correction"] = "rolling_intercept"
    acf_df["week_n"] = series.notna().sum()
    intercept_acf_rows.append(acf_df)
    lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
    lb["target"] = target
    lb["correction"] = "rolling_intercept"
    lb["week_n"] = series.notna().sum()
    intercept_ljung_rows.append(lb)
intercept_acf_table = pd.concat(intercept_acf_rows, ignore_index=True)
intercept_ljung_box_table = pd.concat(intercept_ljung_rows, ignore_index=True)
display(intercept_ljung_box_table.head(36))

fig, axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True)
for ax, target in zip(axes, TARGETS):
    raw = acf_table.loc[(acf_table["target"].eq(target)) & (acf_table["subset"].eq("all_eligible")) & (acf_table["transform"].eq("mean")) & (acf_table["lag"].le(52))]
    corr = intercept_acf_table.loc[(intercept_acf_table["target"].eq(target)) & (intercept_acf_table["lag"].le(52))]
    ax.plot(raw["lag"], raw["acf"], color="#999999", linewidth=1.6, label="raw")
    ax.plot(corr["lag"], corr["acf"], color="#4c78a8", linewidth=1.8, label="rolling intercept corrected")
    ax.axhline(0, color="#111111", linewidth=1)
    ax.set_title(f"Weekly ACF before/after rolling intercept: {target}")
    ax.set_xlabel("lag weeks")
    ax.set_ylabel("ACF")
    ax.legend(frameon=False)
plt.show()

## Feature-Specific Residual Plots

A smoothed curve alone is not enough. These binned summaries show mean residuals with confidence intervals and split views by holiday corridor and fan-driven status.

In [ ]:
def binned_residual_summary(df, feature, target, q=5, split_col=None):
    cols = [feature, target] + ([split_col] if split_col else [])
    plot = df[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if plot.empty or plot[feature].nunique() < 2:
        return pd.DataFrame()
    if split_col is None:
        plot["split"] = "all"
    else:
        plot["split"] = plot[split_col].astype(str)
    rows = []
    for split, g in plot.groupby("split", observed=True):
        if g[feature].nunique() < 2 or len(g) < 15:
            continue
        g = g.copy()
        g["bin"] = pd.qcut(g[feature].rank(method="first"), q=min(q, max(2, len(g) // 20)), labels=False, duplicates="drop") + 1
        for bin_id, b in g.groupby("bin", observed=True):
            se = b[target].std(ddof=1) / np.sqrt(len(b)) if len(b) > 1 else np.nan
            rows.append({
                "feature": feature,
                "target": target,
                "split": split,
                "bin": int(bin_id),
                "n": len(b),
                "feature_mean": b[feature].mean(),
                "residual_mean": b[target].mean(),
                "ci_low": b[target].mean() - 1.96 * se if pd.notna(se) else np.nan,
                "ci_high": b[target].mean() + 1.96 * se if pd.notna(se) else np.nan,
            })
    return pd.DataFrame(rows)

candidate_features = ["log_own_expected_ow", "log_theaters", "latest_estimate_mid_usd", "rt_fresh_share", "wiki_views_cume_to_m1"]
binned_tables = []
for target in TARGETS:
    for feature in candidate_features:
        if feature in residual_table.columns:
            binned_tables.append(binned_residual_summary(residual_table, feature, target, split_col=None))
            binned_tables.append(binned_residual_summary(residual_table, feature, target, split_col="is_holiday_corridor"))
            binned_tables.append(binned_residual_summary(residual_table, feature, target, split_col="is_fan_driven"))
feature_binned_summary = pd.concat([t for t in binned_tables if not t.empty], ignore_index=True) if binned_tables else pd.DataFrame()

def plot_binned(feature, target, split="all"):
    plot = feature_binned_summary.loc[
        feature_binned_summary["feature"].eq(feature)
        & feature_binned_summary["target"].eq(target)
        & feature_binned_summary["split"].eq(split)
    ].copy()
    if plot.empty:
        print(f"No binned data for {target} vs {feature}, split={split}")
        return
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.errorbar(plot["feature_mean"], plot["residual_mean"], yerr=[plot["residual_mean"] - plot["ci_low"], plot["ci_high"] - plot["residual_mean"]], marker="o", linewidth=1.8, capsize=3)
    for _, row in plot.iterrows():
        ax.annotate(f"n={int(row['n'])}", (row["feature_mean"], row["residual_mean"]), textcoords="offset points", xytext=(3, 4), fontsize=8)
    ax.axhline(0, color="#111111", linestyle="--", linewidth=1)
    ax.set_title(f"Binned {target} vs {feature} ({split})")
    ax.set_xlabel(feature)
    ax.set_ylabel(target)
    plt.show()

for target in TARGETS:
    plot_binned("log_own_expected_ow", target)
    plot_binned("rt_fresh_share", target)

display(feature_binned_summary.head(30))

## Category-Aware Shrinkage Shape Corrections

These are not continuous predictor soups. They test whether stable category residual structure exists beyond an intercept-only correction:

\[
\widehat\delta_g = \frac{n_g}{n_g+\lambda}\bar e_g
\]

The correction is applied to both log-ratio axes, then converted back to coherent shares.

In [ ]:
def add_category_columns(df):
    out = df.copy()
    out["weak_fall_corridor"] = out["release_month"].isin([9, 10]) & ~out["is_holiday_corridor"].fillna(False)
    out["category_shape_priority"] = np.select(
        [
            out["is_doc_concert"].fillna(False),
            out["is_family_animation"].fillna(False),
            out["is_horror"].fillna(False),
            out["is_action"].fillna(False) | out["is_franchise"].fillna(False),
            out["is_holiday_corridor"].fillna(False),
            out["weak_fall_corridor"].fillna(False),
        ],
        ["doc/concert", "family/animation", "horror", "action/franchise", "holiday corridor", "weak fall corridor"],
        default="ordinary non-holiday",
    )
    out["category_franchise_fan"] = np.select(
        [out["is_franchise"].fillna(False), out["is_fan_driven"].fillna(False)],
        ["franchise/IP", "fan-driven non-franchise"],
        default="other",
    )
    out["category_mpaa"] = out["mpa_rating"].fillna("unknown")
    out["category_month_corridor"] = np.where(
        out["is_holiday_corridor"].fillna(False),
        "holiday corridor",
        "month_" + out["release_month"].fillna(0).astype(int).astype(str).str.zfill(2),
    )
    return out

residual_table = add_category_columns(residual_table)


def shrinkage_delta(train, test, target, group_col=None, lam=25, include_intercept=False):
    clean = train[[target] + ([group_col] if group_col else [])].dropna(subset=[target]).copy()
    if clean.empty:
        return pd.Series(0.0, index=test.index)
    global_mean = clean[target].mean()
    if group_col is None:
        return pd.Series(global_mean if include_intercept else 0.0, index=test.index)
    stats_by_group = clean.groupby(group_col, dropna=False, observed=True)[target].agg(["mean", "count"])
    if include_intercept:
        stats_by_group["delta"] = global_mean + (stats_by_group["count"] / (stats_by_group["count"] + lam)) * (stats_by_group["mean"] - global_mean)
        fallback = global_mean
    else:
        stats_by_group["delta"] = (stats_by_group["count"] / (stats_by_group["count"] + lam)) * stats_by_group["mean"]
        fallback = 0.0
    return test[group_col].map(stats_by_group["delta"]).fillna(fallback).astype(float)


def shares_from_vector_ab(a, b):
    exp_a = np.exp(a)
    exp_b = np.exp(b)
    denom = 1.0 + exp_a + exp_b
    return pd.DataFrame({
        "new_friday_share": exp_a / denom,
        "new_saturday_share": 1.0 / denom,
        "new_sunday_share": exp_b / denom,
    }, index=a.index)


def rolling_category_shrinkage(df, specs, start_year=ROLLING_START_YEAR):
    frames = []
    for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= start_year):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        for spec in specs:
            pred_e_a = shrinkage_delta(train, test, "e_a", spec.get("group_col"), spec.get("lambda", 25), spec.get("include_intercept", False))
            pred_e_b = shrinkage_delta(train, test, "e_b", spec.get("group_col"), spec.get("lambda", 25), spec.get("include_intercept", False))
            out = test.copy()
            out["shrinkage_model"] = spec["model"]
            out["group_col"] = spec.get("group_col") or "none"
            out["lambda"] = spec.get("lambda", np.nan)
            out["pred_e_a"] = pred_e_a.values
            out["pred_e_b"] = pred_e_b.values
            out["corrected_e_a"] = out["e_a"] - out["pred_e_a"]
            out["corrected_e_b"] = out["e_b"] - out["pred_e_b"]
            new_a = out["pred_log_fri_sat"] + out["pred_e_a"]
            new_b = out["pred_log_sun_sat"] + out["pred_e_b"]
            new_shares = shares_from_vector_ab(new_a, new_b)
            out = pd.concat([out, new_shares], axis=1)
            out["new_friday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["new_friday_share"]
            out["new_saturday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["new_saturday_share"]
            out["new_sunday_gross_shape_only"] = out["opening_weekend_gross_usd"] * out["new_sunday_share"]
            out["new_friday_dollar_error_shape_only"] = out["friday_gross_usd"] - out["new_friday_gross_shape_only"]
            out["new_saturday_dollar_error_shape_only"] = out["saturday_gross_usd"] - out["new_saturday_gross_shape_only"]
            out["new_sunday_dollar_error_shape_only"] = out["sunday_gross_usd"] - out["new_sunday_gross_shape_only"]
            out["new_avg_abs_daily_dollar_error_shape_only"] = out[[
                "new_friday_dollar_error_shape_only", "new_saturday_dollar_error_shape_only", "new_sunday_dollar_error_shape_only",
            ]].abs().mean(axis=1)
            out["test_year"] = test_year
            frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

shrinkage_specs = [
    {"model": "baseline_no_residual_correction", "group_col": None, "lambda": np.nan, "include_intercept": False},
    {"model": "intercept_only_correction", "group_col": None, "lambda": np.nan, "include_intercept": True},
]
for group_col in ["segment_current_broad", "category_shape_priority", "category_franchise_fan", "category_mpaa", "category_month_corridor"]:
    for lam in [10, 25, 50, 100]:
        shrinkage_specs.append({"model": f"{group_col}_shrink_lambda_{lam}", "group_col": group_col, "lambda": lam, "include_intercept": False})
        shrinkage_specs.append({"model": f"{group_col}_partial_pool_lambda_{lam}", "group_col": group_col, "lambda": lam, "include_intercept": True})

category_predictions = rolling_category_shrinkage(residual_table, shrinkage_specs)

def combined_logratio_mae(g, corrected=False):
    a_col = "corrected_e_a" if corrected else "e_a"
    b_col = "corrected_e_b" if corrected else "e_b"
    return pd.concat([g[a_col].abs(), g[b_col].abs()], ignore_index=True).mean()


def summarize_category_predictions(preds):
    rows = []
    for model, g in preds.groupby("shrinkage_model", observed=True):
        baseline_log_mae = combined_logratio_mae(g, corrected=False)
        corrected_log_mae = combined_logratio_mae(g, corrected=True)
        baseline_daily_mae = g["avg_abs_daily_dollar_error_shape_only"].mean()
        corrected_daily_mae = g["new_avg_abs_daily_dollar_error_shape_only"].mean()

        recent = g.loc[g["release_year"].ge(2022)]
        if recent.empty:
            recent_improvement = np.nan
        else:
            recent_base = combined_logratio_mae(recent, corrected=False)
            recent_corr = combined_logratio_mae(recent, corrected=True)
            recent_improvement = (recent_base - recent_corr) / recent_base if recent_base else np.nan

        year_rows = []
        for year, yg in g.groupby("release_year", observed=True):
            if len(yg) < 5:
                continue
            base_y = combined_logratio_mae(yg, corrected=False)
            corr_y = combined_logratio_mae(yg, corrected=True)
            year_rows.append((base_y - corr_y) / base_y if base_y else np.nan)
        year_improvements = pd.Series(year_rows, dtype="float64").dropna()

        rows.append({
            "shrinkage_model": model,
            "n": len(g),
            "group_col": g["group_col"].iloc[0],
            "lambda": g["lambda"].iloc[0],
            "baseline_logratio_mae": baseline_log_mae,
            "corrected_logratio_mae": corrected_log_mae,
            "logratio_mae_improvement_pct": (baseline_log_mae - corrected_log_mae) / baseline_log_mae,
            "baseline_daily_mae_usd": baseline_daily_mae,
            "corrected_daily_mae_usd": corrected_daily_mae,
            "daily_mae_improvement_pct": (baseline_daily_mae - corrected_daily_mae) / baseline_daily_mae,
            "recent_2022_logratio_improvement_pct": recent_improvement,
            "leave_one_year_min_improvement_pct": year_improvements.min() if len(year_improvements) else np.nan,
            "leave_one_year_positive_share": year_improvements.gt(0).mean() if len(year_improvements) else np.nan,
            "mean_corrected_e_a": g["corrected_e_a"].mean(),
            "mean_corrected_e_b": g["corrected_e_b"].mean(),
        })
    out = pd.DataFrame(rows).sort_values(["corrected_logratio_mae", "corrected_daily_mae_usd"]).reset_index(drop=True)
    intercept_mae = out.loc[out["shrinkage_model"].eq("intercept_only_correction"), "corrected_logratio_mae"].iloc[0]
    out["beats_baseline_5pct"] = out["logratio_mae_improvement_pct"].ge(0.05)
    out["beats_intercept"] = out["corrected_logratio_mae"].lt(intercept_mae)
    out["survives_2022_validation"] = out["recent_2022_logratio_improvement_pct"].gt(0)
    out["survives_leave_one_year"] = out["leave_one_year_min_improvement_pct"].gt(0)
    out["promotion_candidate"] = (
        out["beats_baseline_5pct"]
        & out["beats_intercept"]
        & out["daily_mae_improvement_pct"].ge(0)
        & out["survives_2022_validation"]
        & out["survives_leave_one_year"]
    )
    return out

category_summary = summarize_category_predictions(category_predictions)
display(category_summary.head(25))

## Rolling Calendar/Regime Shrinkage

Static all-history month/corridor shrinkage had signal but failed promotion checks. This section tests rolling calendar/regime corrections instead:

\[
\widehat e_{k,i}=\widehat\mu^{roll}_{k,t}+\widehat\delta^{roll}_{k,c(i),t}
\]

with shrinkage toward the rolling intercept. Targets are evaluated separately because `e_b` and `r_sun_sat` show stronger seasonal structure than `e_a`.

In [ ]:
def rolling_train_window(train, test_year, window):
    if window == "all_prior":
        return train.copy()
    if window == "last5y":
        return train.loc[train["release_year"] >= test_year - 5].copy()
    if window == "current_regime_2022":
        current = train.loc[train["release_year"] >= 2022].copy()
        return current if len(current) >= MIN_TRAIN_MOVIES else train.copy()
    raise ValueError(window)


def rolling_shrinkage_prediction(train, test, target, group_col="release_corridor", lam=25, window="last5y", include_regime_intercept=False, test_year=None):
    base_train = rolling_train_window(train, test_year, window)
    if len(base_train.dropna(subset=[target])) < MIN_TRAIN_MOVIES:
        base_train = train.copy()
    mu = base_train[target].dropna().mean()
    if include_regime_intercept and test_year is not None and test_year >= 2023:
        regime_train = train.loc[train["release_year"] >= 2022].dropna(subset=[target]).copy()
        if len(regime_train) >= 20:
            mu = regime_train[target].mean()
            base_train = regime_train
    stats_by_group = base_train.dropna(subset=[target]).groupby(group_col, observed=True)[target].agg(["mean", "count"])
    stats_by_group["delta"] = (stats_by_group["count"] / (stats_by_group["count"] + lam)) * (stats_by_group["mean"] - mu)
    pred = mu + test[group_col].map(stats_by_group["delta"]).fillna(0.0)
    return pred.astype(float), len(base_train), mu


def rolling_calendar_target_predictions(df, specs, targets=TARGETS):
    frames = []
    for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= ROLLING_START_YEAR):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        for spec in specs:
            for target in targets:
                pred, train_n, mu = rolling_shrinkage_prediction(
                    train,
                    test,
                    target,
                    group_col=spec["group_col"],
                    lam=spec["lambda"],
                    window=spec["window"],
                    include_regime_intercept=spec.get("include_regime_intercept", False),
                    test_year=test_year,
                )
                out = test[["release_run_id", "title", "opening_weekend_start", "release_year", "release_corridor", target]].copy()
                out["target"] = target
                out["rolling_calendar_model"] = spec["model"]
                out["group_col"] = spec["group_col"]
                out["window"] = spec["window"]
                out["lambda"] = spec["lambda"]
                out["include_regime_intercept"] = spec.get("include_regime_intercept", False)
                out["rolling_mu"] = mu
                out["calendar_train_n"] = train_n
                out["predicted_residual"] = pred.values
                out["corrected_residual"] = out[target] - out["predicted_residual"]
                out["test_year"] = test_year
                frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

rolling_calendar_specs = []
for window in ["all_prior", "last5y", "current_regime_2022"]:
    for lam in [10, 25, 50, 100]:
        rolling_calendar_specs.append({"model": f"corridor_{window}_lambda_{lam}", "group_col": "release_corridor", "window": window, "lambda": lam})
        rolling_calendar_specs.append({"model": f"corridor_fan_{window}_lambda_{lam}", "group_col": "corridor_fan_driven", "window": window, "lambda": lam})
for lam in [10, 25, 50]:
    rolling_calendar_specs.append({"model": f"corridor_current_regime_intercept_lambda_{lam}", "group_col": "release_corridor", "window": "all_prior", "lambda": lam, "include_regime_intercept": True})

rolling_calendar_predictions = rolling_calendar_target_predictions(residual_table, rolling_calendar_specs)

rolling_calendar_summary_rows = []
for (target, model), g in rolling_calendar_predictions.groupby(["target", "rolling_calendar_model"], observed=True):
    base_mae = g[target].abs().mean()
    corr_mae = g["corrected_residual"].abs().mean()
    recent = g.loc[g["release_year"].ge(2022)]
    recent_improvement = np.nan
    if not recent.empty:
        recent_base = recent[target].abs().mean()
        recent_corr = recent["corrected_residual"].abs().mean()
        recent_improvement = (recent_base - recent_corr) / recent_base if recent_base else np.nan
    year_improvements = []
    for _, yg in g.groupby("release_year", observed=True):
        if len(yg) < 5:
            continue
        base_y = yg[target].abs().mean()
        corr_y = yg["corrected_residual"].abs().mean()
        year_improvements.append((base_y - corr_y) / base_y if base_y else np.nan)
    year_improvements = pd.Series(year_improvements, dtype="float64").dropna()
    rolling_calendar_summary_rows.append({
        "target": target,
        "rolling_calendar_model": model,
        "n": len(g),
        "baseline_mae": base_mae,
        "corrected_mae": corr_mae,
        "mae_improvement_pct": (base_mae - corr_mae) / base_mae,
        "recent_2022_improvement_pct": recent_improvement,
        "leave_one_year_min_improvement_pct": year_improvements.min() if len(year_improvements) else np.nan,
        "leave_one_year_positive_share": year_improvements.gt(0).mean() if len(year_improvements) else np.nan,
        "corrected_bias": g["corrected_residual"].mean(),
        "group_col": g["group_col"].iloc[0],
        "window": g["window"].iloc[0],
        "lambda": g["lambda"].iloc[0],
        "include_regime_intercept": g["include_regime_intercept"].iloc[0],
    })
rolling_calendar_summary = pd.DataFrame(rolling_calendar_summary_rows).sort_values(["target", "corrected_mae"]).reset_index(drop=True)
rolling_calendar_summary["passes_5pct"] = rolling_calendar_summary["mae_improvement_pct"].ge(0.05)
rolling_calendar_summary["survives_2022"] = rolling_calendar_summary["recent_2022_improvement_pct"].gt(0)
rolling_calendar_summary["survives_leave_one_year"] = rolling_calendar_summary["leave_one_year_min_improvement_pct"].gt(0)
rolling_calendar_summary["promotion_candidate"] = rolling_calendar_summary["passes_5pct"] & rolling_calendar_summary["survives_2022"] & rolling_calendar_summary["survives_leave_one_year"]

display(rolling_calendar_summary.groupby("target", observed=True).head(10))

# Corrected ACF using the best rolling calendar model by target as a diagnostic, not a promotion.
best_calendar_models = rolling_calendar_summary.sort_values(["target", "corrected_mae"]).groupby("target", observed=True).head(1)[["target", "rolling_calendar_model"]]
calendar_corrected = []
for _, row in best_calendar_models.iterrows():
    subset = rolling_calendar_predictions.loc[
        rolling_calendar_predictions["target"].eq(row["target"])
        & rolling_calendar_predictions["rolling_calendar_model"].eq(row["rolling_calendar_model"])
    ].copy()
    subset = subset.rename(columns={"corrected_residual": f"{row['target']}_calendar_corrected"})
    calendar_corrected.append(subset)
calendar_corrected_predictions = pd.concat(calendar_corrected, ignore_index=True) if calendar_corrected else pd.DataFrame()

calendar_acf_rows = []
calendar_ljung_rows = []
for _, row in best_calendar_models.iterrows():
    target = row["target"]
    col = f"{target}_calendar_corrected"
    subset = calendar_corrected_predictions.loc[calendar_corrected_predictions["target"].eq(target)].copy()
    series = weekly_series(subset.rename(columns={col: "calendar_corrected"}), "calendar_corrected", "calendar_corrected")
    acf_df = acf_pairwise(series, max_lag=104)
    acf_df["target"] = target
    acf_df["correction"] = "best_rolling_calendar"
    acf_df["model"] = row["rolling_calendar_model"]
    acf_df["week_n"] = series.notna().sum()
    calendar_acf_rows.append(acf_df)
    lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
    lb["target"] = target
    lb["correction"] = "best_rolling_calendar"
    lb["model"] = row["rolling_calendar_model"]
    lb["week_n"] = series.notna().sum()
    calendar_ljung_rows.append(lb)
calendar_acf_table = pd.concat(calendar_acf_rows, ignore_index=True) if calendar_acf_rows else pd.DataFrame()
calendar_ljung_box_table = pd.concat(calendar_ljung_rows, ignore_index=True) if calendar_ljung_rows else pd.DataFrame()
display(calendar_ljung_box_table.head(36))

## Current-Regime Corridor x Movie-Type Screen

The broad rolling-calendar corrections are useful evidence, not a promoted forecast. The next diagnostic model is deliberately compact and post-2022 focused:

- `e_a`: current-regime intercept plus broad corridor only;
- `e_b`: current-regime corridor plus compact movie type, with a high-shrinkage interaction test;
- `r_sun_sat`: summer and Christmas Sunday lift with current-regime shrinkage.

Validation is softer than the old every-year-positive rule but more specific to the problem: overall improvement above 5%, post-2022 improvement above 2%, no large corridor-year miss, and at least 60% positive corridor-year hit rate. These are still screening rules, not promotion rules; any candidate also needs a final weekend-share/daily-dollar backtest before it can ship.


In [ ]:
def add_current_regime_model_columns(df):
    out = df.copy()
    out["post_2022_regime"] = out["release_year"].ge(2022)
    out["compact_movie_type"] = np.select(
        [
            out["is_doc_concert"].fillna(False),
            out["is_family_animation"].fillna(False),
            out["is_horror"].fillna(False),
            out["is_franchise"].fillna(False) | out["is_fan_driven"].fillna(False),
        ],
        ["doc/concert", "family/animation", "horror", "franchise/fan-driven"],
        default="other wide",
    )
    # Keep doc/concert visible for intervals, but avoid sparse point-model interaction cells.
    out["compact_movie_type_model"] = np.where(out["compact_movie_type"].eq("doc/concert"), "other wide", out["compact_movie_type"])
    out["corridor_movie_type"] = out["release_corridor"].astype(str) + " | " + out["compact_movie_type_model"].astype(str)
    out["sun_lift_corridor"] = np.select(
        [out["release_corridor"].eq("summer"), out["release_corridor"].eq("Christmas/New Year")],
        ["summer", "Christmas/New Year"],
        default="other_corridor",
    )
    return out


residual_table = add_current_regime_model_columns(residual_table)


def residual_mean_ci_table(df, group_cols, targets=TARGETS, min_n=3):
    rows = []
    for target in targets:
        clean = df.dropna(subset=[target]).copy()
        for keys, g in clean.groupby(group_cols, observed=True):
            if len(g) < min_n:
                continue
            if not isinstance(keys, tuple):
                keys = (keys,)
            se = g[target].std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else np.nan
            row = {col: key for col, key in zip(group_cols, keys)}
            row.update({
                "target": target,
                "n": len(g),
                "mean_residual": g[target].mean(),
                "ci_low": g[target].mean() - 1.96 * se if pd.notna(se) else np.nan,
                "ci_high": g[target].mean() + 1.96 * se if pd.notna(se) else np.nan,
                "mae": g[target].abs().mean(),
            })
            rows.append(row)
    return pd.DataFrame(rows)


post_2022 = residual_table.loc[residual_table["release_year"].ge(2022)].copy()
post_2022_corridor_residual_summary = residual_mean_ci_table(post_2022, ["release_corridor"])
post_2022_corridor_movie_residual_summary = residual_mean_ci_table(post_2022, ["release_corridor", "compact_movie_type_model"])

def plot_post_2022_residual_bars(summary, group_col, title_prefix, max_groups=18):
    if summary.empty:
        return
    for target in TARGETS:
        plot = summary.loc[summary["target"].eq(target)].sort_values("mean_residual").tail(max_groups)
        if plot.empty:
            continue
        fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(plot))))
        yerr = [plot["mean_residual"] - plot["ci_low"], plot["ci_high"] - plot["mean_residual"]]
        ax.barh(plot[group_col].astype(str), plot["mean_residual"], color="#4c78a8", alpha=0.85)
        ax.errorbar(plot["mean_residual"], plot[group_col].astype(str), xerr=yerr, fmt="none", color="#222222", capsize=3)
        for _, row in plot.iterrows():
            ax.text(row["mean_residual"], str(row[group_col]), f" n={int(row['n'])}", va="center", ha="left" if row["mean_residual"] >= 0 else "right", fontsize=8)
        ax.axvline(0, color="#222222", linewidth=1)
        ax.set_title(f"{title_prefix}: {target}")
        ax.set_xlabel("mean log residual, post-2022")
        plt.show()


plot_post_2022_residual_bars(post_2022_corridor_residual_summary, "release_corridor", "Post-2022 residual by corridor")

post_2022_corridor_movie_plot = post_2022_corridor_movie_residual_summary.copy()
post_2022_corridor_movie_plot["corridor_movie"] = post_2022_corridor_movie_plot["release_corridor"].astype(str) + " | " + post_2022_corridor_movie_plot["compact_movie_type_model"].astype(str)
plot_post_2022_residual_bars(post_2022_corridor_movie_plot, "corridor_movie", "Post-2022 residual by corridor x movie type", max_groups=24)

share_summary_rows = []
for period_name, mask in {
    "pre_2020": residual_table["release_year"].between(2010, 2019),
    "post_2022": residual_table["release_year"].ge(2022),
}.items():
    subset = residual_table.loc[mask].copy()
    for corridor, g in subset.groupby("release_corridor", observed=True):
        if len(g) < 5:
            continue
        for actual_col, pred_col, day in zip(SHARE_COLS, PRED_SHARE_COLS, ["Friday", "Saturday", "Sunday"]):
            share_summary_rows.append({
                "period": period_name,
                "release_corridor": corridor,
                "day": day,
                "n": len(g),
                "actual_share": g[actual_col].mean(),
                "predicted_share": g[pred_col].mean(),
                "share_residual": g[actual_col].mean() - g[pred_col].mean(),
            })
share_regime_summary = pd.DataFrame(share_summary_rows)

def plot_share_regime_shift(summary):
    if summary.empty:
        return
    for period_name, period_df in summary.groupby("period", observed=True):
        corridors = period_df.sort_values(["release_corridor", "day"])["release_corridor"].drop_duplicates().tolist()
        fig, axes = plt.subplots(len(corridors), 1, figsize=(9, max(3, 1.8 * len(corridors))), sharex=True)
        if len(corridors) == 1:
            axes = [axes]
        for ax, corridor in zip(axes, corridors):
            plot = period_df.loc[period_df["release_corridor"].eq(corridor)].set_index("day").reindex(["Friday", "Saturday", "Sunday"])
            x = np.arange(len(plot))
            ax.plot(x, plot["actual_share"], marker="o", label="actual", color="#4c78a8")
            ax.plot(x, plot["predicted_share"], marker="o", label="baseline", color="#e45756")
            ax.set_title(f"{period_name}: {corridor} (n={int(plot['n'].max())})")
            ax.set_ylabel("share")
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
            ax.set_xticks(x)
            ax.set_xticklabels(plot.index)
            ax.legend(loc="best")
        fig.tight_layout()
        plt.show()


plot_share_regime_shift(share_regime_summary)


def component_shrink_delta(train, test, target, group_col, mu, lam):
    stats_by_group = train.dropna(subset=[target]).groupby(group_col, observed=True)[target].agg(["mean", "count"])
    if stats_by_group.empty:
        return pd.Series(0.0, index=test.index)
    stats_by_group["delta"] = (stats_by_group["count"] / (stats_by_group["count"] + lam)) * (stats_by_group["mean"] - mu)
    return test[group_col].map(stats_by_group["delta"]).fillna(0.0).astype(float)


def interaction_shrink_delta(train, test, target, corridor_col, movie_col, mu, lam):
    clean = train.dropna(subset=[target]).copy()
    if clean.empty:
        return pd.Series(0.0, index=test.index)
    corridor_mean = clean.groupby(corridor_col, observed=True)[target].mean()
    movie_mean = clean.groupby(movie_col, observed=True)[target].mean()
    stats = clean.groupby([corridor_col, movie_col], observed=True)[target].agg(["mean", "count"])
    stats["centered_interaction"] = stats["mean"] - stats.index.get_level_values(0).map(corridor_mean) - stats.index.get_level_values(1).map(movie_mean) + mu
    stats["delta"] = (stats["count"] / (stats["count"] + lam)) * stats["centered_interaction"]
    keys = pd.MultiIndex.from_frame(test[[corridor_col, movie_col]])
    return pd.Series(keys.map(stats["delta"]).fillna(0.0).astype(float), index=test.index)


def current_regime_training_window(train, target, test_year, min_n=20):
    regime_train = train.loc[train["release_year"].ge(2022)].dropna(subset=[target]).copy()
    if test_year >= 2023 and len(regime_train) >= min_n:
        return regime_train, "post_2022_prior"
    fallback = train.dropna(subset=[target]).copy()
    return fallback, "all_prior_fallback"


def predict_current_regime_residual(train, test, target, spec, test_year):
    model_train, train_window = current_regime_training_window(train, target, test_year, spec.get("min_regime_n", 20))
    if len(model_train) < 5:
        return pd.Series(0.0, index=test.index), train_window, len(model_train), 0.0
    mu = model_train[target].mean() if spec.get("include_intercept", True) else 0.0
    pred = pd.Series(mu, index=test.index, dtype="float64")
    if spec.get("corridor_lam") is not None:
        pred = pred + component_shrink_delta(model_train, test, target, "release_corridor", mu, spec["corridor_lam"])
    if spec.get("movie_lam") is not None:
        pred = pred + component_shrink_delta(model_train, test, target, "compact_movie_type_model", mu, spec["movie_lam"])
    if spec.get("interaction_lam") is not None:
        pred = pred + interaction_shrink_delta(model_train, test, target, "release_corridor", "compact_movie_type_model", mu, spec["interaction_lam"])
    if spec.get("sun_lift_lam") is not None:
        pred = pred + component_shrink_delta(model_train, test, target, "sun_lift_corridor", mu, spec["sun_lift_lam"])
    return pred.astype(float), train_window, len(model_train), mu


current_regime_specs = [
    {
        "model": "e_a_regime_intercept_broad_corridor_lam150",
        "target": "e_a",
        "corridor_lam": 150,
        "include_intercept": True,
        "min_regime_n": 20,
    },
    {
        "model": "e_b_regime_corridor_movie_additive_lam100",
        "target": "e_b",
        "corridor_lam": 100,
        "movie_lam": 100,
        "include_intercept": True,
        "min_regime_n": 20,
    },
    {
        "model": "e_b_regime_corridor_movie_interaction_lam150",
        "target": "e_b",
        "corridor_lam": 100,
        "movie_lam": 100,
        "interaction_lam": 150,
        "include_intercept": True,
        "min_regime_n": 20,
    },
    {
        "model": "r_sun_sat_summer_christmas_lift_lam100",
        "target": "r_sun_sat",
        "sun_lift_lam": 100,
        "include_intercept": True,
        "min_regime_n": 20,
    },
]


def rolling_current_regime_predictions(df, specs, start_year=ROLLING_START_YEAR):
    frames = []
    for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= start_year):
        train = df.loc[df["release_year"] < test_year].copy()
        test = df.loc[df["release_year"].eq(test_year)].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        for spec in specs:
            target = spec["target"]
            pred, train_window, train_n, mu = predict_current_regime_residual(train, test, target, spec, test_year)
            out = test[["release_run_id", "title", "opening_weekend_start", "release_year", "release_week", "release_corridor", "compact_movie_type_model", "sun_lift_corridor", "pred_log_fri_sat", "pred_log_sun_sat", "opening_weekend_gross_usd", "friday_gross_usd", "saturday_gross_usd", "sunday_gross_usd", "pred_sunday_gross_base", target]].copy()
            out["target"] = target
            out["current_regime_model"] = spec["model"]
            out["predicted_residual"] = pred.values
            out["corrected_residual"] = out[target] - out["predicted_residual"]
            out["test_year"] = test_year
            out["train_window"] = train_window
            out["train_n"] = train_n
            out["rolling_mu"] = mu
            frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


current_regime_predictions = rolling_current_regime_predictions(residual_table, current_regime_specs)


def summarize_current_regime_predictions(preds, min_cell_n=3, catastrophic_floor=-0.10):
    rows = []
    for (target, model), g in preds.groupby(["target", "current_regime_model"], observed=True):
        base_mae = g[target].abs().mean()
        corr_mae = g["corrected_residual"].abs().mean()
        recent = g.loc[g["release_year"].ge(2022)]
        recent_base = recent[target].abs().mean() if not recent.empty else np.nan
        recent_corr = recent["corrected_residual"].abs().mean() if not recent.empty else np.nan
        regime_year_rows = []
        for year, yg in recent.groupby("release_year", observed=True):
            if len(yg) < 5:
                continue
            base_y = yg[target].abs().mean()
            corr_y = yg["corrected_residual"].abs().mean()
            regime_year_rows.append((base_y - corr_y) / base_y if base_y else np.nan)
        regime_year_improvements = pd.Series(regime_year_rows, dtype="float64").dropna()
        cell_rows = []
        for _, cg in recent.groupby(["release_corridor", "release_year"], observed=True):
            if len(cg) < min_cell_n:
                continue
            base_cell = cg[target].abs().mean()
            corr_cell = cg["corrected_residual"].abs().mean()
            cell_rows.append((base_cell - corr_cell) / base_cell if base_cell else np.nan)
        cell_improvements = pd.Series(cell_rows, dtype="float64").dropna()
        overall_improvement = (base_mae - corr_mae) / base_mae if base_mae else np.nan
        recent_improvement = (recent_base - recent_corr) / recent_base if recent_base else np.nan
        corridor_hit_rate = cell_improvements.ge(0).mean() if len(cell_improvements) else np.nan
        corridor_min_improvement = cell_improvements.min() if len(cell_improvements) else np.nan
        rows.append({
            "target": target,
            "current_regime_model": model,
            "n": len(g),
            "post_2022_n": len(recent),
            "baseline_mae": base_mae,
            "corrected_mae": corr_mae,
            "mae_improvement_pct": overall_improvement,
            "post_2022_mae_improvement_pct": recent_improvement,
            "post_2022_year_min_improvement_pct": regime_year_improvements.min() if len(regime_year_improvements) else np.nan,
            "post_2022_year_positive_share": regime_year_improvements.ge(0).mean() if len(regime_year_improvements) else np.nan,
            "corridor_year_cell_count": len(cell_improvements),
            "corridor_year_positive_share": corridor_hit_rate,
            "corridor_year_min_improvement_pct": corridor_min_improvement,
            "corrected_bias": g["corrected_residual"].mean(),
            "train_window_share_post_2022": g["train_window"].eq("post_2022_prior").mean(),
        })
    out = pd.DataFrame(rows).sort_values(["target", "corrected_mae"]).reset_index(drop=True)
    out["passes_overall_5pct"] = out["mae_improvement_pct"].gt(0.05)
    out["passes_post_2022_2pct"] = out["post_2022_mae_improvement_pct"].gt(0.02)
    out["no_corridor_year_catastrophic_miss"] = out["corridor_year_min_improvement_pct"].ge(catastrophic_floor)
    out["corridor_year_hit_rate_60pct"] = out["corridor_year_positive_share"].ge(0.60)
    out["passes_screen"] = (
        out["passes_overall_5pct"]
        & out["passes_post_2022_2pct"]
        & out["no_corridor_year_catastrophic_miss"]
        & out["corridor_year_hit_rate_60pct"]
    )
    out["promotion_candidate"] = False
    out["decision"] = np.where(
        out["passes_screen"],
        "screen pass; needs final share/daily-dollar and interval backtest",
        "do not promote yet",
    )
    return out


current_regime_summary = summarize_current_regime_predictions(current_regime_predictions)
display(current_regime_summary)

# Corrected ACF after the current-regime deterministic screen. Use best model per target diagnostically only.
best_current_regime_models = current_regime_summary.sort_values(["target", "corrected_mae"]).groupby("target", observed=True).head(1)[["target", "current_regime_model"]]
current_regime_acf_rows = []
current_regime_ljung_rows = []
for _, row in best_current_regime_models.iterrows():
    target = row["target"]
    subset = current_regime_predictions.loc[
        current_regime_predictions["target"].eq(target)
        & current_regime_predictions["current_regime_model"].eq(row["current_regime_model"])
    ].copy()
    series = weekly_series(subset.rename(columns={"corrected_residual": "current_regime_corrected"}), "current_regime_corrected", "current_regime_corrected")
    acf_df = acf_pairwise(series, max_lag=60)
    acf_df["target"] = target
    acf_df["correction"] = "best_current_regime_screen"
    acf_df["model"] = row["current_regime_model"]
    current_regime_acf_rows.append(acf_df)
    lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
    lb["target"] = target
    lb["correction"] = "best_current_regime_screen"
    lb["model"] = row["current_regime_model"]
    current_regime_ljung_rows.append(lb)
current_regime_acf_table = pd.concat(current_regime_acf_rows, ignore_index=True) if current_regime_acf_rows else pd.DataFrame()
current_regime_ljung_box_table = pd.concat(current_regime_ljung_rows, ignore_index=True) if current_regime_ljung_rows else pd.DataFrame()
display(current_regime_ljung_box_table.head(36))


## Seasonal Residual Add-On Screen

The current-regime screen is the base correction. This section tests one small residual layer after that correction, not a replacement for it:

\[
\hat e^{seasonal}_{k,w}=\omega_{k,w}\rho_k\bar e^{corrected}_{k,w-52},
\quad
\omega_{k,w}=\frac{n_{w-52}}{n_{w-52}+\lambda}
\]

Two sources are tested with hard shrinkage (`lambda` 50, 100, 200): exact same calendar week in the prior year, and same corridor in the prior year. Promotion remains target-specific: `e_b` and `r_sun_sat` can move to candidate stage if this improves 2022+ validation without hurting year/corridor stability; `e_a` remains experimental.


In [ ]:
FROZEN_CURRENT_REGIME_MODELS = {
    "e_a": "e_a_regime_intercept_broad_corridor_lam150",
    "e_b": "e_b_regime_corridor_movie_interaction_lam150",
    "r_sun_sat": "r_sun_sat_summer_christmas_lift_lam100",
}

frozen_current_regime_predictions = pd.concat([
    current_regime_predictions.loc[
        current_regime_predictions["target"].eq(target)
        & current_regime_predictions["current_regime_model"].eq(model)
    ].copy()
    for target, model in FROZEN_CURRENT_REGIME_MODELS.items()
], ignore_index=True)


def shares_from_log_ratios(a, b):
    exp_a = np.exp(a)
    exp_b = np.exp(b)
    denom = 1.0 + exp_a + exp_b
    return exp_a / denom, 1.0 / denom, exp_b / denom


def add_target_dollar_performance(preds, residual_pred_col="predicted_residual", corrected_col="corrected_residual", model_col="current_regime_model"):
    rows = []
    for (target, model), g in preds.groupby(["target", model_col], observed=True):
        base_mae = g[target].abs().mean()
        corrected_mae = g[corrected_col].abs().mean()
        recent = g.loc[g["release_year"].ge(2022)]
        recent_base_mae = recent[target].abs().mean() if not recent.empty else np.nan
        recent_corrected_mae = recent[corrected_col].abs().mean() if not recent.empty else np.nan
        if target in ["e_a", "e_b"]:
            pred_a = g["pred_log_fri_sat"] + np.where(target == "e_a", g[residual_pred_col], 0.0)
            pred_b = g["pred_log_sun_sat"] + np.where(target == "e_b", g[residual_pred_col], 0.0)
            fri_share, sat_share, sun_share = shares_from_log_ratios(pred_a, pred_b)
            corrected_daily_mae = pd.concat([
                (g["friday_gross_usd"] - g["opening_weekend_gross_usd"] * fri_share).abs(),
                (g["saturday_gross_usd"] - g["opening_weekend_gross_usd"] * sat_share).abs(),
                (g["sunday_gross_usd"] - g["opening_weekend_gross_usd"] * sun_share).abs(),
            ], axis=1).mean(axis=1).mean()
            base_fri, base_sat, base_sun = shares_from_log_ratios(g["pred_log_fri_sat"], g["pred_log_sun_sat"])
            base_daily_mae = pd.concat([
                (g["friday_gross_usd"] - g["opening_weekend_gross_usd"] * base_fri).abs(),
                (g["saturday_gross_usd"] - g["opening_weekend_gross_usd"] * base_sat).abs(),
                (g["sunday_gross_usd"] - g["opening_weekend_gross_usd"] * base_sun).abs(),
            ], axis=1).mean(axis=1).mean()
            dollar_metric = "avg_daily_shape_mae_usd"
        else:
            corrected_sunday = g["pred_sunday_gross_base"] * np.exp(g[residual_pred_col])
            corrected_daily_mae = (g["sunday_gross_usd"] - corrected_sunday).abs().mean()
            base_daily_mae = (g["sunday_gross_usd"] - g["pred_sunday_gross_base"]).abs().mean()
            dollar_metric = "after_sat_sunday_mae_usd"
        rows.append({
            "target": target,
            "model": model,
            "n": len(g),
            "post_2022_n": len(recent),
            "baseline_log_mae": base_mae,
            "corrected_log_mae": corrected_mae,
            "log_mae_improvement_pct": (base_mae - corrected_mae) / base_mae if base_mae else np.nan,
            "post_2022_log_mae_improvement_pct": (recent_base_mae - recent_corrected_mae) / recent_base_mae if recent_base_mae else np.nan,
            "baseline_dollar_mae": base_daily_mae,
            "corrected_dollar_mae": corrected_daily_mae,
            "dollar_mae_improvement_pct": (base_daily_mae - corrected_daily_mae) / base_daily_mae if base_daily_mae else np.nan,
            "dollar_metric": dollar_metric,
        })
    return pd.DataFrame(rows).sort_values(["target", "corrected_log_mae"]).reset_index(drop=True)


current_regime_target_performance = add_target_dollar_performance(frozen_current_regime_predictions)
display(current_regime_target_performance)


def estimate_seasonal_rho(train, source):
    if source == "exact_week_52":
        agg = train.groupby(["release_year", "release_week"], observed=True)["corrected_residual"].agg(["mean", "count"]).reset_index()
        lag = agg.rename(columns={"mean": "lag_mean", "count": "lag_n"})
        lag["release_year"] = lag["release_year"] + 1
        pairs = agg.merge(lag[["release_year", "release_week", "lag_mean"]], on=["release_year", "release_week"], how="inner")
    elif source == "same_corridor_prior_year":
        agg = train.groupby(["release_year", "release_corridor"], observed=True)["corrected_residual"].agg(["mean", "count"]).reset_index()
        lag = agg.rename(columns={"mean": "lag_mean", "count": "lag_n"})
        lag["release_year"] = lag["release_year"] + 1
        pairs = agg.merge(lag[["release_year", "release_corridor", "lag_mean"]], on=["release_year", "release_corridor"], how="inner")
    else:
        raise ValueError(source)
    pairs = pairs.replace([np.inf, -np.inf], np.nan).dropna(subset=["mean", "lag_mean"])
    denom = np.square(pairs["lag_mean"]).sum()
    if len(pairs) < 8 or denom <= 0:
        return 0.0, len(pairs)
    rho = (pairs["mean"] * pairs["lag_mean"]).sum() / denom
    return float(np.clip(rho, -1.0, 1.0)), len(pairs)


def seasonal_residual_prediction(train, test, source, lam):
    rho, pair_n = estimate_seasonal_rho(train, source)
    if source == "exact_week_52":
        source_stats = train.groupby(["release_year", "release_week"], observed=True)["corrected_residual"].agg(["mean", "count"]).reset_index()
        lookup = source_stats.set_index(["release_year", "release_week"])
        keys = pd.MultiIndex.from_frame(pd.DataFrame({
            "release_year": test["release_year"] - 1,
            "release_week": test["release_week"],
        }))
    elif source == "same_corridor_prior_year":
        source_stats = train.groupby(["release_year", "release_corridor"], observed=True)["corrected_residual"].agg(["mean", "count"]).reset_index()
        lookup = source_stats.set_index(["release_year", "release_corridor"])
        keys = pd.MultiIndex.from_frame(pd.DataFrame({
            "release_year": test["release_year"] - 1,
            "release_corridor": test["release_corridor"],
        }))
    else:
        raise ValueError(source)
    lag_mean = pd.Series(keys.map(lookup["mean"]).astype(float), index=test.index).fillna(0.0)
    lag_n = pd.Series(keys.map(lookup["count"]).astype(float), index=test.index).fillna(0.0)
    omega = lag_n / (lag_n + lam)
    pred = omega * rho * lag_mean
    return pred.astype(float), rho, pair_n


seasonal_specs = [
    {"seasonal_model": f"exact_week_52_lambda_{lam}", "source": "exact_week_52", "lambda": lam}
    for lam in [50, 100, 200]
] + [
    {"seasonal_model": f"same_corridor_prior_year_lambda_{lam}", "source": "same_corridor_prior_year", "lambda": lam}
    for lam in [50, 100, 200]
]


def rolling_seasonal_addon_predictions(base_preds, specs):
    frames = []
    for (target, base_model), target_df in base_preds.groupby(["target", "current_regime_model"], observed=True):
        target_df = target_df.sort_values("opening_weekend_start").copy()
        for test_year in sorted(target_df["release_year"].dropna().astype(int).unique()):
            train = target_df.loc[target_df["release_year"] < test_year].copy()
            test = target_df.loc[target_df["release_year"].eq(test_year)].copy()
            if len(train) < MIN_TRAIN_MOVIES or test.empty:
                continue
            for spec in specs:
                seasonal_pred, rho, pair_n = seasonal_residual_prediction(train, test, spec["source"], spec["lambda"])
                out = test.copy()
                out["seasonal_model"] = spec["seasonal_model"]
                out["seasonal_source"] = spec["source"]
                out["seasonal_lambda"] = spec["lambda"]
                out["seasonal_rho"] = rho
                out["seasonal_pair_n"] = pair_n
                out["seasonal_predicted_residual"] = seasonal_pred.values
                out["total_predicted_residual"] = out["predicted_residual"] + out["seasonal_predicted_residual"]
                out["seasonal_corrected_residual"] = out["corrected_residual"] - out["seasonal_predicted_residual"]
                frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


seasonal_addon_predictions = rolling_seasonal_addon_predictions(frozen_current_regime_predictions, seasonal_specs)


def summarize_seasonal_addons(preds, min_cell_n=3):
    rows = []
    for (target, base_model, seasonal_model), g in preds.groupby(["target", "current_regime_model", "seasonal_model"], observed=True):
        baseline_mae = g[target].abs().mean()
        current_mae = g["corrected_residual"].abs().mean()
        seasonal_mae = g["seasonal_corrected_residual"].abs().mean()
        recent = g.loc[g["release_year"].ge(2022)]
        recent_current_mae = recent["corrected_residual"].abs().mean() if not recent.empty else np.nan
        recent_seasonal_mae = recent["seasonal_corrected_residual"].abs().mean() if not recent.empty else np.nan
        year_rows = []
        for _, yg in recent.groupby("release_year", observed=True):
            if len(yg) < 5:
                continue
            current_y = yg["corrected_residual"].abs().mean()
            seasonal_y = yg["seasonal_corrected_residual"].abs().mean()
            year_rows.append((current_y - seasonal_y) / current_y if current_y else np.nan)
        year_improvements = pd.Series(year_rows, dtype="float64").dropna()
        cell_rows = []
        for _, cg in recent.groupby(["release_corridor", "release_year"], observed=True):
            if len(cg) < min_cell_n:
                continue
            current_cell = cg["corrected_residual"].abs().mean()
            seasonal_cell = cg["seasonal_corrected_residual"].abs().mean()
            cell_rows.append((current_cell - seasonal_cell) / current_cell if current_cell else np.nan)
        cell_improvements = pd.Series(cell_rows, dtype="float64").dropna()
        rows.append({
            "target": target,
            "current_regime_model": base_model,
            "seasonal_model": seasonal_model,
            "seasonal_source": g["seasonal_source"].iloc[0],
            "seasonal_lambda": g["seasonal_lambda"].iloc[0],
            "n": len(g),
            "post_2022_n": len(recent),
            "baseline_mae": baseline_mae,
            "current_regime_mae": current_mae,
            "seasonal_corrected_mae": seasonal_mae,
            "current_regime_improvement_pct": (baseline_mae - current_mae) / baseline_mae if baseline_mae else np.nan,
            "seasonal_total_improvement_pct": (baseline_mae - seasonal_mae) / baseline_mae if baseline_mae else np.nan,
            "seasonal_addon_improvement_pct": (current_mae - seasonal_mae) / current_mae if current_mae else np.nan,
            "post_2022_seasonal_addon_improvement_pct": (recent_current_mae - recent_seasonal_mae) / recent_current_mae if recent_current_mae else np.nan,
            "post_2022_year_min_addon_improvement_pct": year_improvements.min() if len(year_improvements) else np.nan,
            "post_2022_year_positive_share": year_improvements.gt(0).mean() if len(year_improvements) else np.nan,
            "corridor_year_cell_count": len(cell_improvements),
            "corridor_year_positive_share": cell_improvements.ge(0).mean() if len(cell_improvements) else np.nan,
            "corridor_year_min_addon_improvement_pct": cell_improvements.min() if len(cell_improvements) else np.nan,
            "mean_seasonal_rho": g["seasonal_rho"].mean(),
            "min_seasonal_pair_n": g["seasonal_pair_n"].min(),
        })
    out = pd.DataFrame(rows).sort_values(["target", "seasonal_corrected_mae"]).reset_index(drop=True)
    out["improves_2022_vs_current"] = out["post_2022_seasonal_addon_improvement_pct"].gt(0)
    out["does_not_hurt_leave_one_regime_year"] = out["post_2022_year_min_addon_improvement_pct"].ge(0)
    out["corridor_hit_rate_60pct"] = out["corridor_year_positive_share"].ge(0.60)
    out["candidate_stage"] = out["target"].isin(["e_b", "r_sun_sat"]) & out["improves_2022_vs_current"] & out["does_not_hurt_leave_one_regime_year"] & out["corridor_hit_rate_60pct"]
    out["promotion_candidate"] = False
    out["decision"] = np.select(
        [out["candidate_stage"], out["target"].eq("e_a")],
        ["candidate-stage only; requires dollar validation", "keep experimental"],
        default="do not promote yet",
    )
    return out


seasonal_addon_summary = summarize_seasonal_addons(seasonal_addon_predictions)
display(seasonal_addon_summary.groupby("target", observed=True).head(6))

seasonal_target_performance = add_target_dollar_performance(
    seasonal_addon_predictions,
    residual_pred_col="total_predicted_residual",
    corrected_col="seasonal_corrected_residual",
    model_col="seasonal_model",
)
display(seasonal_target_performance.groupby("target", observed=True).head(6))

seasonal_acf_rows = []
seasonal_ljung_rows = []
best_seasonal_models = seasonal_addon_summary.sort_values(["target", "seasonal_corrected_mae"]).groupby("target", observed=True).head(1)
for _, row in best_seasonal_models.iterrows():
    target = row["target"]
    subset = seasonal_addon_predictions.loc[
        seasonal_addon_predictions["target"].eq(target)
        & seasonal_addon_predictions["seasonal_model"].eq(row["seasonal_model"])
    ].copy()
    series = weekly_series(subset.rename(columns={"seasonal_corrected_residual": "seasonal_corrected"}), "seasonal_corrected", "seasonal_corrected")
    acf_df = acf_pairwise(series, max_lag=60)
    acf_df["target"] = target
    acf_df["correction"] = "current_regime_plus_seasonal"
    acf_df["model"] = row["seasonal_model"]
    seasonal_acf_rows.append(acf_df)
    lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
    lb["target"] = target
    lb["correction"] = "current_regime_plus_seasonal"
    lb["model"] = row["seasonal_model"]
    seasonal_ljung_rows.append(lb)
seasonal_acf_table = pd.concat(seasonal_acf_rows, ignore_index=True) if seasonal_acf_rows else pd.DataFrame()
seasonal_ljung_box_table = pd.concat(seasonal_ljung_rows, ignore_index=True) if seasonal_ljung_rows else pd.DataFrame()
display(seasonal_ljung_box_table.head(36))


## Regime and Training-Window Diagnostics

This is the primary decision section. It asks whether the old seasonal pattern is stale and whether the correction layer should use all prior history, 2022+ only, or hybrid shrinkage. The plots come before new model complexity: corridor/regime stability, corridor-year persistence, movie-mix confounding, release composition, actual share movement, share-error movement, and forecast-useful lag-52 structure.


In [ ]:
def add_regime_diagnostic_columns(df):
    out = df.copy()
    out["regime_period"] = np.select(
        [out["release_year"].between(2015, 2019), out["release_year"].ge(2022)],
        ["2015-2019", "2022+"],
        default="other",
    )
    out["compact_movie_type_diag"] = np.select(
        [
            out["is_family_animation"].fillna(False),
            out["is_horror"].fillna(False),
            out["is_franchise"].fillna(False) | out["is_fan_driven"].fillna(False),
        ],
        ["family/animation", "horror", "franchise/fan"],
        default="other",
    )
    return out


residual_table = add_regime_diagnostic_columns(residual_table)
REGIME_ORDER = ["2015-2019", "2022+"]
CORRIDOR_ORDER = ["Jan-Mar", "Apr-May", "summer", "weak fall", "Thanksgiving", "Christmas/New Year"]
MOVIE_TYPE_ORDER = ["franchise/fan", "family/animation", "horror", "other"]
DIAG_TARGET_LABELS = {"e_a": "e_a log(Fri/Sat)", "e_b": "e_b log(Sun/Sat)", "r_sun_sat": "r_sun_sat after-Sat Sunday"}


def mean_ci_stats(g, target):
    vals = g[target].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(vals)
    mean = vals.mean() if n else np.nan
    sd = vals.std(ddof=1) if n > 1 else np.nan
    se = sd / np.sqrt(n) if n > 1 else np.nan
    return {
        "n": n,
        "mean": mean,
        "se": se,
        "ci_low": mean - 1.96 * se if pd.notna(se) else np.nan,
        "ci_high": mean + 1.96 * se if pd.notna(se) else np.nan,
        "median": vals.median() if n else np.nan,
        "mae": vals.abs().mean() if n else np.nan,
        "rmse": np.sqrt(np.nanmean(np.square(vals))) if n else np.nan,
        "sd": sd,
        "q10": vals.quantile(0.10) if n else np.nan,
        "q25": vals.quantile(0.25) if n else np.nan,
        "q75": vals.quantile(0.75) if n else np.nan,
        "q90": vals.quantile(0.90) if n else np.nan,
    }


def grouped_target_summary(df, group_cols, targets=TARGETS, min_n=1):
    rows = []
    for target in targets:
        for keys, g in df.groupby(group_cols, observed=True):
            if not isinstance(keys, tuple):
                keys = (keys,)
            stats_row = mean_ci_stats(g, target)
            if stats_row["n"] < min_n:
                continue
            row = {col: key for col, key in zip(group_cols, keys)}
            row["target"] = target
            row.update(stats_row)
            rows.append(row)
    return pd.DataFrame(rows)


regime_sample = residual_table.loc[residual_table["regime_period"].isin(REGIME_ORDER)].copy()

# Table 1: residual mean and uncertainty by target x regime.
residual_regime_summary = grouped_target_summary(regime_sample, ["regime_period"])
residual_regime_summary = residual_regime_summary[[
    "target", "regime_period", "n", "mean", "median", "mae", "rmse", "sd", "q10", "q25", "q75", "q90"
]].sort_values(["target", "regime_period"]).reset_index(drop=True)
display(residual_regime_summary)

# Table 2: corridor-regime summary.
corridor_regime_summary = grouped_target_summary(regime_sample, ["release_corridor", "regime_period"])
corridor_regime_summary["release_corridor"] = pd.Categorical(corridor_regime_summary["release_corridor"], CORRIDOR_ORDER, ordered=True)
corridor_regime_summary["regime_period"] = pd.Categorical(corridor_regime_summary["regime_period"], REGIME_ORDER, ordered=True)
corridor_regime_summary = corridor_regime_summary.sort_values(["target", "release_corridor", "regime_period"]).reset_index(drop=True)
display(corridor_regime_summary.head(36))

# Plot 1: regime-comparison seasonal plot by corridor.
def plot_corridor_regime_means(summary):
    for target in TARGETS:
        plot = summary.loc[summary["target"].eq(target)].dropna(subset=["mean"]).copy()
        if plot.empty:
            continue
        plot["release_corridor"] = pd.Categorical(plot["release_corridor"], CORRIDOR_ORDER, ordered=True)
        x = np.arange(len(CORRIDOR_ORDER))
        width = 0.36
        fig, ax = plt.subplots(figsize=(12, 5))
        for j, regime in enumerate(REGIME_ORDER):
            sub = plot.loc[plot["regime_period"].astype(str).eq(regime)].set_index("release_corridor").reindex(CORRIDOR_ORDER)
            offset = (j - 0.5) * width
            yerr = np.vstack([(sub["mean"] - sub["ci_low"]).fillna(0), (sub["ci_high"] - sub["mean"]).fillna(0)])
            ax.bar(x + offset, sub["mean"], width=width, label=regime, alpha=0.85)
            ax.errorbar(x + offset, sub["mean"], yerr=yerr, fmt="none", color="#222222", capsize=3)
            for xi, (_, row) in zip(x + offset, sub.iterrows()):
                if pd.notna(row["mean"]):
                    ax.text(xi, row["mean"], f"n={int(row['n'])}", ha="center", va="bottom" if row["mean"] >= 0 else "top", fontsize=8)
        ax.axhline(0, color="#222222", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(CORRIDOR_ORDER, rotation=25, ha="right")
        ax.set_title(f"Regime comparison by corridor: {DIAG_TARGET_LABELS[target]}")
        ax.set_ylabel("mean log residual")
        ax.legend()
        fig.tight_layout()
        plt.show()


plot_corridor_regime_means(corridor_regime_summary)

# Plot 2 / table: corridor x year heatmap with sample-size overlay.
corridor_year_summary = grouped_target_summary(residual_table, ["release_corridor", "release_year"])
corridor_year_summary["release_corridor"] = pd.Categorical(corridor_year_summary["release_corridor"], CORRIDOR_ORDER, ordered=True)
corridor_year_summary = corridor_year_summary.sort_values(["target", "release_corridor", "release_year"]).reset_index(drop=True)
display(corridor_year_summary.head(36))


def plot_corridor_year_heatmap(summary, min_n=5):
    years = sorted(y for y in summary["release_year"].dropna().astype(int).unique() if y not in EXCLUDED_RELEASE_YEARS)
    for target in TARGETS:
        sub = summary.loc[summary["target"].eq(target)].copy()
        pivot = sub.pivot(index="release_corridor", columns="release_year", values="mean").reindex(CORRIDOR_ORDER)[years]
        n_pivot = sub.pivot(index="release_corridor", columns="release_year", values="n").reindex(CORRIDOR_ORDER)[years]
        fig, ax = plt.subplots(figsize=(14, 4.8))
        masked = pivot.mask(n_pivot.lt(min_n))
        vmax = np.nanpercentile(np.abs(masked.values), 90) if np.isfinite(masked.values).any() else 0.2
        im = ax.imshow(masked, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
        ax.set_yticks(np.arange(len(CORRIDOR_ORDER)))
        ax.set_yticklabels(CORRIDOR_ORDER)
        ax.set_xticks(np.arange(len(years)))
        ax.set_xticklabels(years, rotation=45, ha="right")
        for i, corridor in enumerate(CORRIDOR_ORDER):
            for j, year in enumerate(years):
                n = n_pivot.loc[corridor, year] if year in n_pivot.columns else np.nan
                val = pivot.loc[corridor, year] if year in pivot.columns else np.nan
                if pd.notna(n):
                    label = f"n={int(n)}" if n < min_n or pd.isna(val) else f"{val:+.2f}\nn={int(n)}"
                    ax.text(j, i, label, ha="center", va="center", fontsize=7, color="#222222")
        ax.set_title(f"Corridor x year residual mean with n overlay: {DIAG_TARGET_LABELS[target]}")
        fig.colorbar(im, ax=ax, label="mean residual")
        fig.tight_layout()
        plt.show()


plot_corridor_year_heatmap(corridor_year_summary)

# Plot 3: seasonal subseries by corridor.
def plot_seasonal_subseries_by_corridor(summary):
    for target in TARGETS:
        sub = summary.loc[summary["target"].eq(target)].copy()
        years = sorted(sub["release_year"].dropna().astype(int).unique())
        fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
        for ax, corridor in zip(axes.ravel(), CORRIDOR_ORDER):
            g = sub.loc[sub["release_corridor"].astype(str).eq(corridor)].sort_values("release_year")
            ax.plot(g["release_year"], g["mean"], marker="o", linewidth=1.5)
            pre_mean = corridor_regime_summary.loc[
                corridor_regime_summary["target"].eq(target)
                & corridor_regime_summary["release_corridor"].astype(str).eq(corridor)
                & corridor_regime_summary["regime_period"].astype(str).eq("2015-2019"),
                "mean",
            ]
            post_mean = corridor_regime_summary.loc[
                corridor_regime_summary["target"].eq(target)
                & corridor_regime_summary["release_corridor"].astype(str).eq(corridor)
                & corridor_regime_summary["regime_period"].astype(str).eq("2022+"),
                "mean",
            ]
            if len(pre_mean):
                ax.axhline(pre_mean.iloc[0], color="#4c78a8", linestyle="--", linewidth=1.2, label="2015-2019 mean")
            if len(post_mean):
                ax.axhline(post_mean.iloc[0], color="#e45756", linestyle="--", linewidth=1.2, label="2022+ mean")
            ax.axhline(0, color="#222222", linewidth=0.8)
            ax.set_title(corridor)
            ax.grid(True, alpha=0.25)
        handles, labels = axes.ravel()[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="upper right")
        fig.suptitle(f"Seasonal subseries by corridor: {DIAG_TARGET_LABELS[target]}")
        fig.tight_layout()
        plt.show()


plot_seasonal_subseries_by_corridor(corridor_year_summary)

# Plot 4: rolling-origin residual mean by training window.
def shrink_group_means(train, target, group_col, lam=100):
    clean = train.dropna(subset=[target]).copy()
    if clean.empty:
        return pd.Series(dtype="float64"), np.nan, pd.Series(dtype="float64")
    mu = clean[target].mean()
    stats_by_group = clean.groupby(group_col, observed=True)[target].agg(["mean", "count"])
    stats_by_group["shrunk_mean"] = mu + (stats_by_group["count"] / (stats_by_group["count"] + lam)) * (stats_by_group["mean"] - mu)
    return stats_by_group["shrunk_mean"], mu, stats_by_group["count"]


def rolling_correction_estimates(df, targets=TARGETS, lam=100):
    rows = []
    years = sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= ROLLING_START_YEAR)
    for target in targets:
        for forecast_year in years:
            prior = df.loc[df["release_year"] < forecast_year].copy()
            if len(prior) < MIN_TRAIN_MOVIES:
                continue
            windows = {
                "all_prior": prior,
                "last5y": prior.loc[prior["release_year"].ge(forecast_year - 5)],
                "post_2022_only": prior.loc[prior["release_year"].ge(2022)],
            }
            all_est, _, _ = shrink_group_means(windows["all_prior"], target, "release_corridor", lam)
            post_est, _, post_n = shrink_group_means(windows["post_2022_only"], target, "release_corridor", lam)
            for window_name, train_window in windows.items():
                est, mu, counts = shrink_group_means(train_window, target, "release_corridor", lam)
                for corridor in CORRIDOR_ORDER:
                    rows.append({
                        "target": target,
                        "forecast_year": forecast_year,
                        "release_corridor": corridor,
                        "training_window": window_name,
                        "estimated_correction": est.get(corridor, mu if pd.notna(mu) else 0.0),
                        "train_n": int(counts.get(corridor, 0)) if len(counts) else 0,
                    })
            for corridor in CORRIDOR_ORDER:
                n_post = post_n.get(corridor, 0) if len(post_n) else 0
                w = n_post / (n_post + lam)
                all_val = all_est.get(corridor, 0.0) if len(all_est) else 0.0
                post_val = post_est.get(corridor, all_val) if len(post_est) else all_val
                rows.append({
                    "target": target,
                    "forecast_year": forecast_year,
                    "release_corridor": corridor,
                    "training_window": "hybrid_post2022_shrink_to_all_prior",
                    "estimated_correction": all_val + w * (post_val - all_val),
                    "train_n": int(n_post),
                })
    return pd.DataFrame(rows)


training_window_correction_estimates = rolling_correction_estimates(residual_table)
display(training_window_correction_estimates.head())


def plot_rolling_correction_estimates(estimates):
    for target in TARGETS:
        fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
        for ax, corridor in zip(axes.ravel(), CORRIDOR_ORDER):
            sub = estimates.loc[estimates["target"].eq(target) & estimates["release_corridor"].eq(corridor)]
            for window_name, g in sub.groupby("training_window", observed=True):
                ax.plot(g["forecast_year"], g["estimated_correction"], marker="o", linewidth=1.2, label=window_name)
            ax.axhline(0, color="#222222", linewidth=0.8)
            ax.set_title(corridor)
        handles, labels = axes.ravel()[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=4)
        fig.suptitle(f"Rolling-origin correction estimate by training window: {DIAG_TARGET_LABELS[target]}")
        fig.tight_layout(rect=[0, 0, 1, 0.92])
        plt.show()


plot_rolling_correction_estimates(training_window_correction_estimates)

# Table 3 / Plot 5: post-2022 corridor x movie-type residual summary.
post_2022 = residual_table.loc[residual_table["release_year"].ge(2022)].copy()
post_2022_corridor_movie_summary = grouped_target_summary(post_2022, ["release_corridor", "compact_movie_type_diag"])
post_2022_corridor_movie_summary["release_corridor"] = pd.Categorical(post_2022_corridor_movie_summary["release_corridor"], CORRIDOR_ORDER, ordered=True)
post_2022_corridor_movie_summary["compact_movie_type_diag"] = pd.Categorical(post_2022_corridor_movie_summary["compact_movie_type_diag"], MOVIE_TYPE_ORDER, ordered=True)
post_2022_corridor_movie_summary = post_2022_corridor_movie_summary.sort_values(["target", "release_corridor", "compact_movie_type_diag"]).reset_index(drop=True)
display(post_2022_corridor_movie_summary.head(48))


def plot_post_2022_corridor_movie(summary):
    for target in TARGETS:
        plot = summary.loc[summary["target"].eq(target)].copy()
        if plot.empty:
            continue
        fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
        for ax, corridor in zip(axes.ravel(), CORRIDOR_ORDER):
            sub = plot.loc[plot["release_corridor"].astype(str).eq(corridor)].set_index("compact_movie_type_diag").reindex(MOVIE_TYPE_ORDER)
            x = np.arange(len(MOVIE_TYPE_ORDER))
            yerr = np.vstack([(sub["mean"] - sub["ci_low"]).fillna(0), (sub["ci_high"] - sub["mean"]).fillna(0)])
            ax.bar(x, sub["mean"], color="#4c78a8", alpha=0.85)
            ax.errorbar(x, sub["mean"], yerr=yerr, fmt="none", color="#222222", capsize=3)
            for xi, (_, row) in zip(x, sub.iterrows()):
                if pd.notna(row["mean"]):
                    ax.text(xi, row["mean"], f"n={int(row['n'])}", ha="center", va="bottom" if row["mean"] >= 0 else "top", fontsize=8)
            ax.axhline(0, color="#222222", linewidth=0.8)
            ax.set_title(corridor)
            ax.set_xticks(x)
            ax.set_xticklabels(MOVIE_TYPE_ORDER, rotation=35, ha="right")
        fig.suptitle(f"Post-2022 corridor x movie type residuals: {DIAG_TARGET_LABELS[target]}")
        fig.tight_layout()
        plt.show()


plot_post_2022_corridor_movie(post_2022_corridor_movie_summary)

# Table 4 / Plot 6: composition shift by regime.
composition_base = regime_sample.copy()
composition_base["expected_ow_weight"] = composition_base["latest_estimate_mid_usd"].where(composition_base["latest_estimate_mid_usd"].gt(0), np.nan)
count_mix = composition_base.groupby(["release_corridor", "regime_period", "compact_movie_type_diag"], observed=True).size().rename("movie_count").reset_index()
count_mix["movie_count_share"] = count_mix["movie_count"] / count_mix.groupby(["release_corridor", "regime_period"], observed=True)["movie_count"].transform("sum")
ow_mix = composition_base.dropna(subset=["expected_ow_weight"]).groupby(["release_corridor", "regime_period", "compact_movie_type_diag"], observed=True)["expected_ow_weight"].sum().rename("expected_ow_sum").reset_index()
ow_mix["expected_ow_share"] = ow_mix["expected_ow_sum"] / ow_mix.groupby(["release_corridor", "regime_period"], observed=True)["expected_ow_sum"].transform("sum")
composition_medians = composition_base.groupby(["release_corridor", "regime_period"], observed=True).agg(
    n=("release_run_id", "count"),
    median_expected_ow=("latest_estimate_mid_usd", "median"),
    median_theaters=("opening_weekend_theaters", "median"),
).reset_index()
composition_shift_table = count_mix.merge(ow_mix, on=["release_corridor", "regime_period", "compact_movie_type_diag"], how="outer").merge(composition_medians, on=["release_corridor", "regime_period"], how="left")
composition_shift_table["release_corridor"] = pd.Categorical(composition_shift_table["release_corridor"], CORRIDOR_ORDER, ordered=True)
composition_shift_table["regime_period"] = pd.Categorical(composition_shift_table["regime_period"], REGIME_ORDER, ordered=True)
composition_shift_table["compact_movie_type_diag"] = pd.Categorical(composition_shift_table["compact_movie_type_diag"], MOVIE_TYPE_ORDER, ordered=True)
composition_shift_table = composition_shift_table.sort_values(["release_corridor", "regime_period", "compact_movie_type_diag"]).reset_index(drop=True)
display(composition_shift_table.head(48))


def plot_composition_shift(table, value_col, title):
    for regime in REGIME_ORDER:
        pivot = table.loc[table["regime_period"].astype(str).eq(regime)].pivot_table(
            index="release_corridor", columns="compact_movie_type_diag", values=value_col, aggfunc="sum", observed=True
        ).reindex(CORRIDOR_ORDER).fillna(0)
        pivot = pivot.reindex(columns=MOVIE_TYPE_ORDER).fillna(0)
        ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 4.5), width=0.82)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        ax.set_title(f"{title}: {regime}")
        ax.set_xlabel("corridor")
        ax.set_ylabel("share")
        ax.legend(title="movie type", loc="upper left", bbox_to_anchor=(1.01, 1.0))
        plt.tight_layout()
        plt.show()


plot_composition_shift(composition_shift_table, "movie_count_share", "Release composition by count")
plot_composition_shift(composition_shift_table, "expected_ow_share", "Release composition by expected OW")

# Table 5: training-window sufficiency by target x corridor x movie_type.
def training_window_sufficiency(df):
    rows = []
    for target in TARGETS:
        for (corridor, movie_type), g in df.groupby(["release_corridor", "compact_movie_type_diag"], observed=True):
            valid = g.dropna(subset=[target])
            n_all = len(valid)
            n_pre = valid["release_year"].between(2015, 2019).sum()
            n_post = valid["release_year"].ge(2022).sum()
            latest_year = int(df["release_year"].max())
            n_last5 = valid["release_year"].ge(latest_year - 4).sum()
            rows.append({
                "target": target,
                "release_corridor": corridor,
                "compact_movie_type_diag": movie_type,
                "n_all": n_all,
                "n_2015_2019": int(n_pre),
                "n_2022_plus": int(n_post),
                "n_last5y": int(n_last5),
                "n_2022_share_of_all": n_post / n_all if n_all else np.nan,
                "post_2022_point_feasible_n20": n_post >= 20,
            })
    out = pd.DataFrame(rows)
    out["release_corridor"] = pd.Categorical(out["release_corridor"], CORRIDOR_ORDER, ordered=True)
    out["compact_movie_type_diag"] = pd.Categorical(out["compact_movie_type_diag"], MOVIE_TYPE_ORDER, ordered=True)
    return out.sort_values(["target", "release_corridor", "compact_movie_type_diag"]).reset_index(drop=True)


training_window_sufficiency_table = training_window_sufficiency(residual_table)
display(training_window_sufficiency_table.head(48))

# Table 6: correction stability by target x corridor.
def pooled_sd(a, b):
    a = pd.Series(a).dropna()
    b = pd.Series(b).dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))


def correction_stability(df):
    rows = []
    for target in TARGETS:
        for corridor in CORRIDOR_ORDER:
            pre = df.loc[df["release_corridor"].eq(corridor) & df["release_year"].between(2015, 2019), target].dropna()
            post = df.loc[df["release_corridor"].eq(corridor) & df["release_year"].ge(2022), target].dropna()
            diff = post.mean() - pre.mean() if len(pre) and len(post) else np.nan
            se_diff = np.sqrt(pre.var(ddof=1) / len(pre) + post.var(ddof=1) / len(post)) if len(pre) > 1 and len(post) > 1 else np.nan
            sd_pool = pooled_sd(pre, post)
            rows.append({
                "target": target,
                "release_corridor": corridor,
                "n_2015_2019": len(pre),
                "n_2022_plus": len(post),
                "mean_2015_2019": pre.mean() if len(pre) else np.nan,
                "mean_2022_plus": post.mean() if len(post) else np.nan,
                "difference_2022_minus_pre": diff,
                "ci_difference_low": diff - 1.96 * se_diff if pd.notna(se_diff) else np.nan,
                "ci_difference_high": diff + 1.96 * se_diff if pd.notna(se_diff) else np.nan,
                "sign_same": np.sign(pre.mean()) == np.sign(post.mean()) if len(pre) and len(post) else np.nan,
                "effect_size": diff / sd_pool if pd.notna(sd_pool) and sd_pool else np.nan,
            })
    return pd.DataFrame(rows)


correction_stability_table = correction_stability(residual_table)
display(correction_stability_table)

# Plot 7: actual share curves by corridor and regime.
share_regime_actual_summary = regime_sample.groupby(["release_corridor", "regime_period"], observed=True).agg(
    n=("release_run_id", "count"),
    s_fri=("s_fri", "mean"),
    s_sat=("s_sat", "mean"),
    s_sun=("s_sun", "mean"),
).reset_index()
share_regime_actual_long = share_regime_actual_summary.melt(
    id_vars=["release_corridor", "regime_period", "n"],
    value_vars=SHARE_COLS,
    var_name="day_share",
    value_name="actual_share",
)
display(share_regime_actual_long.head())

def plot_actual_share_curves(summary):
    for regime in REGIME_ORDER:
        fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharey=True)
        for ax, corridor in zip(axes.ravel(), CORRIDOR_ORDER):
            sub = summary.loc[summary["regime_period"].astype(str).eq(regime) & summary["release_corridor"].astype(str).eq(corridor)].set_index("day_share").reindex(SHARE_COLS)
            ax.plot(["Fri", "Sat", "Sun"], sub["actual_share"], marker="o", linewidth=2)
            n = sub["n"].dropna().max()
            ax.set_title(f"{corridor} (n={int(n) if pd.notna(n) else 0})")
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        fig.suptitle(f"Actual weekend share curves by corridor: {regime}")
        fig.tight_layout()
        plt.show()


plot_actual_share_curves(share_regime_actual_long)

# Plot 8: predicted-vs-actual share error by corridor/regime/model state.
def corrected_shape_predictions_for_state(state_name):
    base = residual_table[["release_run_id", "release_corridor", "regime_period", "opening_weekend_start", "release_year", "s_fri", "s_sat", "s_sun", "pred_log_fri_sat", "pred_log_sun_sat"]].copy()
    base["model_state"] = state_name
    if state_name == "baseline":
        pred_a = base["pred_log_fri_sat"]
        pred_b = base["pred_log_sun_sat"]
    elif state_name == "current_regime_shape":
        ea = frozen_current_regime_predictions.loc[frozen_current_regime_predictions["target"].eq("e_a"), ["release_run_id", "predicted_residual"]].rename(columns={"predicted_residual": "pred_e_a"})
        eb = frozen_current_regime_predictions.loc[frozen_current_regime_predictions["target"].eq("e_b"), ["release_run_id", "predicted_residual"]].rename(columns={"predicted_residual": "pred_e_b"})
        base = base.merge(ea, on="release_run_id", how="left").merge(eb, on="release_run_id", how="left")
        pred_a = base["pred_log_fri_sat"] + base["pred_e_a"].fillna(0.0)
        pred_b = base["pred_log_sun_sat"] + base["pred_e_b"].fillna(0.0)
    elif state_name == "seasonal_shape":
        ea_best = seasonal_addon_summary.loc[seasonal_addon_summary["target"].eq("e_a")].sort_values("seasonal_corrected_mae").head(1)["seasonal_model"].iloc[0]
        eb_best = seasonal_addon_summary.loc[seasonal_addon_summary["target"].eq("e_b")].sort_values("seasonal_corrected_mae").head(1)["seasonal_model"].iloc[0]
        ea = seasonal_addon_predictions.loc[seasonal_addon_predictions["target"].eq("e_a") & seasonal_addon_predictions["seasonal_model"].eq(ea_best), ["release_run_id", "total_predicted_residual"]].rename(columns={"total_predicted_residual": "pred_e_a"})
        eb = seasonal_addon_predictions.loc[seasonal_addon_predictions["target"].eq("e_b") & seasonal_addon_predictions["seasonal_model"].eq(eb_best), ["release_run_id", "total_predicted_residual"]].rename(columns={"total_predicted_residual": "pred_e_b"})
        base = base.merge(ea, on="release_run_id", how="left").merge(eb, on="release_run_id", how="left")
        pred_a = base["pred_log_fri_sat"] + base["pred_e_a"].fillna(0.0)
        pred_b = base["pred_log_sun_sat"] + base["pred_e_b"].fillna(0.0)
    else:
        raise ValueError(state_name)
    pred_fri, pred_sat, pred_sun = shares_from_log_ratios(pred_a, pred_b)
    base["pred_s_fri"] = pred_fri
    base["pred_s_sat"] = pred_sat
    base["pred_s_sun"] = pred_sun
    for actual_col, pred_col, day in [("s_fri", "pred_s_fri", "Friday"), ("s_sat", "pred_s_sat", "Saturday"), ("s_sun", "pred_s_sun", "Sunday")]:
        base[f"share_error_{day}"] = base[actual_col] - base[pred_col]
    return base


share_error_states = pd.concat([corrected_shape_predictions_for_state(s) for s in ["baseline", "current_regime_shape", "seasonal_shape"]], ignore_index=True)
share_error_regime_summary = share_error_states.loc[share_error_states["regime_period"].isin(REGIME_ORDER)].groupby(["model_state", "regime_period", "release_corridor"], observed=True).agg(
    n=("release_run_id", "count"),
    friday_share_error=("share_error_Friday", "mean"),
    saturday_share_error=("share_error_Saturday", "mean"),
    sunday_share_error=("share_error_Sunday", "mean"),
).reset_index()
display(share_error_regime_summary.head())


def plot_share_error_by_state(summary):
    for regime in REGIME_ORDER:
        for day_col, day_label in [("friday_share_error", "Friday"), ("saturday_share_error", "Saturday"), ("sunday_share_error", "Sunday")]:
            pivot = summary.loc[summary["regime_period"].astype(str).eq(regime)].pivot_table(
                index="release_corridor", columns="model_state", values=day_col, observed=True
            ).reindex(CORRIDOR_ORDER)
            ax = pivot.plot(kind="bar", figsize=(11, 4.5), width=0.82)
            ax.axhline(0, color="#222222", linewidth=1)
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
            ax.set_title(f"Actual minus predicted {day_label} share by corridor: {regime}")
            ax.set_xlabel("corridor")
            ax.set_ylabel("share error")
            plt.tight_layout()
            plt.show()


plot_share_error_by_state(share_error_regime_summary)

# Table 7: diagnostic ACF/Ljung-Box table by correction state.
def ljung_pivot(df, state):
    out = df.loc[df["lag"].isin([1, 4, 8, 26, 52])].copy()
    out["correction_state"] = state
    return out.pivot_table(index=["target", "correction_state"], columns="lag", values="p_value", aggfunc="first").rename(columns={1: "p_lag1", 4: "p_lag4", 8: "p_lag8", 26: "p_lag26", 52: "p_lag52"}).reset_index()

baseline_ljung_diag = ljung_box_table.loc[
    ljung_box_table["subset"].eq("all_eligible") & ljung_box_table["transform"].eq("mean")
].copy()
current_ljung_diag = current_regime_ljung_box_table.copy()
seasonal_ljung_diag = seasonal_ljung_box_table.copy()
diagnostic_acf_ljung_summary = pd.concat([
    ljung_pivot(baseline_ljung_diag, "baseline"),
    ljung_pivot(current_ljung_diag, "current_regime"),
    ljung_pivot(seasonal_ljung_diag, "seasonal_addon"),
], ignore_index=True)
display(diagnostic_acf_ljung_summary)

# Plot 9: ACF by target/correction state at selected lags.
def acf_selected_table():
    base = acf_table.loc[acf_table["subset"].eq("all_eligible") & acf_table["transform"].eq("mean")].copy()
    base["correction_state"] = "baseline"
    cur = current_regime_acf_table.copy()
    cur["correction_state"] = "current_regime"
    seas = seasonal_acf_table.copy()
    seas["correction_state"] = "seasonal_addon"
    cols = ["target", "correction_state", "lag", "acf"]
    return pd.concat([base[cols], cur[cols], seas[cols]], ignore_index=True)

selected_acf_summary = acf_selected_table().loc[lambda x: x["lag"].isin([1, 2, 4, 8, 13, 26, 52])]
display(selected_acf_summary.head())

for target in TARGETS:
    plot = selected_acf_summary.loc[selected_acf_summary["target"].eq(target)]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for state, g in plot.groupby("correction_state", observed=True):
        ax.plot(g["lag"], g["acf"], marker="o", linewidth=1.8, label=state)
    ax.axhline(0, color="#222222", linewidth=1)
    ax.set_title(f"Selected-lag ACF by correction state: {DIAG_TARGET_LABELS[target]}")
    ax.set_xlabel("lag")
    ax.set_ylabel("ACF")
    ax.legend()
    plt.tight_layout()
    plt.show()

# Plot 10: seasonal lag scatterplot after current-regime correction.
def weekly_corrected_series_from_predictions(preds, target):
    subset = preds.loc[preds["target"].eq(target)].copy()
    subset["week_start"] = pd.to_datetime(subset["opening_weekend_start"])
    out = subset.groupby("week_start", observed=True).agg(
        corrected_residual=("corrected_residual", "mean"),
        release_year=("release_year", "max"),
        n=("release_run_id", "count"),
    ).reset_index().sort_values("week_start")
    out["lag_52_week"] = out["week_start"] - pd.DateOffset(weeks=52)
    lag = out[["week_start", "corrected_residual"]].rename(columns={"week_start": "lag_52_week", "corrected_residual": "lag52_corrected_residual"})
    return out.merge(lag, on="lag_52_week", how="left")

seasonal_lag_scatter_summary_rows = []
for target in TARGETS:
    model = FROZEN_CURRENT_REGIME_MODELS[target]
    preds = frozen_current_regime_predictions.loc[frozen_current_regime_predictions["target"].eq(target) & frozen_current_regime_predictions["current_regime_model"].eq(model)]
    weekly = weekly_corrected_series_from_predictions(preds, target).dropna(subset=["lag52_corrected_residual", "corrected_residual"])
    for period_name, mask in {"all_years": pd.Series(True, index=weekly.index), "post_2022": weekly["release_year"].ge(2022)}.items():
        sub = weekly.loc[mask]
        slope = np.nan
        corr = np.nan
        if len(sub) >= 8 and np.square(sub["lag52_corrected_residual"]).sum() > 0:
            slope = (sub["lag52_corrected_residual"] * sub["corrected_residual"]).sum() / np.square(sub["lag52_corrected_residual"]).sum()
            corr = sub[["lag52_corrected_residual", "corrected_residual"]].corr().iloc[0, 1]
        seasonal_lag_scatter_summary_rows.append({"target": target, "period": period_name, "n": len(sub), "slope": slope, "corr": corr})
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        ax.scatter(sub["lag52_corrected_residual"], sub["corrected_residual"], alpha=0.75)
        if pd.notna(slope):
            xs = np.linspace(sub["lag52_corrected_residual"].min(), sub["lag52_corrected_residual"].max(), 50)
            ax.plot(xs, slope * xs, color="#e45756", linewidth=2)
        ax.axhline(0, color="#222222", linewidth=0.8)
        ax.axvline(0, color="#222222", linewidth=0.8)
        ax.set_title(f"Lag-52 scatter after current-regime correction: {target}, {period_name}")
        ax.set_xlabel("corrected residual at w-52")
        ax.set_ylabel("corrected residual at w")
        plt.tight_layout()
        plt.show()
seasonal_lag_scatter_summary = pd.DataFrame(seasonal_lag_scatter_summary_rows)
display(seasonal_lag_scatter_summary)

# Table 8: operational validation table for training-period choice on e_b and r_sun_sat.
def group_shrink_prediction(train, test, target, strategy, group_col="release_corridor", lam=100):
    train = train.dropna(subset=[target]).copy()
    if train.empty:
        return pd.Series(0.0, index=test.index)
    if strategy == "baseline_no_correction":
        return pd.Series(0.0, index=test.index)
    if strategy == "all_prior_correction":
        window = train
        stats, mu, _ = shrink_group_means(window, target, group_col, lam)
        return test[group_col].map(stats).fillna(mu if pd.notna(mu) else 0.0).astype(float)
    if strategy == "last5y_correction":
        max_test_year = int(test["release_year"].max())
        window = train.loc[train["release_year"].ge(max_test_year - 5)]
        if len(window) < MIN_GROUP_N:
            window = train
        stats, mu, _ = shrink_group_means(window, target, group_col, lam)
        return test[group_col].map(stats).fillna(mu if pd.notna(mu) else 0.0).astype(float)
    if strategy == "post_2022_only_correction":
        window = train.loc[train["release_year"].ge(2022)]
        if len(window) < MIN_GROUP_N:
            window = train
        stats, mu, _ = shrink_group_means(window, target, group_col, lam)
        return test[group_col].map(stats).fillna(mu if pd.notna(mu) else 0.0).astype(float)
    if strategy == "hybrid_post2022_shrink_to_all_prior":
        all_stats, all_mu, _ = shrink_group_means(train, target, group_col, lam)
        post = train.loc[train["release_year"].ge(2022)]
        post_stats, _, post_counts = shrink_group_means(post, target, group_col, lam)
        all_pred = test[group_col].map(all_stats).fillna(all_mu if pd.notna(all_mu) else 0.0).astype(float)
        post_pred = test[group_col].map(post_stats).fillna(all_pred).astype(float) if len(post_stats) else all_pred
        n_post = test[group_col].map(post_counts).fillna(0.0).astype(float) if len(post_counts) else pd.Series(0.0, index=test.index)
        w = n_post / (n_post + lam)
        return all_pred + w * (post_pred - all_pred)
    raise ValueError(strategy)


def target_dollar_error_for_predictions(g, target, pred):
    if target == "e_b":
        pred_a = g["pred_log_fri_sat"]
        pred_b = g["pred_log_sun_sat"] + pred
        fri_share, sat_share, sun_share = shares_from_log_ratios(pred_a, pred_b)
        return pd.concat([
            (g["friday_gross_usd"] - g["opening_weekend_gross_usd"] * fri_share).abs(),
            (g["saturday_gross_usd"] - g["opening_weekend_gross_usd"] * sat_share).abs(),
            (g["sunday_gross_usd"] - g["opening_weekend_gross_usd"] * sun_share).abs(),
        ], axis=1).mean(axis=1)
    if target == "r_sun_sat":
        corrected_sunday = g["pred_sunday_gross_base"] * np.exp(pred)
        return (g["sunday_gross_usd"] - corrected_sunday).abs()
    raise ValueError(target)


def rolling_operational_validation(df, targets=("e_b", "r_sun_sat"), strategies=None, lam=100):
    if strategies is None:
        strategies = ["baseline_no_correction", "all_prior_correction", "last5y_correction", "post_2022_only_correction", "hybrid_post2022_shrink_to_all_prior"]
    frames = []
    for target in targets:
        for test_year in sorted(y for y in df["release_year"].dropna().astype(int).unique() if y >= ROLLING_START_YEAR):
            train = df.loc[df["release_year"] < test_year].copy()
            test = df.loc[df["release_year"].eq(test_year)].copy()
            if len(train) < MIN_TRAIN_MOVIES or test.empty:
                continue
            for strategy in strategies:
                pred = group_shrink_prediction(train, test, target, strategy, lam=lam)
                out = test[["release_run_id", "title", "opening_weekend_start", "release_year", "release_corridor", target, "pred_log_fri_sat", "pred_log_sun_sat", "opening_weekend_gross_usd", "friday_gross_usd", "saturday_gross_usd", "sunday_gross_usd", "pred_sunday_gross_base"]].copy()
                out["target"] = target
                out["training_strategy"] = strategy
                out["predicted_residual"] = pred.values
                out["corrected_residual"] = out[target] - out["predicted_residual"]
                out["dollar_abs_error"] = target_dollar_error_for_predictions(out, target, out["predicted_residual"])
                frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


training_window_validation_predictions = rolling_operational_validation(residual_table)

def summarize_training_window_validation(preds):
    rows = []
    for (target, strategy), g in preds.groupby(["target", "training_strategy"], observed=True):
        log_mae = g["corrected_residual"].abs().mean()
        dollar_mae = g["dollar_abs_error"].mean()
        recent = g.loc[g["release_year"].ge(2022)]
        recent_log_mae = recent["corrected_residual"].abs().mean() if len(recent) else np.nan
        recent_dollar_mae = recent["dollar_abs_error"].mean() if len(recent) else np.nan
        base = g.loc[g["training_strategy"].eq("baseline_no_correction")]
        year_improvements = []
        for _, yg in g.groupby("release_year", observed=True):
            if len(yg) < 5:
                continue
            base_y = yg[target].abs().mean()
            corr_y = yg["corrected_residual"].abs().mean()
            year_improvements.append((base_y - corr_y) / base_y if base_y else np.nan)
        year_improvements = pd.Series(year_improvements, dtype="float64").dropna()
        rows.append({
            "target": target,
            "training_strategy": strategy,
            "n": len(g),
            "log_mae": log_mae,
            "dollar_mae": dollar_mae,
            "post_2022_log_mae": recent_log_mae,
            "post_2022_dollar_mae": recent_dollar_mae,
            "year_min_improvement_vs_baseline": year_improvements.min() if len(year_improvements) else np.nan,
            "year_positive_share_vs_baseline": year_improvements.gt(0).mean() if len(year_improvements) else np.nan,
        })
    out = pd.DataFrame(rows)
    baseline = out.loc[out["training_strategy"].eq("baseline_no_correction"), ["target", "log_mae", "dollar_mae", "post_2022_log_mae", "post_2022_dollar_mae"]].rename(columns={
        "log_mae": "baseline_log_mae",
        "dollar_mae": "baseline_dollar_mae",
        "post_2022_log_mae": "baseline_post_2022_log_mae",
        "post_2022_dollar_mae": "baseline_post_2022_dollar_mae",
    })
    out = out.merge(baseline, on="target", how="left")
    out["log_mae_improvement_pct"] = (out["baseline_log_mae"] - out["log_mae"]) / out["baseline_log_mae"]
    out["dollar_mae_improvement_pct"] = (out["baseline_dollar_mae"] - out["dollar_mae"]) / out["baseline_dollar_mae"]
    out["post_2022_log_mae_improvement_pct"] = (out["baseline_post_2022_log_mae"] - out["post_2022_log_mae"]) / out["baseline_post_2022_log_mae"]
    out["post_2022_dollar_mae_improvement_pct"] = (out["baseline_post_2022_dollar_mae"] - out["post_2022_dollar_mae"]) / out["baseline_post_2022_dollar_mae"]
    return out.sort_values(["target", "log_mae"]).reset_index(drop=True)


training_window_validation_summary = summarize_training_window_validation(training_window_validation_predictions)
display(training_window_validation_summary)


## Group-Specific Residual Intervals

Continuous predictors did not earn promotion as point corrections. These interval diagnostics check whether group-specific residual distributions are more useful for uncertainty. Quantiles are shrunk toward global quantiles for small groups.

In [ ]:
def interval_coverage_by_group(df, target, group_col, nominal=0.80):
    alpha = (1 - nominal) / 2
    rows = []
    residual = df[target].replace([np.inf, -np.inf], np.nan).dropna()
    if residual.empty:
        return pd.DataFrame()
    global_lo = residual.quantile(alpha)
    global_hi = residual.quantile(1 - alpha)
    for group, g in df.dropna(subset=[target]).groupby(group_col, dropna=False, observed=True):
        rows.append({
            "target": target,
            "group_col": group_col,
            "group": group,
            "n": len(g),
            "nominal": nominal,
            "global_interval_coverage": g[target].between(global_lo, global_hi).mean(),
            "group_residual_mae": g[target].abs().mean(),
            "group_q_low": g[target].quantile(alpha),
            "group_q_high": g[target].quantile(1 - alpha),
            "global_q_low": global_lo,
            "global_q_high": global_hi,
        })
    return pd.DataFrame(rows)

coverage_tables = []
for target in TARGETS:
    for group_col in ["segment_current_broad", "category_shape_priority", "category_franchise_fan", "category_mpaa", "is_holiday_corridor"]:
        for nominal in [0.50, 0.80, 0.95]:
            coverage_tables.append(interval_coverage_by_group(residual_table, target, group_col, nominal))
interval_group_coverage = pd.concat([t for t in coverage_tables if not t.empty], ignore_index=True)

def shrink_quantiles(table, k=25):
    out = table.copy()
    w = out["n"] / (out["n"] + k)
    out["shrunk_q_low"] = w * out["group_q_low"] + (1 - w) * out["global_q_low"]
    out["shrunk_q_high"] = w * out["group_q_high"] + (1 - w) * out["global_q_high"]
    return out

interval_group_quantiles = shrink_quantiles(interval_group_coverage)

display(interval_group_quantiles.sort_values(["target", "group_col", "nominal", "group"]).head(40))

plot = interval_group_quantiles.loc[
    interval_group_quantiles["nominal"].eq(0.80)
    & interval_group_quantiles["group_col"].eq("category_shape_priority")
].copy()
if not plot.empty:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)
    for ax, target in zip(axes, TARGETS):
        p = plot.loc[plot["target"].eq(target)].sort_values("group_residual_mae")
        ax.barh(p["group"].astype(str), p["group_residual_mae"], color="#4c78a8", alpha=0.85)
        ax.set_title(f"{target}: group residual MAE")
        ax.set_xlabel("absolute log residual")
    plt.show()

## Final Production Validation Plots

No more broad EDA. These checks validate the two production candidates directly: `e_b` with all-prior corridor correction and `r_sun_sat` with hybrid post-2022 shrinkage. The goal is to verify year-by-year stability, corridor share/dollar errors, actual-vs-predicted Sunday dollars, residual distribution behavior, and empirical residual interval coverage before final promotion.


In [ ]:
FINAL_VALIDATION_SPECS = [
    {"target": "e_b", "training_strategy": "all_prior_correction", "label": "e_b all-prior correction"},
    {"target": "r_sun_sat", "training_strategy": "hybrid_post2022_shrink_to_all_prior", "label": "r_sun_sat hybrid correction"},
]


def get_validation_subset(target, strategy):
    return training_window_validation_predictions.loc[
        training_window_validation_predictions["target"].eq(target)
        & training_window_validation_predictions["training_strategy"].eq(strategy)
    ].copy()


def get_validation_baseline(target):
    return training_window_validation_predictions.loc[
        training_window_validation_predictions["target"].eq(target)
        & training_window_validation_predictions["training_strategy"].eq("baseline_no_correction")
    ].copy()


# 1. Year-by-year improvement plot.
def yearly_improvement_table():
    rows = []
    for spec in FINAL_VALIDATION_SPECS:
        target = spec["target"]
        strategy = spec["training_strategy"]
        candidate = get_validation_subset(target, strategy)
        baseline = get_validation_baseline(target)[["release_run_id", "release_year", target, "corrected_residual", "dollar_abs_error"]].rename(
            columns={"corrected_residual": "baseline_corrected_residual", "dollar_abs_error": "baseline_dollar_abs_error"}
        )
        combined = candidate.merge(baseline, on=["release_run_id", "release_year", target], how="inner")
        for year, g in combined.groupby("release_year", observed=True):
            base_log = g["baseline_corrected_residual"].abs().mean()
            cand_log = g["corrected_residual"].abs().mean()
            base_dollar = g["baseline_dollar_abs_error"].mean()
            cand_dollar = g["dollar_abs_error"].mean()
            rows.append({
                "target": target,
                "training_strategy": strategy,
                "model_label": spec["label"],
                "release_year": int(year),
                "n": len(g),
                "log_mae_improvement_pct": (base_log - cand_log) / base_log if base_log else np.nan,
                "dollar_mae_improvement_pct": (base_dollar - cand_dollar) / base_dollar if base_dollar else np.nan,
                "baseline_log_mae": base_log,
                "corrected_log_mae": cand_log,
                "baseline_dollar_mae": base_dollar,
                "corrected_dollar_mae": cand_dollar,
            })
    return pd.DataFrame(rows).sort_values(["target", "release_year"]).reset_index(drop=True)


yearly_production_improvement = yearly_improvement_table()
display(yearly_production_improvement)

for spec in FINAL_VALIDATION_SPECS:
    plot = yearly_production_improvement.loc[yearly_production_improvement["model_label"].eq(spec["label"])]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(plot["release_year"], plot["log_mae_improvement_pct"], marker="o", linewidth=2, label="log MAE improvement")
    ax.plot(plot["release_year"], plot["dollar_mae_improvement_pct"], marker="o", linewidth=2, label="dollar MAE improvement")
    ax.axhline(0, color="#222222", linewidth=1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"Year-by-year improvement vs baseline: {spec['label']}")
    ax.set_xlabel("release year")
    ax.set_ylabel("improvement vs baseline")
    ax.legend()
    plt.tight_layout()
    plt.show()

# 2. 2022+ corridor before/after production errors.
share_cols_for_merge = residual_table[["release_run_id", "s_sun", "release_corridor", "release_year"]].copy()

def eb_sunday_share_error_by_corridor():
    cand = get_validation_subset("e_b", "all_prior_correction").merge(share_cols_for_merge, on=["release_run_id", "release_corridor", "release_year"], how="left")
    cand = cand.loc[cand["release_year"].ge(2022)].copy()
    base_fri, base_sat, base_sun = shares_from_log_ratios(cand["pred_log_fri_sat"], cand["pred_log_sun_sat"])
    corr_fri, corr_sat, corr_sun = shares_from_log_ratios(cand["pred_log_fri_sat"], cand["pred_log_sun_sat"] + cand["predicted_residual"])
    cand["baseline_sunday_share_error"] = cand["s_sun"] - base_sun
    cand["corrected_sunday_share_error"] = cand["s_sun"] - corr_sun
    return cand.groupby("release_corridor", observed=True).agg(
        n=("release_run_id", "count"),
        baseline_sunday_share_error=("baseline_sunday_share_error", "mean"),
        corrected_sunday_share_error=("corrected_sunday_share_error", "mean"),
        baseline_abs_sunday_share_error=("baseline_sunday_share_error", lambda s: s.abs().mean()),
        corrected_abs_sunday_share_error=("corrected_sunday_share_error", lambda s: s.abs().mean()),
    ).reset_index()


def r_sunday_dollar_error_by_corridor():
    cand = get_validation_subset("r_sun_sat", "hybrid_post2022_shrink_to_all_prior")
    cand = cand.loc[cand["release_year"].ge(2022)].copy()
    cand["baseline_sunday_dollar_error"] = cand["sunday_gross_usd"] - cand["pred_sunday_gross_base"]
    cand["corrected_sunday_gross"] = cand["pred_sunday_gross_base"] * np.exp(cand["predicted_residual"])
    cand["corrected_sunday_dollar_error"] = cand["sunday_gross_usd"] - cand["corrected_sunday_gross"]
    return cand.groupby("release_corridor", observed=True).agg(
        n=("release_run_id", "count"),
        baseline_sunday_dollar_error=("baseline_sunday_dollar_error", "mean"),
        corrected_sunday_dollar_error=("corrected_sunday_dollar_error", "mean"),
        baseline_abs_sunday_dollar_error=("baseline_sunday_dollar_error", lambda s: s.abs().mean()),
        corrected_abs_sunday_dollar_error=("corrected_sunday_dollar_error", lambda s: s.abs().mean()),
    ).reset_index()


eb_corridor_sunday_share_error = eb_sunday_share_error_by_corridor()
r_corridor_sunday_dollar_error = r_sunday_dollar_error_by_corridor()
display(eb_corridor_sunday_share_error)
display(r_corridor_sunday_dollar_error)

for table, base_col, corr_col, title, ylabel in [
    (eb_corridor_sunday_share_error, "baseline_sunday_share_error", "corrected_sunday_share_error", "2022+ e_b Sunday share error by corridor", "actual - predicted Sunday share"),
    (r_corridor_sunday_dollar_error, "baseline_sunday_dollar_error", "corrected_sunday_dollar_error", "2022+ r_sun_sat Sunday dollar error by corridor", "actual - predicted Sunday dollars"),
]:
    plot = table.copy()
    plot["release_corridor"] = pd.Categorical(plot["release_corridor"], CORRIDOR_ORDER, ordered=True)
    plot = plot.sort_values("release_corridor")
    x = np.arange(len(plot))
    width = 0.38
    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.bar(x - width/2, plot[base_col], width=width, label="baseline", alpha=0.85)
    ax.bar(x + width/2, plot[corr_col], width=width, label="corrected", alpha=0.85)
    for xi, (_, row) in zip(x, plot.iterrows()):
        ax.text(xi, 0, f"n={int(row['n'])}", ha="center", va="bottom", fontsize=8)
    ax.axhline(0, color="#222222", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(plot["release_corridor"].astype(str), rotation=25, ha="right")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()
    plt.tight_layout()
    plt.show()

# 3. Actual vs predicted Sunday scatter for r_sun_sat.
r_post = get_validation_subset("r_sun_sat", "hybrid_post2022_shrink_to_all_prior")
r_post = r_post.loc[r_post["release_year"].ge(2022)].copy()
r_post["corrected_pred_sunday_gross"] = r_post["pred_sunday_gross_base"] * np.exp(r_post["predicted_residual"])

for pred_col, title in [("pred_sunday_gross_base", "Baseline predicted Sunday"), ("corrected_pred_sunday_gross", "Corrected predicted Sunday")]:
    fig, ax = plt.subplots(figsize=(6, 6))
    for corridor, g in r_post.groupby("release_corridor", observed=True):
        ax.scatter(g[pred_col], g["sunday_gross_usd"], alpha=0.75, label=corridor)
    lim = max(r_post[[pred_col, "sunday_gross_usd"]].max().max(), 1)
    ax.plot([0, lim], [0, lim], color="#222222", linewidth=1)
    ax.set_title(f"Post-2022 actual vs predicted Sunday: {title}")
    ax.set_xlabel(title)
    ax.set_ylabel("actual Sunday")
    ax.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()

# 4. Error distribution before/after.
def residual_distribution_stats():
    rows = []
    for spec in FINAL_VALIDATION_SPECS:
        target = spec["target"]
        strategy = spec["training_strategy"]
        cand = get_validation_subset(target, strategy).loc[lambda x: x["release_year"].ge(2022)].copy()
        for state, values in {
            "baseline": cand[target],
            "corrected": cand["corrected_residual"],
        }.items():
            vals = values.replace([np.inf, -np.inf], np.nan).dropna()
            rows.append({
                "target": target,
                "training_strategy": strategy,
                "state": state,
                "n": len(vals),
                "mean": vals.mean(),
                "median": vals.median(),
                "mae": vals.abs().mean(),
                "rmse": np.sqrt(np.nanmean(np.square(vals))),
                "q10": vals.quantile(0.10),
                "q90": vals.quantile(0.90),
            })
    return pd.DataFrame(rows)


production_error_distribution_stats = residual_distribution_stats()
display(production_error_distribution_stats)

for spec in FINAL_VALIDATION_SPECS:
    target = spec["target"]
    strategy = spec["training_strategy"]
    cand = get_validation_subset(target, strategy).loc[lambda x: x["release_year"].ge(2022)].copy()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(cand[target].dropna(), bins=24, alpha=0.55, density=True, label="baseline residual")
    ax.hist(cand["corrected_residual"].dropna(), bins=24, alpha=0.55, density=True, label="corrected residual")
    ax.axvline(0, color="#222222", linewidth=1)
    ax.set_title(f"Post-2022 residual distribution before/after: {spec['label']}")
    ax.set_xlabel("log residual")
    ax.set_ylabel("density")
    ax.legend()
    plt.tight_layout()
    plt.show()

# 5. Empirical residual interval coverage table.
def build_interval_validation_table(nominals=(0.50, 0.80, 0.95)):
    rows = []
    split_defs = {
        "all": lambda d: pd.Series(True, index=d.index),
        "2022+": lambda d: d["release_year"].ge(2022),
        "summer": lambda d: d["release_corridor"].eq("summer"),
        "Christmas/New Year": lambda d: d["release_corridor"].eq("Christmas/New Year"),
    }
    for spec in FINAL_VALIDATION_SPECS:
        target = spec["target"]
        strategy = spec["training_strategy"]
        preds = get_validation_subset(target, strategy).sort_values("opening_weekend_start").copy()
        for nominal in nominals:
            alpha = (1 - nominal) / 2
            interval_rows = []
            for idx, row in preds.iterrows():
                prior = preds.loc[preds["release_year"] < row["release_year"], "corrected_residual"].dropna()
                if len(prior) < 30:
                    prior = preds.loc[preds.index < idx, "corrected_residual"].dropna()
                if len(prior) < 30:
                    continue
                q_low = prior.quantile(alpha)
                q_high = prior.quantile(1 - alpha)
                interval_rows.append({
                    "release_run_id": row["release_run_id"],
                    "release_year": row["release_year"],
                    "release_corridor": row["release_corridor"],
                    "target": target,
                    "training_strategy": strategy,
                    "nominal": nominal,
                    "actual_residual": row[target],
                    "predicted_residual": row["predicted_residual"],
                    "interval_low": row["predicted_residual"] + q_low,
                    "interval_high": row["predicted_residual"] + q_high,
                    "interval_width": q_high - q_low,
                })
            interval_df = pd.DataFrame(interval_rows)
            if interval_df.empty:
                continue
            interval_df["covered"] = interval_df["actual_residual"].between(interval_df["interval_low"], interval_df["interval_high"])
            for split_name, split_fn in split_defs.items():
                g = interval_df.loc[split_fn(interval_df)]
                if g.empty:
                    continue
                rows.append({
                    "target": target,
                    "training_strategy": strategy,
                    "split": split_name,
                    "nominal": nominal,
                    "n": len(g),
                    "empirical_coverage": g["covered"].mean(),
                    "avg_interval_width": g["interval_width"].mean(),
                })
    return pd.DataFrame(rows).sort_values(["target", "training_strategy", "split", "nominal"]).reset_index(drop=True)


production_interval_coverage = build_interval_validation_table()
display(production_interval_coverage)

# Compact decision flags for final reporting.
production_validation_decision_summary = pd.DataFrame([
    {
        "target": "e_b",
        "model": "all_prior_correction",
        "post_2022_year_min_log_improvement": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("e_b") & yearly_production_improvement["release_year"].ge(2022), "log_mae_improvement_pct"].min(),
        "post_2022_year_min_dollar_improvement": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("e_b") & yearly_production_improvement["release_year"].ge(2022), "dollar_mae_improvement_pct"].min(),
        "post_2022_year_positive_log_share": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("e_b") & yearly_production_improvement["release_year"].ge(2022), "log_mae_improvement_pct"].gt(0).mean(),
        "post_2022_year_positive_dollar_share": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("e_b") & yearly_production_improvement["release_year"].ge(2022), "dollar_mae_improvement_pct"].ge(0).mean(),
    },
    {
        "target": "r_sun_sat",
        "model": "hybrid_post2022_shrink_to_all_prior",
        "post_2022_year_min_log_improvement": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("r_sun_sat") & yearly_production_improvement["release_year"].ge(2022), "log_mae_improvement_pct"].min(),
        "post_2022_year_min_dollar_improvement": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("r_sun_sat") & yearly_production_improvement["release_year"].ge(2022), "dollar_mae_improvement_pct"].min(),
        "post_2022_year_positive_log_share": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("r_sun_sat") & yearly_production_improvement["release_year"].ge(2022), "log_mae_improvement_pct"].gt(0).mean(),
        "post_2022_year_positive_dollar_share": yearly_production_improvement.loc[yearly_production_improvement["target"].eq("r_sun_sat") & yearly_production_improvement["release_year"].ge(2022), "dollar_mae_improvement_pct"].ge(0).mean(),
    },
])
display(production_validation_decision_summary)


## Frozen Shape Stack and r_sun_sat Interval Recalibration

Freeze the point stack before adding any new predictors:

- `e_a`: no point correction.
- `e_b`: all-prior corridor correction, candidate only.
- `r_sun_sat`: hybrid post-2022 shrink-to-all-prior correction, final candidate.

The only production-branch point change considered here is after-Saturday Sunday:

\[
Sun_{afterSat}=Sat_{actual}	imes\exp(\widehat{\log(Sun/Sat)}_{base}+r^{hybrid}_{sun/sat}).
\]

This section recalibrates empirical residual intervals for the corrected `r_sun_sat` model and tests a corridor guardrail so the point lift is applied only where it has reliable operational payoff.


In [ ]:
FROZEN_SHAPE_STACK_DECISION = pd.DataFrame([
    {"target": "e_a", "point_model": "no_correction", "status": "frozen_no_point_correction"},
    {"target": "e_b", "point_model": "all_prior_corridor_correction", "status": "candidate_only"},
    {"target": "r_sun_sat", "point_model": "hybrid_post2022_shrink_to_all_prior", "status": "final_candidate"},
])
display(FROZEN_SHAPE_STACK_DECISION)

R_FINAL_STRATEGY = "hybrid_post2022_shrink_to_all_prior"
r_final_validation = get_validation_subset("r_sun_sat", R_FINAL_STRATEGY).copy()
r_final_validation["baseline_pred_sunday_gross"] = r_final_validation["pred_sunday_gross_base"]
r_final_validation["corrected_pred_sunday_gross"] = r_final_validation["pred_sunday_gross_base"] * np.exp(r_final_validation["predicted_residual"])
r_final_validation["baseline_sunday_error"] = r_final_validation["sunday_gross_usd"] - r_final_validation["baseline_pred_sunday_gross"]
r_final_validation["corrected_sunday_error"] = r_final_validation["sunday_gross_usd"] - r_final_validation["corrected_pred_sunday_gross"]
# Positive means corrected prediction is above actual; these are the residuals used for interval calibration around corrected prediction.
r_final_validation["r_sun_sat_corr"] = np.log(r_final_validation["corrected_pred_sunday_gross"] / r_final_validation["sunday_gross_usd"])
r_final_validation["abs_baseline_sunday_error"] = r_final_validation["baseline_sunday_error"].abs()
r_final_validation["abs_corrected_sunday_error"] = r_final_validation["corrected_sunday_error"].abs()

INTERVAL_GROUPS = {
    "all": lambda d: pd.Series(True, index=d.index),
    "2022+": lambda d: d["release_year"].ge(2022),
    "summer": lambda d: d["release_corridor"].eq("summer"),
    "Christmas/New Year": lambda d: d["release_corridor"].eq("Christmas/New Year"),
    "Jan-Mar": lambda d: d["release_corridor"].eq("Jan-Mar"),
    "weak fall": lambda d: d["release_corridor"].eq("weak fall"),
}


def empirical_quantile_rows(df, residual_col="r_sun_sat_corr", alphas=(0.025, 0.10, 0.25, 0.75, 0.90, 0.975), lam=35):
    rows = []
    global_resid = df[residual_col].replace([np.inf, -np.inf], np.nan).dropna()
    global_q = global_resid.quantile(list(alphas))
    for group_name, mask_fn in INTERVAL_GROUPS.items():
        g = df.loc[mask_fn(df), residual_col].replace([np.inf, -np.inf], np.nan).dropna()
        n = len(g)
        weight = n / (n + lam) if n else 0.0
        raw_q = g.quantile(list(alphas)) if n else global_q.copy()
        for alpha in alphas:
            rows.append({
                "target": "r_sun_sat",
                "model": R_FINAL_STRATEGY,
                "interval_group": group_name,
                "alpha": alpha,
                "n": n,
                "lambda": lam,
                "shrink_weight": weight,
                "raw_quantile": raw_q.loc[alpha] if alpha in raw_q.index else np.nan,
                "global_quantile": global_q.loc[alpha],
                "shrunk_quantile": weight * (raw_q.loc[alpha] if alpha in raw_q.index else global_q.loc[alpha]) + (1 - weight) * global_q.loc[alpha],
            })
    return pd.DataFrame(rows)

r_interval_quantiles = empirical_quantile_rows(r_final_validation, lam=35)
display(r_interval_quantiles)


def interval_group_for_row(row):
    corridor = row["release_corridor"]
    if corridor in ["summer", "Christmas/New Year", "Jan-Mar", "weak fall"]:
        return corridor
    if row["release_year"] >= 2022:
        return "2022+"
    return "all"


def interval_coverage_from_quantiles(df, quantiles, nominals=(0.50, 0.80, 0.95), split_defs=None, use_shrunk=True):
    if split_defs is None:
        split_defs = INTERVAL_GROUPS
    q_col = "shrunk_quantile" if use_shrunk else "raw_quantile"
    q_lookup = quantiles.set_index(["interval_group", "alpha"])[q_col]
    rows = []
    scored_frames = []
    scored = df.copy()
    scored["interval_group"] = scored.apply(interval_group_for_row, axis=1)
    for nominal in nominals:
        alpha_low = round((1 - nominal) / 2, 3)
        alpha_high = round(1 - alpha_low, 3)
        scored_nom = scored.copy()
        scored_nom["nominal"] = nominal
        fallback_low = float(q_lookup.loc[("all", alpha_low)])
        fallback_high = float(q_lookup.loc[("all", alpha_high)])
        scored_nom["q_low"] = [float(q_lookup.get((g, alpha_low), fallback_low) if q_lookup.get((g, alpha_low), fallback_low) is not None else fallback_low) for g in scored_nom["interval_group"]]
        scored_nom["q_high"] = [float(q_lookup.get((g, alpha_high), fallback_high) if q_lookup.get((g, alpha_high), fallback_high) is not None else fallback_high) for g in scored_nom["interval_group"]]
        scored_nom["interval_low_sunday"] = scored_nom["corrected_pred_sunday_gross"] / np.exp(scored_nom["q_high"].astype(float))
        scored_nom["interval_high_sunday"] = scored_nom["corrected_pred_sunday_gross"] / np.exp(scored_nom["q_low"].astype(float))
        scored_nom["interval_width_sunday"] = scored_nom["interval_high_sunday"] - scored_nom["interval_low_sunday"]
        scored_nom["covered"] = scored_nom["sunday_gross_usd"].between(scored_nom["interval_low_sunday"], scored_nom["interval_high_sunday"])
        scored_frames.append(scored_nom)
        for split_name, split_fn in split_defs.items():
            g = scored_nom.loc[split_fn(scored_nom)]
            if g.empty:
                continue
            rows.append({
                "target": "r_sun_sat",
                "model": R_FINAL_STRATEGY,
                "split": split_name,
                "nominal": nominal,
                "n": len(g),
                "empirical_coverage": g["covered"].mean(),
                "avg_interval_width_usd": g["interval_width_sunday"].mean(),
                "median_interval_width_usd": g["interval_width_sunday"].median(),
                "quantile_type": "shrunk" if use_shrunk else "raw",
            })
    return pd.DataFrame(rows), pd.concat(scored_frames, ignore_index=True)

r_interval_coverage_recalibrated, r_interval_scored = interval_coverage_from_quantiles(r_final_validation, r_interval_quantiles, use_shrunk=True)
display(r_interval_coverage_recalibrated)

# Corridor guardrail for the r_sun_sat point lift.
POINT_LIFT_CORRIDORS = {"summer", "Christmas/New Year", "Apr-May"}
GUARDRAIL_GAMMAS = [0.0, 0.25, 0.5, 1.0]

def evaluate_guardrail_gamma(df, gamma):
    out = df.copy()
    scale = np.where(out["release_corridor"].isin(POINT_LIFT_CORRIDORS), 1.0, gamma)
    out["guardrail_gamma"] = gamma
    out["guardrail_scale"] = scale
    out["guardrail_predicted_residual"] = out["predicted_residual"] * out["guardrail_scale"]
    out["guardrail_pred_sunday_gross"] = out["pred_sunday_gross_base"] * np.exp(out["guardrail_predicted_residual"])
    out["guardrail_sunday_error"] = out["sunday_gross_usd"] - out["guardrail_pred_sunday_gross"]
    out["guardrail_abs_sunday_error"] = out["guardrail_sunday_error"].abs()
    out["guardrail_corrected_residual"] = out["r_sun_sat"] - out["guardrail_predicted_residual"]
    return out

guardrail_predictions = pd.concat([evaluate_guardrail_gamma(r_final_validation, gamma) for gamma in GUARDRAIL_GAMMAS], ignore_index=True)

def summarize_guardrail(preds):
    rows = []
    for gamma, g in preds.groupby("guardrail_gamma", observed=True):
        recent = g.loc[g["release_year"].ge(2022)]
        year_rows = []
        for year, yg in g.groupby("release_year", observed=True):
            base_dollar = yg["abs_baseline_sunday_error"].mean()
            corr_dollar = yg["guardrail_abs_sunday_error"].mean()
            base_log = yg["r_sun_sat"].abs().mean()
            corr_log = yg["guardrail_corrected_residual"].abs().mean()
            year_rows.append({
                "guardrail_gamma": gamma,
                "release_year": int(year),
                "n": len(yg),
                "log_mae_improvement_pct": (base_log - corr_log) / base_log if base_log else np.nan,
                "dollar_mae_improvement_pct": (base_dollar - corr_dollar) / base_dollar if base_dollar else np.nan,
            })
        year_df = pd.DataFrame(year_rows)
        recent_years = year_df.loc[year_df["release_year"].ge(2022)]
        rows.append({
            "guardrail_gamma": gamma,
            "n": len(g),
            "post_2022_n": len(recent),
            "log_mae": g["guardrail_corrected_residual"].abs().mean(),
            "dollar_mae": g["guardrail_abs_sunday_error"].mean(),
            "post_2022_log_mae": recent["guardrail_corrected_residual"].abs().mean(),
            "post_2022_dollar_mae": recent["guardrail_abs_sunday_error"].mean(),
            "log_mae_improvement_pct": (g["r_sun_sat"].abs().mean() - g["guardrail_corrected_residual"].abs().mean()) / g["r_sun_sat"].abs().mean(),
            "dollar_mae_improvement_pct": (g["abs_baseline_sunday_error"].mean() - g["guardrail_abs_sunday_error"].mean()) / g["abs_baseline_sunday_error"].mean(),
            "post_2022_log_mae_improvement_pct": (recent["r_sun_sat"].abs().mean() - recent["guardrail_corrected_residual"].abs().mean()) / recent["r_sun_sat"].abs().mean(),
            "post_2022_dollar_mae_improvement_pct": (recent["abs_baseline_sunday_error"].mean() - recent["guardrail_abs_sunday_error"].mean()) / recent["abs_baseline_sunday_error"].mean(),
            "post_2022_year_min_log_improvement": recent_years["log_mae_improvement_pct"].min(),
            "post_2022_year_min_dollar_improvement": recent_years["dollar_mae_improvement_pct"].min(),
            "post_2022_year_positive_log_share": recent_years["log_mae_improvement_pct"].gt(0).mean(),
            "post_2022_year_positive_dollar_share": recent_years["dollar_mae_improvement_pct"].ge(0).mean(),
        })
    return pd.DataFrame(rows).sort_values(["post_2022_dollar_mae", "post_2022_log_mae"]).reset_index(drop=True)

guardrail_summary = summarize_guardrail(guardrail_predictions)
display(guardrail_summary)

best_guardrail_gamma = guardrail_summary.iloc[0]["guardrail_gamma"]
best_guardrail_predictions = guardrail_predictions.loc[guardrail_predictions["guardrail_gamma"].eq(best_guardrail_gamma)].copy()

guardrail_corridor_summary = best_guardrail_predictions.loc[best_guardrail_predictions["release_year"].ge(2022)].groupby("release_corridor", observed=True).agg(
    n=("release_run_id", "count"),
    baseline_abs_sunday_error=("abs_baseline_sunday_error", "mean"),
    hybrid_abs_sunday_error=("abs_corrected_sunday_error", "mean"),
    guardrail_abs_sunday_error=("guardrail_abs_sunday_error", "mean"),
    baseline_signed_sunday_error=("baseline_sunday_error", "mean"),
    hybrid_signed_sunday_error=("corrected_sunday_error", "mean"),
    guardrail_signed_sunday_error=("guardrail_sunday_error", "mean"),
).reset_index()
guardrail_corridor_summary["hybrid_abs_improvement_pct"] = (guardrail_corridor_summary["baseline_abs_sunday_error"] - guardrail_corridor_summary["hybrid_abs_sunday_error"]) / guardrail_corridor_summary["baseline_abs_sunday_error"]
guardrail_corridor_summary["guardrail_abs_improvement_pct"] = (guardrail_corridor_summary["baseline_abs_sunday_error"] - guardrail_corridor_summary["guardrail_abs_sunday_error"]) / guardrail_corridor_summary["baseline_abs_sunday_error"]
display(guardrail_corridor_summary)

fig, ax = plt.subplots(figsize=(10, 4.5))
plot = guardrail_summary.sort_values("guardrail_gamma")
ax.plot(plot["guardrail_gamma"], plot["post_2022_log_mae_improvement_pct"], marker="o", label="2022+ log MAE improvement")
ax.plot(plot["guardrail_gamma"], plot["post_2022_dollar_mae_improvement_pct"], marker="o", label="2022+ dollar MAE improvement")
ax.axhline(0, color="#222222", linewidth=1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("r_sun_sat corridor gate validation")
ax.set_xlabel("gamma applied outside summer / Christmas-New Year / Apr-May")
ax.set_ylabel("improvement vs baseline")
ax.legend()
plt.tight_layout()
plt.show()

plot = guardrail_corridor_summary.copy()
plot["release_corridor"] = pd.Categorical(plot["release_corridor"], CORRIDOR_ORDER, ordered=True)
plot = plot.sort_values("release_corridor")
x = np.arange(len(plot))
width = 0.28
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.bar(x - width, plot["baseline_abs_sunday_error"], width=width, label="baseline", alpha=0.85)
ax.bar(x, plot["hybrid_abs_sunday_error"], width=width, label="hybrid", alpha=0.85)
ax.bar(x + width, plot["guardrail_abs_sunday_error"], width=width, label=f"guardrail gamma={best_guardrail_gamma:g}", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(plot["release_corridor"].astype(str), rotation=25, ha="right")
ax.set_title("2022+ r_sun_sat abs Sunday dollar error by corridor")
ax.set_ylabel("mean absolute Sunday dollar error")
ax.legend()
plt.tight_layout()
plt.show()

frozen_point_stack_summary = pd.DataFrame([
    {"component": "shape_prior_e_a", "production_action": "no correction", "promotion_status": "frozen"},
    {"component": "shape_prior_e_b", "production_action": "all-prior corridor correction", "promotion_status": "candidate only"},
    {"component": "after_sat_r_sun_sat", "production_action": f"hybrid correction with corridor gate gamma={best_guardrail_gamma:g}", "promotion_status": "final candidate pending interval calibration"},
])
display(frozen_point_stack_summary)


## Final Seasonality Audit Before Predictor Variables

This section audits the final frozen after-Saturday guardrail residual before adding predictor variables. It uses only:

\[
u_i = \log(Sun_{afterSat, guardrail, i} / Sun_{actual, i})
\]

Positive residuals mean the guardrail forecast is above actual Sunday. The goal is to decide whether any remaining seasonality is a point-forecast problem or mainly an interval-width problem.


In [ ]:
# Final seasonality audit on the frozen after-Saturday guardrail residual.
seasonality_source = best_guardrail_predictions.copy()
seasonality_source["opening_weekend_start"] = pd.to_datetime(seasonality_source["opening_weekend_start"], errors="coerce")

expected_cols = ["release_run_id", "latest_estimate_mid_usd", "opening_weekend_gross_usd"]
expected_source = residual_table[[c for c in expected_cols if c in residual_table.columns]].drop_duplicates("release_run_id")
seasonality_source = seasonality_source.merge(expected_source, on="release_run_id", how="left", suffixes=("", "_residual_table"))

if "latest_estimate_mid_usd" in seasonality_source.columns:
    seasonality_source["expected_ow_for_weight"] = pd.to_numeric(seasonality_source["latest_estimate_mid_usd"], errors="coerce")
else:
    seasonality_source["expected_ow_for_weight"] = np.nan
if "opening_weekend_gross_usd_residual_table" in seasonality_source.columns:
    ow_fallback = seasonality_source["opening_weekend_gross_usd_residual_table"]
elif "opening_weekend_gross_usd" in seasonality_source.columns:
    ow_fallback = seasonality_source["opening_weekend_gross_usd"]
else:
    ow_fallback = np.nan
seasonality_source["expected_ow_for_weight"] = seasonality_source["expected_ow_for_weight"].where(
    seasonality_source["expected_ow_for_weight"].gt(0), ow_fallback
)

seasonality_source["u_guardrail"] = np.log(seasonality_source["guardrail_pred_sunday_gross"] / seasonality_source["sunday_gross_usd"])
seasonality_source["release_week_start"] = seasonality_source["opening_weekend_start"].dt.to_period("W-SUN").dt.start_time
seasonality_source["week_of_year"] = seasonality_source["opening_weekend_start"].dt.isocalendar().week.astype(int)
seasonality_source["seasonality_era"] = np.select(
    [seasonality_source["release_year"].between(2015, 2019), seasonality_source["release_year"].ge(2022)],
    ["2015-2019", "2022+"],
    default="other",
)

def weighted_mean_or_nan(values, weights):
    values = pd.Series(values, dtype="float64")
    weights = pd.Series(weights, dtype="float64")
    valid = values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return np.average(values.loc[valid], weights=weights.loc[valid])

weekly_rows = []
for week, g in seasonality_source.dropna(subset=["release_week_start"]).groupby("release_week_start", observed=True):
    weekly_rows.append({
        "week": week,
        "year": int(pd.Timestamp(week).year),
        "week_of_year": int(pd.Timestamp(week).isocalendar().week),
        "residual_mean": g["u_guardrail"].mean(),
        "residual_weighted_mean": weighted_mean_or_nan(g["u_guardrail"], g["expected_ow_for_weight"]),
        "n": int(g["u_guardrail"].notna().sum()),
        "expected_ow_sum": g["expected_ow_for_weight"].sum(min_count=1),
    })
weekly_residual_audit_table = pd.DataFrame(weekly_rows).sort_values("week").reset_index(drop=True)
full_weeks = pd.date_range(weekly_residual_audit_table["week"].min(), weekly_residual_audit_table["week"].max(), freq="W-MON")
weekly_residual_audit_table = (
    weekly_residual_audit_table.set_index("week")
    .reindex(full_weeks)
    .rename_axis("week")
    .reset_index()
)
weekly_residual_audit_table["year"] = weekly_residual_audit_table["week"].dt.year.astype(int)
weekly_residual_audit_table["week_of_year"] = weekly_residual_audit_table["week"].dt.isocalendar().week.astype(int)
weekly_residual_audit_table["n"] = weekly_residual_audit_table["n"].fillna(0).astype(int)
weekly_residual_audit_table["expected_ow_sum"] = weekly_residual_audit_table["expected_ow_sum"].fillna(0.0)

weekly_count_summary = pd.DataFrame([{
    "weeks": len(weekly_residual_audit_table),
    "nonempty_weeks": int(weekly_residual_audit_table["n"].gt(0).sum()),
    "zero_movie_weeks": int(weekly_residual_audit_table["n"].eq(0).sum()),
    "one_movie_weeks": int(weekly_residual_audit_table["n"].eq(1).sum()),
    "zero_or_one_movie_week_share": weekly_residual_audit_table["n"].le(1).mean(),
    "weighted_unweighted_corr": weekly_residual_audit_table[["residual_mean", "residual_weighted_mean"]].corr().iloc[0, 1],
    "mean_abs_weighted_unweighted_gap": (weekly_residual_audit_table["residual_mean"] - weekly_residual_audit_table["residual_weighted_mean"]).abs().mean(),
}])
display(weekly_count_summary)
display(weekly_residual_audit_table.head())

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
axes[0].plot(weekly_residual_audit_table["week"], weekly_residual_audit_table["residual_mean"], color="#4c78a8", linewidth=1.4)
axes[0].axhline(0, color="#111111", linewidth=1)
axes[0].set_title("Weekly unweighted mean final guardrail residual")
axes[0].set_ylabel("log residual")
axes[1].plot(weekly_residual_audit_table["week"], weekly_residual_audit_table["residual_weighted_mean"], color="#f58518", linewidth=1.4)
axes[1].axhline(0, color="#111111", linewidth=1)
axes[1].set_title("Weekly expected-OW-weighted final guardrail residual")
axes[1].set_ylabel("weighted log residual")
axes[2].bar(weekly_residual_audit_table["week"], weekly_residual_audit_table["n"], width=5, color="#54a24b", alpha=0.85)
axes[2].set_title("Weekly release count")
axes[2].set_ylabel("n")
axes[2].set_xlabel("release week")
plt.show()

seasonal_weekly_era = seasonality_source.loc[seasonality_source["seasonality_era"].isin(["2015-2019", "2022+"])].groupby(
    ["seasonality_era", "week_of_year"], observed=True
).agg(
    residual_mean=("u_guardrail", "mean"),
    residual_median=("u_guardrail", "median"),
    residual_std=("u_guardrail", "std"),
    n=("release_run_id", "count"),
).reset_index()

def centered_rolling_by_era(df, value_col, window=5):
    out = []
    for era, g in df.sort_values("week_of_year").groupby("seasonality_era", observed=True):
        gg = g.copy()
        gg[f"{value_col}_smooth"] = gg[value_col].rolling(window=window, min_periods=2, center=True).mean()
        out.append(gg)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

seasonal_weekly_era = centered_rolling_by_era(seasonal_weekly_era, "residual_mean", window=5)
display(seasonal_weekly_era.head())

fig, ax = plt.subplots(figsize=(13, 5))
for era, g in seasonal_weekly_era.groupby("seasonality_era", observed=True):
    ax.plot(g["week_of_year"], g["residual_mean_smooth"], marker="o", linewidth=1.8, markersize=3, label=era)
ax.axhline(0, color="#111111", linewidth=1)
ax.set_title("Seasonal plot by week of year: final guardrail residual")
ax.set_xlabel("week of year")
ax.set_ylabel("5-week smoothed mean log residual")
ax.legend()
plt.tight_layout()
plt.show()

corridor_order = ["Jan-Mar", "Apr-May", "summer", "weak fall", "Thanksgiving", "Christmas/New Year", "ordinary non-holiday"]
final_corridor_residual_summary = seasonality_source.groupby("release_corridor", observed=True).agg(
    n=("release_run_id", "count"),
    residual_mean=("u_guardrail", "mean"),
    residual_median=("u_guardrail", "median"),
    residual_std=("u_guardrail", "std"),
    residual_mae=("u_guardrail", lambda s: s.abs().mean()),
    q10=("u_guardrail", lambda s: s.quantile(0.10)),
    q90=("u_guardrail", lambda s: s.quantile(0.90)),
).reset_index()
final_corridor_residual_summary["release_corridor"] = pd.Categorical(final_corridor_residual_summary["release_corridor"], corridor_order, ordered=True)
final_corridor_residual_summary = final_corridor_residual_summary.sort_values("release_corridor").reset_index(drop=True)
display(final_corridor_residual_summary)

fig, ax = plt.subplots(figsize=(11, 4.8))
plot_corr = final_corridor_residual_summary.dropna(subset=["release_corridor"])
ax.bar(plot_corr["release_corridor"].astype(str), plot_corr["residual_mean"], color="#4c78a8", alpha=0.85, label="mean")
ax.errorbar(plot_corr["release_corridor"].astype(str), plot_corr["residual_mean"], yerr=plot_corr["residual_std"], fmt="none", ecolor="#333333", capsize=3, label="std")
ax.axhline(0, color="#111111", linewidth=1)
ax.set_title("Final guardrail residual by corridor")
ax.set_ylabel("log residual")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

seasonal_subseries_corridor = seasonality_source.groupby(["release_corridor", "release_year"], observed=True).agg(
    n=("release_run_id", "count"),
    residual_mean=("u_guardrail", "mean"),
    residual_median=("u_guardrail", "median"),
    residual_std=("u_guardrail", "std"),
).reset_index()
seasonal_subseries_corridor["release_corridor"] = pd.Categorical(seasonal_subseries_corridor["release_corridor"], corridor_order, ordered=True)
seasonal_subseries_corridor = seasonal_subseries_corridor.sort_values(["release_corridor", "release_year"]).reset_index(drop=True)
display(seasonal_subseries_corridor.head(20))

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharey=True, constrained_layout=True)
axes = axes.ravel()
for ax, corridor in zip(axes, corridor_order[:6]):
    g = seasonal_subseries_corridor.loc[seasonal_subseries_corridor["release_corridor"].astype(str).eq(corridor)]
    ax.plot(g["release_year"], g["residual_mean"], marker="o", linewidth=1.5)
    ax.axhline(0, color="#111111", linewidth=1)
    ax.set_title(corridor)
    ax.set_xlabel("year")
    ax.set_ylabel("mean log residual")
plt.show()

# Robust STL. Prefer statsmodels if available; otherwise use a robust seasonal-decomposition fallback.
stl_input = weekly_residual_audit_table[["week", "residual_mean", "n"]].copy()
stl_series = stl_input.set_index("week")["residual_mean"].astype(float).interpolate(method="time", limit_direction="both")

def robust_weekly_decomposition(series, n_series, model_name, seasonal_window_weeks=53, trend_window_weeks=53):
    observed = series.astype(float).copy()
    trend = observed.rolling(trend_window_weeks, center=True, min_periods=max(8, trend_window_weeks // 4)).median()
    trend = trend.interpolate(method="time", limit_direction="both")
    detrended = observed - trend
    week_no = observed.index.isocalendar().week.astype(int)
    seasonal_lookup = pd.DataFrame({"week_of_year": week_no, "detrended": detrended.values}).groupby("week_of_year", observed=True)["detrended"].median()
    seasonal = pd.Series([seasonal_lookup.get(w, 0.0) for w in week_no], index=observed.index, dtype="float64")
    if seasonal_window_weeks and seasonal_window_weeks < 53:
        seasonal = seasonal.rolling(seasonal_window_weeks, center=True, min_periods=max(3, seasonal_window_weeks // 4)).median()
        seasonal = seasonal.interpolate(method="time", limit_direction="both")
    seasonal = seasonal - seasonal.mean()
    remainder = observed - trend - seasonal
    return pd.DataFrame({
        "stl_model": model_name,
        "week": observed.index,
        "observed": observed.values,
        "trend": trend.values,
        "seasonal": seasonal.values,
        "remainder": remainder.values,
        "n": n_series.reindex(observed.index).fillna(0).astype(int).values,
    })

stl_tables = []
try:
    from statsmodels.tsa.seasonal import STL
    stl_specs = [
        ("fixed_annual_statsmodels", 53, 105),
        ("flexible_annual_statsmodels", 13, 53),
    ]
    for name, seasonal_len, trend_len in stl_specs:
        result = STL(stl_series, period=52, seasonal=seasonal_len, trend=trend_len, robust=True).fit()
        stl_tables.append(pd.DataFrame({
            "stl_model": name,
            "week": stl_series.index,
            "observed": stl_series.values,
            "trend": result.trend.values,
            "seasonal": result.seasonal.values,
            "remainder": result.resid.values,
            "n": stl_input.set_index("week")["n"].reindex(stl_series.index).fillna(0).astype(int).values,
        }))
    stl_method = "statsmodels_STL"
except Exception as exc:
    stl_tables.append(robust_weekly_decomposition(stl_series, stl_input.set_index("week")["n"], "fixed_annual_robust_fallback", seasonal_window_weeks=53, trend_window_weeks=53))
    stl_tables.append(robust_weekly_decomposition(stl_series, stl_input.set_index("week")["n"], "flexible_annual_robust_fallback", seasonal_window_weeks=13, trend_window_weeks=53))
    stl_method = f"robust_fallback_no_statsmodels: {type(exc).__name__}"

stl_decomposition_table = pd.concat(stl_tables, ignore_index=True)
stl_component_summary = stl_decomposition_table.groupby("stl_model", observed=True).agg(
    observed_std=("observed", "std"),
    trend_std=("trend", "std"),
    seasonal_std=("seasonal", "std"),
    remainder_std=("remainder", "std"),
    seasonal_abs_mean=("seasonal", lambda s: s.abs().mean()),
    remainder_abs_mean=("remainder", lambda s: s.abs().mean()),
    max_abs_seasonal=("seasonal", lambda s: s.abs().max()),
).reset_index()
stl_component_summary["method"] = stl_method
display(stl_component_summary)

for model_name, g in stl_decomposition_table.groupby("stl_model", observed=True):
    fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
    for ax, col in zip(axes, ["observed", "trend", "seasonal", "remainder"]):
        ax.plot(g["week"], g[col], linewidth=1.3)
        ax.axhline(0, color="#111111", linewidth=1)
        ax.set_ylabel(col)
    axes[0].set_title(f"Robust STL decomposition: {model_name}")
    axes[-1].set_xlabel("release week")
    plt.show()

primary_stl_model = stl_decomposition_table["stl_model"].iloc[0]
seasonally_adjusted_residual_table = stl_decomposition_table.loc[stl_decomposition_table["stl_model"].eq(primary_stl_model)].copy()
seasonally_adjusted_residual_table["seasonally_adjusted"] = seasonally_adjusted_residual_table["observed"] - seasonally_adjusted_residual_table["seasonal"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(seasonally_adjusted_residual_table["week"], seasonally_adjusted_residual_table["observed"], label="observed weekly residual", linewidth=1.2)
ax.plot(seasonally_adjusted_residual_table["week"], seasonally_adjusted_residual_table["seasonally_adjusted"], label="seasonally adjusted", linewidth=1.2)
ax.plot(seasonally_adjusted_residual_table["week"], seasonally_adjusted_residual_table["remainder"], label="remainder", linewidth=1.2)
ax.axhline(0, color="#111111", linewidth=1)
ax.set_title(f"Seasonally adjusted residual plot: {primary_stl_model}")
ax.set_ylabel("log residual")
ax.set_xlabel("release week")
ax.legend()
plt.tight_layout()
plt.show()

final_seasonality_acf_rows = []
final_seasonality_ljung_rows = []
series_specs = {
    "observed_weekly_residual": seasonally_adjusted_residual_table.set_index("week")["observed"],
    "seasonally_adjusted": seasonally_adjusted_residual_table.set_index("week")["seasonally_adjusted"],
    "stl_remainder": seasonally_adjusted_residual_table.set_index("week")["remainder"],
}
for series_name, series in series_specs.items():
    acf_df = acf_pairwise(series, max_lag=104)
    acf_df["series"] = series_name
    acf_df["stl_model"] = primary_stl_model
    final_seasonality_acf_rows.append(acf_df)
    lb = ljung_box_from_acf(acf_df, acf_lags_to_test)
    lb["series"] = series_name
    lb["stl_model"] = primary_stl_model
    lb["week_n"] = series.notna().sum()
    final_seasonality_ljung_rows.append(lb)

final_seasonality_acf = pd.concat(final_seasonality_acf_rows, ignore_index=True)
final_seasonality_ljung = pd.concat(final_seasonality_ljung_rows, ignore_index=True)
display(final_seasonality_ljung.loc[final_seasonality_ljung["lag"].isin([13, 26, 52, 104])].sort_values(["series", "lag"]))

final_seasonality_decision_summary = pd.DataFrame([{
    "weekly_zero_or_one_movie_share": weekly_count_summary.loc[0, "zero_or_one_movie_week_share"],
    "weighted_unweighted_corr": weekly_count_summary.loc[0, "weighted_unweighted_corr"],
    "mean_abs_weighted_unweighted_gap": weekly_count_summary.loc[0, "mean_abs_weighted_unweighted_gap"],
    "primary_stl_model": primary_stl_model,
    "stl_method": stl_method,
    "seasonal_abs_mean": stl_component_summary.loc[stl_component_summary["stl_model"].eq(primary_stl_model), "seasonal_abs_mean"].iloc[0],
    "remainder_abs_mean": stl_component_summary.loc[stl_component_summary["stl_model"].eq(primary_stl_model), "remainder_abs_mean"].iloc[0],
    "seasonal_to_remainder_abs_ratio": stl_component_summary.loc[stl_component_summary["stl_model"].eq(primary_stl_model), "seasonal_abs_mean"].iloc[0] / stl_component_summary.loc[stl_component_summary["stl_model"].eq(primary_stl_model), "remainder_abs_mean"].iloc[0],
    "observed_lag52_acf": final_seasonality_acf.loc[final_seasonality_acf["series"].eq("observed_weekly_residual") & final_seasonality_acf["lag"].eq(52), "acf"].iloc[0],
    "adjusted_lag52_acf": final_seasonality_acf.loc[final_seasonality_acf["series"].eq("seasonally_adjusted") & final_seasonality_acf["lag"].eq(52), "acf"].iloc[0],
    "remainder_lag52_acf": final_seasonality_acf.loc[final_seasonality_acf["series"].eq("stl_remainder") & final_seasonality_acf["lag"].eq(52), "acf"].iloc[0],
    "christmas_mean": final_corridor_residual_summary.loc[final_corridor_residual_summary["release_corridor"].astype(str).eq("Christmas/New Year"), "residual_mean"].iloc[0],
    "christmas_std": final_corridor_residual_summary.loc[final_corridor_residual_summary["release_corridor"].astype(str).eq("Christmas/New Year"), "residual_std"].iloc[0],
    "summer_mean": final_corridor_residual_summary.loc[final_corridor_residual_summary["release_corridor"].astype(str).eq("summer"), "residual_mean"].iloc[0],
    "summer_std": final_corridor_residual_summary.loc[final_corridor_residual_summary["release_corridor"].astype(str).eq("summer"), "residual_std"].iloc[0],
}])
display(final_seasonality_decision_summary)


## Seasonality Sign-off and Interval Finalization

This final section turns the seasonality audit into production decisions before predictor variables. It signs off against adding another point-seasonality layer, switches the final `r_sun_sat` point guardrail to the stricter summer + Christmas/New Year rule, and tunes only the Christmas/New Year interval width.


In [ ]:
# 1. Seasonality sign-off table.
signoff_source = final_seasonality_decision_summary.iloc[0]
seasonality_signoff_table = pd.DataFrame([
    {
        "item": "zero-or-one movie week share",
        "result": f"{signoff_source['weekly_zero_or_one_movie_share']:.1%}",
        "numeric_result": signoff_source["weekly_zero_or_one_movie_share"],
        "decision": "exact weekly decomposition fragile",
    },
    {
        "item": "weighted/unweighted corr",
        "result": f"{signoff_source['weighted_unweighted_corr']:.3f}",
        "numeric_result": signoff_source["weighted_unweighted_corr"],
        "decision": "aggregation acceptable but scale mix matters",
    },
    {
        "item": "fixed STL seasonal/remainder ratio",
        "result": f"{stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('fixed'), 'seasonal_abs_mean'].iloc[0] / stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('fixed'), 'remainder_abs_mean'].iloc[0]:.3f}",
        "numeric_result": stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('fixed'), 'seasonal_abs_mean'].iloc[0] / stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('fixed'), 'remainder_abs_mean'].iloc[0],
        "decision": "seasonal component smaller than irregular",
    },
    {
        "item": "flexible STL seasonal/remainder ratio",
        "result": f"{stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('flexible'), 'seasonal_abs_mean'].iloc[0] / stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('flexible'), 'remainder_abs_mean'].iloc[0]:.3f}",
        "numeric_result": stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('flexible'), 'seasonal_abs_mean'].iloc[0] / stl_component_summary.loc[stl_component_summary['stl_model'].str.startswith('flexible'), 'remainder_abs_mean'].iloc[0],
        "decision": "no strong flexible seasonal mean",
    },
    {
        "item": "observed lag-52 ACF",
        "result": f"{signoff_source['observed_lag52_acf']:.3f}",
        "numeric_result": signoff_source["observed_lag52_acf"],
        "decision": "no clean annual point signal",
    },
    {
        "item": "Christmas/New Year std",
        "result": f"{signoff_source['christmas_std']:.3f}",
        "numeric_result": signoff_source["christmas_std"],
        "decision": "interval problem remains",
    },
])
seasonality_point_decision = pd.DataFrame([{
    "decision_line": "No additional point-seasonality layer.",
    "reason": "Weekly decomposition is sparse, flexible seasonality is weak, lag-52 ACF is flat, and Christmas/New Year is primarily high-variance.",
}])
display(seasonality_signoff_table[["item", "result", "decision"]])
display(seasonality_point_decision)

# 2. Finalize the stricter production guardrail: summer + Christmas/New Year only.
STRICT_POINT_LIFT_CORRIDORS = {"summer", "Christmas/New Year"}
strict_guardrail_predictions = r_final_validation.copy()
strict_guardrail_predictions["strict_guardrail_scale"] = np.where(
    strict_guardrail_predictions["release_corridor"].isin(STRICT_POINT_LIFT_CORRIDORS),
    1.0,
    0.0,
)
strict_guardrail_predictions["strict_predicted_residual"] = strict_guardrail_predictions["predicted_residual"] * strict_guardrail_predictions["strict_guardrail_scale"]
strict_guardrail_predictions["strict_pred_sunday_gross"] = strict_guardrail_predictions["pred_sunday_gross_base"] * np.exp(strict_guardrail_predictions["strict_predicted_residual"])
strict_guardrail_predictions["strict_sunday_error"] = strict_guardrail_predictions["sunday_gross_usd"] - strict_guardrail_predictions["strict_pred_sunday_gross"]
strict_guardrail_predictions["strict_abs_sunday_error"] = strict_guardrail_predictions["strict_sunday_error"].abs()
strict_guardrail_predictions["strict_corrected_residual"] = strict_guardrail_predictions["r_sun_sat"] - strict_guardrail_predictions["strict_predicted_residual"]
strict_guardrail_predictions["strict_interval_residual"] = np.log(strict_guardrail_predictions["strict_pred_sunday_gross"] / strict_guardrail_predictions["sunday_gross_usd"])
strict_guardrail_predictions["full_hybrid_pred_sunday_gross"] = strict_guardrail_predictions["pred_sunday_gross_base"] * np.exp(strict_guardrail_predictions["predicted_residual"])
strict_guardrail_predictions["full_hybrid_abs_sunday_error"] = (strict_guardrail_predictions["sunday_gross_usd"] - strict_guardrail_predictions["full_hybrid_pred_sunday_gross"]).abs()
strict_guardrail_predictions["full_hybrid_corrected_residual"] = strict_guardrail_predictions["r_sun_sat"] - strict_guardrail_predictions["predicted_residual"]

def summarize_strict_guardrail(df):
    rows = []
    for split_name, g in [("all", df), ("2022+", df.loc[df["release_year"].ge(2022)])]:
        base_log = g["r_sun_sat"].abs().mean()
        strict_log = g["strict_corrected_residual"].abs().mean()
        full_log = g["full_hybrid_corrected_residual"].abs().mean()
        base_dollar = g["abs_baseline_sunday_error"].mean()
        strict_dollar = g["strict_abs_sunday_error"].mean()
        full_dollar = g["full_hybrid_abs_sunday_error"].mean()
        rows.append({
            "split": split_name,
            "n": len(g),
            "baseline_log_mae": base_log,
            "strict_log_mae": strict_log,
            "full_hybrid_log_mae": full_log,
            "strict_log_improvement_pct": (base_log - strict_log) / base_log,
            "full_hybrid_log_improvement_pct": (base_log - full_log) / base_log,
            "baseline_dollar_mae": base_dollar,
            "strict_dollar_mae": strict_dollar,
            "full_hybrid_dollar_mae": full_dollar,
            "strict_dollar_improvement_pct": (base_dollar - strict_dollar) / base_dollar,
            "full_hybrid_dollar_improvement_pct": (base_dollar - full_dollar) / base_dollar,
        })
    return pd.DataFrame(rows)

strict_guardrail_summary = summarize_strict_guardrail(strict_guardrail_predictions)
display(strict_guardrail_summary)

strict_guardrail_corridor_summary = strict_guardrail_predictions.loc[strict_guardrail_predictions["release_year"].ge(2022)].groupby("release_corridor", observed=True).agg(
    n=("release_run_id", "count"),
    baseline_abs_sunday_error=("abs_baseline_sunday_error", "mean"),
    full_hybrid_abs_sunday_error=("full_hybrid_abs_sunday_error", "mean"),
    strict_abs_sunday_error=("strict_abs_sunday_error", "mean"),
    baseline_log_mae=("r_sun_sat", lambda s: s.abs().mean()),
    full_hybrid_log_mae=("full_hybrid_corrected_residual", lambda s: s.abs().mean()),
    strict_log_mae=("strict_corrected_residual", lambda s: s.abs().mean()),
).reset_index()
strict_guardrail_corridor_summary["full_hybrid_dollar_improvement_pct"] = (
    strict_guardrail_corridor_summary["baseline_abs_sunday_error"] - strict_guardrail_corridor_summary["full_hybrid_abs_sunday_error"]
) / strict_guardrail_corridor_summary["baseline_abs_sunday_error"]
strict_guardrail_corridor_summary["strict_dollar_improvement_pct"] = (
    strict_guardrail_corridor_summary["baseline_abs_sunday_error"] - strict_guardrail_corridor_summary["strict_abs_sunday_error"]
) / strict_guardrail_corridor_summary["baseline_abs_sunday_error"]
strict_guardrail_corridor_summary["full_hybrid_log_improvement_pct"] = (
    strict_guardrail_corridor_summary["baseline_log_mae"] - strict_guardrail_corridor_summary["full_hybrid_log_mae"]
) / strict_guardrail_corridor_summary["baseline_log_mae"]
strict_guardrail_corridor_summary["strict_log_improvement_pct"] = (
    strict_guardrail_corridor_summary["baseline_log_mae"] - strict_guardrail_corridor_summary["strict_log_mae"]
) / strict_guardrail_corridor_summary["baseline_log_mae"]
display(strict_guardrail_corridor_summary)

apr_may_priority_check = strict_guardrail_corridor_summary.loc[
    strict_guardrail_corridor_summary["release_corridor"].eq("Apr-May"),
    [
        "release_corridor", "n", "full_hybrid_log_improvement_pct", "full_hybrid_dollar_improvement_pct",
        "strict_log_improvement_pct", "strict_dollar_improvement_pct",
    ],
].copy()
apr_may_priority_check["decision"] = "exclude Apr-May from point lift; prioritize Sunday dollar MAE"
display(apr_may_priority_check)

# 3. Christmas/New Year interval widening. Keep point model fixed; tune only interval width.
FINAL_INTERVAL_LAMBDA = 35
FINAL_INTERVAL_ALPHAS = [0.025, 0.10, 0.25, 0.75, 0.90, 0.975]
FINAL_INTERVAL_NOMINALS = [0.50, 0.80, 0.95]
CHRISTMAS_INTERVAL_GROUP = "Christmas/New Year"

def interval_groups_for_strict(df):
    group = pd.Series("all", index=df.index, dtype="object")
    for corridor in ["summer", "Christmas/New Year", "Jan-Mar", "weak fall"]:
        group.loc[df["release_corridor"].eq(corridor)] = corridor
    group.loc[group.eq("all") & df["release_year"].ge(2022)] = "2022+"
    return group

def quantile_lookup_for_train(train, residual_col="strict_interval_residual", lam=FINAL_INTERVAL_LAMBDA):
    global_resid = train[residual_col].replace([np.inf, -np.inf], np.nan).dropna()
    global_q = global_resid.quantile(FINAL_INTERVAL_ALPHAS)
    rows = []
    groups = {
        "all": pd.Series(True, index=train.index),
        "2022+": train["release_year"].ge(2022),
        "summer": train["release_corridor"].eq("summer"),
        "Christmas/New Year": train["release_corridor"].eq("Christmas/New Year"),
        "Jan-Mar": train["release_corridor"].eq("Jan-Mar"),
        "weak fall": train["release_corridor"].eq("weak fall"),
    }
    for group_name, mask in groups.items():
        resid = train.loc[mask, residual_col].replace([np.inf, -np.inf], np.nan).dropna()
        n = len(resid)
        weight = n / (n + lam) if n else 0.0
        raw_q = resid.quantile(FINAL_INTERVAL_ALPHAS) if n else global_q.copy()
        for alpha in FINAL_INTERVAL_ALPHAS:
            raw = raw_q.loc[alpha] if alpha in raw_q.index else global_q.loc[alpha]
            rows.append({
                "interval_group": group_name,
                "alpha": alpha,
                "n": n,
                "lambda": lam,
                "shrink_weight": weight,
                "raw_quantile": raw,
                "global_quantile": global_q.loc[alpha],
                "shrunk_quantile": weight * raw + (1 - weight) * global_q.loc[alpha],
            })
    return pd.DataFrame(rows)

def score_intervals_with_multiplier(test, quantiles, christmas_multiplier=1.0, split_name="all", rolling_year=np.nan):
    scored_frames = []
    rows = []
    q_lookup = quantiles.set_index(["interval_group", "alpha"])["shrunk_quantile"]
    test = test.copy()
    test["interval_group"] = interval_groups_for_strict(test)
    for nominal in FINAL_INTERVAL_NOMINALS:
        alpha_low = round((1 - nominal) / 2, 3)
        alpha_high = round(1 - alpha_low, 3)
        tmp = test.copy()
        tmp["nominal"] = nominal
        tmp["christmas_multiplier"] = christmas_multiplier
        tmp["rolling_year"] = rolling_year
        fallback_low = float(q_lookup.loc[("all", alpha_low)])
        fallback_high = float(q_lookup.loc[("all", alpha_high)])
        tmp["q_low"] = [float(q_lookup.get((g, alpha_low), fallback_low)) for g in tmp["interval_group"]]
        tmp["q_high"] = [float(q_lookup.get((g, alpha_high), fallback_high)) for g in tmp["interval_group"]]
        christmas_mask = tmp["release_corridor"].eq(CHRISTMAS_INTERVAL_GROUP)
        tmp.loc[christmas_mask, "q_low"] = tmp.loc[christmas_mask, "q_low"] * christmas_multiplier
        tmp.loc[christmas_mask, "q_high"] = tmp.loc[christmas_mask, "q_high"] * christmas_multiplier
        # Residual is log(pred / actual), so convert residual quantiles back to actual-dollar bounds.
        tmp["interval_low_sunday"] = tmp["strict_pred_sunday_gross"] / np.exp(tmp["q_high"].astype(float))
        tmp["interval_high_sunday"] = tmp["strict_pred_sunday_gross"] / np.exp(tmp["q_low"].astype(float))
        tmp["interval_width_sunday"] = tmp["interval_high_sunday"] - tmp["interval_low_sunday"]
        tmp["covered"] = tmp["sunday_gross_usd"].between(tmp["interval_low_sunday"], tmp["interval_high_sunday"], inclusive="both")
        scored_frames.append(tmp)
        rows.append({
            "split": split_name,
            "rolling_year": rolling_year,
            "nominal": nominal,
            "n": len(tmp),
            "christmas_multiplier": christmas_multiplier,
            "empirical_coverage": tmp["covered"].mean(),
            "avg_interval_width_usd": tmp["interval_width_sunday"].mean(),
            "median_interval_width_usd": tmp["interval_width_sunday"].median(),
        })
    return pd.DataFrame(rows), pd.concat(scored_frames, ignore_index=True)

# Rolling validation: for each release year, use only prior years to estimate quantiles, then score Christmas/New Year rows.
rolling_multiplier_rows = []
rolling_multiplier_scored = []
christmas_years = sorted(strict_guardrail_predictions.loc[strict_guardrail_predictions["release_corridor"].eq(CHRISTMAS_INTERVAL_GROUP), "release_year"].dropna().unique())
for c in np.round(np.arange(1.0, 3.01, 0.05), 2):
    for year in christmas_years:
        train = strict_guardrail_predictions.loc[strict_guardrail_predictions["release_year"].lt(year)].copy()
        test = strict_guardrail_predictions.loc[
            strict_guardrail_predictions["release_year"].eq(year)
            & strict_guardrail_predictions["release_corridor"].eq(CHRISTMAS_INTERVAL_GROUP)
        ].copy()
        if len(train) < MIN_TRAIN_MOVIES or test.empty:
            continue
        q_train = quantile_lookup_for_train(train)
        rows, scored = score_intervals_with_multiplier(test, q_train, christmas_multiplier=c, split_name="Christmas/New Year rolling", rolling_year=year)
        rolling_multiplier_rows.append(rows)
        rolling_multiplier_scored.append(scored)

christmas_interval_multiplier_rolling_scores = pd.concat(rolling_multiplier_rows, ignore_index=True)
christmas_interval_multiplier_rolling_scored = pd.concat(rolling_multiplier_scored, ignore_index=True)
christmas_interval_multiplier_summary = christmas_interval_multiplier_rolling_scores.groupby(["christmas_multiplier", "nominal"], observed=True).agg(
    n=("n", "sum"),
    empirical_coverage=("empirical_coverage", lambda s: np.average(s, weights=christmas_interval_multiplier_rolling_scores.loc[s.index, "n"])),
    avg_interval_width_usd=("avg_interval_width_usd", lambda s: np.average(s, weights=christmas_interval_multiplier_rolling_scores.loc[s.index, "n"])),
    median_interval_width_usd=("median_interval_width_usd", "median"),
).reset_index()

pivot_cov = christmas_interval_multiplier_summary.pivot(index="christmas_multiplier", columns="nominal", values="empirical_coverage")
pivot_width = christmas_interval_multiplier_summary.pivot(index="christmas_multiplier", columns="nominal", values="avg_interval_width_usd")
eligible_c = pivot_cov.loc[(pivot_cov[0.80] >= 0.75) & (pivot_cov[0.95] >= 0.92)].index
if len(eligible_c):
    selected_christmas_interval_multiplier = float(eligible_c.min())
    selection_rule = "smallest multiplier meeting 80%>=75% and 95%>=92% in rolling validation"
else:
    shortfall = (0.75 - pivot_cov[0.80]).clip(lower=0) + (0.92 - pivot_cov[0.95]).clip(lower=0)
    selected_christmas_interval_multiplier = float(shortfall.sort_values().index[0])
    selection_rule = "no multiplier met both thresholds; selected smallest total coverage shortfall"

christmas_interval_multiplier_selection = pd.DataFrame([{
    "selected_christmas_multiplier": selected_christmas_interval_multiplier,
    "selection_rule": selection_rule,
    "rolling_80_coverage": pivot_cov.loc[selected_christmas_interval_multiplier, 0.80],
    "rolling_95_coverage": pivot_cov.loc[selected_christmas_interval_multiplier, 0.95],
    "rolling_80_avg_width_usd": pivot_width.loc[selected_christmas_interval_multiplier, 0.80],
    "rolling_95_avg_width_usd": pivot_width.loc[selected_christmas_interval_multiplier, 0.95],
}])
display(christmas_interval_multiplier_selection)

def score_named_split(df, quantiles, multiplier, split_name, mask):
    rows, scored = score_intervals_with_multiplier(df.loc[mask].copy(), quantiles, christmas_multiplier=multiplier, split_name=split_name)
    return rows, scored

final_strict_interval_quantiles = quantile_lookup_for_train(strict_guardrail_predictions)
final_interval_rows = []
final_interval_scored_frames = []
final_split_defs = {
    "all": pd.Series(True, index=strict_guardrail_predictions.index),
    "2022+": strict_guardrail_predictions["release_year"].ge(2022),
    "summer": strict_guardrail_predictions["release_corridor"].eq("summer"),
    "Christmas/New Year": strict_guardrail_predictions["release_corridor"].eq("Christmas/New Year"),
    "Jan-Mar": strict_guardrail_predictions["release_corridor"].eq("Jan-Mar"),
    "weak fall": strict_guardrail_predictions["release_corridor"].eq("weak fall"),
}
for split_name, mask in final_split_defs.items():
    rows, scored = score_named_split(strict_guardrail_predictions, final_strict_interval_quantiles, selected_christmas_interval_multiplier, split_name, mask)
    final_interval_rows.append(rows)
    final_interval_scored_frames.append(scored)
final_strict_interval_coverage = pd.concat(final_interval_rows, ignore_index=True)
final_strict_interval_scored = pd.concat(final_interval_scored_frames, ignore_index=True)
display(final_strict_interval_coverage)

christmas_interval_multiplier_summary_selected_window = christmas_interval_multiplier_summary.loc[
    christmas_interval_multiplier_summary["christmas_multiplier"].between(
        max(1.0, selected_christmas_interval_multiplier - 0.20),
        selected_christmas_interval_multiplier + 0.20,
    )
].copy()
display(christmas_interval_multiplier_summary_selected_window)


## Promotion Decision

A point predictor should enter only if it:

1. improves log MAE at least 5% versus the frozen baseline;
2. improves meaningfully versus intercept-only;
3. does not hurt daily dollar MAE;
4. survives recent-period validation and leave-one-year checks;
5. does not create residual bias or interval degradation;
6. is known at forecast origin without leakage.

This notebook tests the category-aware shrinkage prerequisite before continuous Wiki/critic/competition point corrections.

In [ ]:
promotion_table = category_summary[[
    "shrinkage_model", "n", "group_col", "lambda", "logratio_mae_improvement_pct",
    "daily_mae_improvement_pct", "recent_2022_logratio_improvement_pct",
    "leave_one_year_min_improvement_pct", "leave_one_year_positive_share",
    "beats_baseline_5pct", "beats_intercept", "survives_2022_validation",
    "survives_leave_one_year", "promotion_candidate",
    "mean_corrected_e_a", "mean_corrected_e_b",
]].copy()
promotion_table["decision"] = np.where(
    promotion_table["promotion_candidate"],
    "candidate, still requires manual plot review",
    "do not promote yet",
)

display(promotion_table.head(30))
print("Current continuous predictor models remain rejected for point forecasts; evaluate them later only as interactions or interval-width features if residual plots justify it.")

rolling_calendar_decision = rolling_calendar_summary[[
    "target", "rolling_calendar_model", "n", "mae_improvement_pct", "recent_2022_improvement_pct",
    "leave_one_year_min_improvement_pct", "passes_5pct", "survives_2022",
    "survives_leave_one_year", "promotion_candidate", "corrected_bias",
]].copy()
rolling_calendar_decision["decision"] = np.where(
    rolling_calendar_decision["promotion_candidate"],
    "candidate, still requires manual plot review",
    "do not promote yet",
)

display(rolling_calendar_decision.groupby("target", observed=True).head(10))

current_regime_decision = current_regime_summary[[
    "target", "current_regime_model", "n", "post_2022_n", "mae_improvement_pct",
    "post_2022_mae_improvement_pct", "corridor_year_positive_share",
    "corridor_year_min_improvement_pct", "passes_overall_5pct", "passes_post_2022_2pct",
    "no_corridor_year_catastrophic_miss", "corridor_year_hit_rate_60pct",
    "passes_screen", "promotion_candidate", "decision",
]].copy()
current_regime_decision["candidate_stage"] = current_regime_decision["target"].isin(["e_b", "r_sun_sat"])
current_regime_decision.loc[current_regime_decision["target"].eq("e_a"), "decision"] = "keep experimental"
current_regime_decision.loc[current_regime_decision["candidate_stage"], "decision"] = "candidate-stage only; requires seasonal and dollar validation"
display(current_regime_decision)

seasonal_decision = seasonal_addon_summary[[
    "target", "current_regime_model", "seasonal_model", "seasonal_addon_improvement_pct",
    "post_2022_seasonal_addon_improvement_pct", "post_2022_year_min_addon_improvement_pct",
    "corridor_year_positive_share", "candidate_stage", "promotion_candidate", "decision",
]].copy()
display(seasonal_decision.groupby("target", observed=True).head(6))

display(current_regime_target_performance)
display(seasonal_target_performance.groupby("target", observed=True).head(6))

display(training_window_validation_summary)
print("Current decision focus: choose training history, not new predictors. r_sun_sat remains the cleanest candidate; e_b likely needs hybrid shrinkage; e_a remains no point correction because operational dollar MAE is not improved.")
print("Use the regime/corridor, movie-mix, composition, share-error, and training-window tables above to decide all-prior vs 2022+ only vs hybrid shrinkage.")


## Export Outputs

In [ ]:
residual_export = DIAGNOSTICS_DIR / "weekend_shape_residual_diagnostic_table.csv"
acf_export = DIAGNOSTICS_DIR / "weekend_shape_weekly_residual_acf.csv"
ljung_export = DIAGNOSTICS_DIR / "weekend_shape_weekly_residual_ljung_box.csv"
binned_export = DIAGNOSTICS_DIR / "weekend_shape_feature_binned_residual_summary.csv"
category_summary_export = DIAGNOSTICS_DIR / "weekend_shape_category_shrinkage_summary.csv"
category_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_category_shrinkage_predictions.csv"
interval_export = DIAGNOSTICS_DIR / "weekend_shape_group_interval_quantiles.csv"
month_year_export = DIAGNOSTICS_DIR / "weekend_shape_month_year_residual_summary.csv"
intercept_summary_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_intercept_summary.csv"
intercept_acf_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_intercept_acf.csv"
intercept_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_intercept_ljung_box.csv"
rolling_calendar_summary_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_calendar_shrinkage_summary.csv"
rolling_calendar_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_calendar_shrinkage_predictions.csv"
calendar_acf_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_calendar_corrected_acf.csv"
calendar_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_rolling_calendar_corrected_ljung_box.csv"
post_2022_corridor_export = DIAGNOSTICS_DIR / "weekend_shape_post_2022_corridor_residual_summary.csv"
post_2022_corridor_movie_export = DIAGNOSTICS_DIR / "weekend_shape_post_2022_corridor_movie_residual_summary.csv"
share_regime_export = DIAGNOSTICS_DIR / "weekend_shape_share_regime_summary.csv"
current_regime_summary_export = DIAGNOSTICS_DIR / "weekend_shape_current_regime_screen_summary.csv"
current_regime_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_current_regime_screen_predictions.csv"
current_regime_acf_export = DIAGNOSTICS_DIR / "weekend_shape_current_regime_corrected_acf.csv"
current_regime_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_current_regime_corrected_ljung_box.csv"
current_regime_target_performance_export = DIAGNOSTICS_DIR / "weekend_shape_current_regime_target_performance.csv"
seasonal_addon_summary_export = DIAGNOSTICS_DIR / "weekend_shape_seasonal_addon_summary.csv"
seasonal_addon_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_seasonal_addon_predictions.csv"
seasonal_target_performance_export = DIAGNOSTICS_DIR / "weekend_shape_seasonal_target_performance.csv"
seasonal_acf_export = DIAGNOSTICS_DIR / "weekend_shape_seasonal_corrected_acf.csv"
seasonal_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_seasonal_corrected_ljung_box.csv"
residual_regime_summary_export = DIAGNOSTICS_DIR / "weekend_shape_residual_regime_summary.csv"
corridor_regime_summary_export = DIAGNOSTICS_DIR / "weekend_shape_corridor_regime_summary.csv"
corridor_year_summary_export = DIAGNOSTICS_DIR / "weekend_shape_corridor_year_summary.csv"
training_window_correction_estimates_export = DIAGNOSTICS_DIR / "weekend_shape_training_window_correction_estimates.csv"
post_2022_corridor_movie_diag_export = DIAGNOSTICS_DIR / "weekend_shape_post_2022_corridor_movie_diag_summary.csv"
composition_shift_export = DIAGNOSTICS_DIR / "weekend_shape_composition_shift_summary.csv"
training_window_sufficiency_export = DIAGNOSTICS_DIR / "weekend_shape_training_window_sufficiency.csv"
correction_stability_export = DIAGNOSTICS_DIR / "weekend_shape_correction_stability.csv"
share_regime_actual_export = DIAGNOSTICS_DIR / "weekend_shape_actual_share_regime_summary.csv"
share_error_regime_export = DIAGNOSTICS_DIR / "weekend_shape_share_error_regime_summary.csv"
diagnostic_acf_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_diagnostic_acf_ljung_summary.csv"
selected_acf_summary_export = DIAGNOSTICS_DIR / "weekend_shape_selected_acf_summary.csv"
seasonal_lag_scatter_export = DIAGNOSTICS_DIR / "weekend_shape_lag52_scatter_summary.csv"
training_window_validation_summary_export = DIAGNOSTICS_DIR / "weekend_shape_training_window_validation_summary.csv"
training_window_validation_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_training_window_validation_predictions.csv"
production_yearly_improvement_export = DIAGNOSTICS_DIR / "weekend_shape_production_yearly_improvement.csv"
production_eb_corridor_share_error_export = DIAGNOSTICS_DIR / "weekend_shape_production_eb_corridor_share_error.csv"
production_r_corridor_dollar_error_export = DIAGNOSTICS_DIR / "weekend_shape_production_r_corridor_dollar_error.csv"
production_error_distribution_export = DIAGNOSTICS_DIR / "weekend_shape_production_error_distribution_stats.csv"
production_interval_coverage_export = DIAGNOSTICS_DIR / "weekend_shape_production_interval_coverage.csv"
production_decision_summary_export = DIAGNOSTICS_DIR / "weekend_shape_production_decision_summary.csv"
frozen_point_stack_summary_export = DIAGNOSTICS_DIR / "weekend_shape_frozen_point_stack_summary.csv"
r_interval_quantiles_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_interval_quantiles.csv"
r_interval_coverage_recalibrated_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_interval_coverage_recalibrated.csv"
r_interval_scored_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_interval_scored.csv"
guardrail_summary_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_guardrail_summary.csv"
guardrail_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_guardrail_predictions.csv"
guardrail_corridor_summary_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_guardrail_corridor_summary.csv"
final_weekly_residual_audit_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_weekly_residuals.csv"
final_weekly_count_summary_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_weekly_count_summary.csv"
final_seasonal_weekly_era_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_week_of_year_era.csv"
final_corridor_residual_summary_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_corridor_summary.csv"
final_subseries_corridor_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_corridor_year_subseries.csv"
final_stl_decomposition_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_stl_decomposition.csv"
final_stl_component_summary_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_stl_component_summary.csv"
final_seasonally_adjusted_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_adjusted_residuals.csv"
final_seasonality_acf_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_acf.csv"
final_seasonality_ljung_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_ljung_box.csv"
final_seasonality_decision_export = DIAGNOSTICS_DIR / "weekend_shape_final_seasonality_decision_summary.csv"
seasonality_signoff_export = DIAGNOSTICS_DIR / "weekend_shape_seasonality_signoff_table.csv"
seasonality_point_decision_export = DIAGNOSTICS_DIR / "weekend_shape_seasonality_point_decision.csv"
strict_guardrail_summary_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_strict_guardrail_summary.csv"
strict_guardrail_corridor_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_strict_guardrail_corridor_summary.csv"
apr_may_priority_check_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_apr_may_priority_check.csv"
strict_guardrail_predictions_export = DIAGNOSTICS_DIR / "weekend_shape_r_sun_sat_strict_guardrail_predictions.csv"
christmas_interval_multiplier_summary_export = DIAGNOSTICS_DIR / "weekend_shape_christmas_interval_multiplier_summary.csv"
christmas_interval_multiplier_selection_export = DIAGNOSTICS_DIR / "weekend_shape_christmas_interval_multiplier_selection.csv"
christmas_interval_multiplier_rolling_scores_export = DIAGNOSTICS_DIR / "weekend_shape_christmas_interval_multiplier_rolling_scores.csv"
christmas_interval_multiplier_rolling_scored_export = DIAGNOSTICS_DIR / "weekend_shape_christmas_interval_multiplier_rolling_scored.csv"
final_strict_interval_quantiles_export = DIAGNOSTICS_DIR / "weekend_shape_final_strict_interval_quantiles.csv"
final_strict_interval_coverage_export = DIAGNOSTICS_DIR / "weekend_shape_final_strict_interval_coverage.csv"
final_strict_interval_scored_export = DIAGNOSTICS_DIR / "weekend_shape_final_strict_interval_scored.csv"

residual_table.to_csv(residual_export, index=False)
acf_table.to_csv(acf_export, index=False)
ljung_box_table.to_csv(ljung_export, index=False)
feature_binned_summary.to_csv(binned_export, index=False)
category_summary.to_csv(category_summary_export, index=False)
category_predictions[[
    "release_run_id", "title", "opening_weekend_start", "release_year", "shrinkage_model", "group_col", "lambda",
    "e_a", "e_b", "pred_e_a", "pred_e_b", "corrected_e_a", "corrected_e_b",
    "avg_abs_daily_dollar_error_shape_only", "new_avg_abs_daily_dollar_error_shape_only", "test_year",
]].to_csv(category_predictions_export, index=False)
interval_group_quantiles.to_csv(interval_export, index=False)
month_year_residual_summary.to_csv(month_year_export, index=False)
intercept_correction_summary.to_csv(intercept_summary_export, index=False)
intercept_acf_table.to_csv(intercept_acf_export, index=False)
intercept_ljung_box_table.to_csv(intercept_ljung_export, index=False)
rolling_calendar_summary.to_csv(rolling_calendar_summary_export, index=False)
rolling_calendar_predictions.to_csv(rolling_calendar_predictions_export, index=False)
calendar_acf_table.to_csv(calendar_acf_export, index=False)
calendar_ljung_box_table.to_csv(calendar_ljung_export, index=False)
post_2022_corridor_residual_summary.to_csv(post_2022_corridor_export, index=False)
post_2022_corridor_movie_residual_summary.to_csv(post_2022_corridor_movie_export, index=False)
share_regime_summary.to_csv(share_regime_export, index=False)
current_regime_summary.to_csv(current_regime_summary_export, index=False)
current_regime_predictions.to_csv(current_regime_predictions_export, index=False)
current_regime_acf_table.to_csv(current_regime_acf_export, index=False)
current_regime_ljung_box_table.to_csv(current_regime_ljung_export, index=False)
current_regime_target_performance.to_csv(current_regime_target_performance_export, index=False)
seasonal_addon_summary.to_csv(seasonal_addon_summary_export, index=False)
seasonal_addon_predictions.to_csv(seasonal_addon_predictions_export, index=False)
seasonal_target_performance.to_csv(seasonal_target_performance_export, index=False)
seasonal_acf_table.to_csv(seasonal_acf_export, index=False)
seasonal_ljung_box_table.to_csv(seasonal_ljung_export, index=False)
residual_regime_summary.to_csv(residual_regime_summary_export, index=False)
corridor_regime_summary.to_csv(corridor_regime_summary_export, index=False)
corridor_year_summary.to_csv(corridor_year_summary_export, index=False)
training_window_correction_estimates.to_csv(training_window_correction_estimates_export, index=False)
post_2022_corridor_movie_summary.to_csv(post_2022_corridor_movie_diag_export, index=False)
composition_shift_table.to_csv(composition_shift_export, index=False)
training_window_sufficiency_table.to_csv(training_window_sufficiency_export, index=False)
correction_stability_table.to_csv(correction_stability_export, index=False)
share_regime_actual_long.to_csv(share_regime_actual_export, index=False)
share_error_regime_summary.to_csv(share_error_regime_export, index=False)
diagnostic_acf_ljung_summary.to_csv(diagnostic_acf_ljung_export, index=False)
selected_acf_summary.to_csv(selected_acf_summary_export, index=False)
seasonal_lag_scatter_summary.to_csv(seasonal_lag_scatter_export, index=False)
training_window_validation_summary.to_csv(training_window_validation_summary_export, index=False)
training_window_validation_predictions.to_csv(training_window_validation_predictions_export, index=False)
yearly_production_improvement.to_csv(production_yearly_improvement_export, index=False)
eb_corridor_sunday_share_error.to_csv(production_eb_corridor_share_error_export, index=False)
r_corridor_sunday_dollar_error.to_csv(production_r_corridor_dollar_error_export, index=False)
production_error_distribution_stats.to_csv(production_error_distribution_export, index=False)
production_interval_coverage.to_csv(production_interval_coverage_export, index=False)
production_validation_decision_summary.to_csv(production_decision_summary_export, index=False)
frozen_point_stack_summary.to_csv(frozen_point_stack_summary_export, index=False)
r_interval_quantiles.to_csv(r_interval_quantiles_export, index=False)
r_interval_coverage_recalibrated.to_csv(r_interval_coverage_recalibrated_export, index=False)
r_interval_scored.to_csv(r_interval_scored_export, index=False)
guardrail_summary.to_csv(guardrail_summary_export, index=False)
guardrail_predictions.to_csv(guardrail_predictions_export, index=False)
guardrail_corridor_summary.to_csv(guardrail_corridor_summary_export, index=False)
weekly_residual_audit_table.to_csv(final_weekly_residual_audit_export, index=False)
weekly_count_summary.to_csv(final_weekly_count_summary_export, index=False)
seasonal_weekly_era.to_csv(final_seasonal_weekly_era_export, index=False)
final_corridor_residual_summary.to_csv(final_corridor_residual_summary_export, index=False)
seasonal_subseries_corridor.to_csv(final_subseries_corridor_export, index=False)
stl_decomposition_table.to_csv(final_stl_decomposition_export, index=False)
stl_component_summary.to_csv(final_stl_component_summary_export, index=False)
seasonally_adjusted_residual_table.to_csv(final_seasonally_adjusted_export, index=False)
final_seasonality_acf.to_csv(final_seasonality_acf_export, index=False)
final_seasonality_ljung.to_csv(final_seasonality_ljung_export, index=False)
final_seasonality_decision_summary.to_csv(final_seasonality_decision_export, index=False)
seasonality_signoff_table.to_csv(seasonality_signoff_export, index=False)
seasonality_point_decision.to_csv(seasonality_point_decision_export, index=False)
strict_guardrail_summary.to_csv(strict_guardrail_summary_export, index=False)
strict_guardrail_corridor_summary.to_csv(strict_guardrail_corridor_export, index=False)
apr_may_priority_check.to_csv(apr_may_priority_check_export, index=False)
strict_guardrail_predictions.to_csv(strict_guardrail_predictions_export, index=False)
christmas_interval_multiplier_summary.to_csv(christmas_interval_multiplier_summary_export, index=False)
christmas_interval_multiplier_selection.to_csv(christmas_interval_multiplier_selection_export, index=False)
christmas_interval_multiplier_rolling_scores.to_csv(christmas_interval_multiplier_rolling_scores_export, index=False)
christmas_interval_multiplier_rolling_scored.to_csv(christmas_interval_multiplier_rolling_scored_export, index=False)
final_strict_interval_quantiles.to_csv(final_strict_interval_quantiles_export, index=False)
final_strict_interval_coverage.to_csv(final_strict_interval_coverage_export, index=False)
final_strict_interval_scored.to_csv(final_strict_interval_scored_export, index=False)

export_paths = [
    residual_export, acf_export, ljung_export, binned_export, category_summary_export,
    category_predictions_export, interval_export, month_year_export, intercept_summary_export,
    intercept_acf_export, intercept_ljung_export, rolling_calendar_summary_export,
    rolling_calendar_predictions_export, calendar_acf_export, calendar_ljung_export,
    post_2022_corridor_export, post_2022_corridor_movie_export, share_regime_export,
    current_regime_summary_export, current_regime_predictions_export,
    current_regime_acf_export, current_regime_ljung_export,
    current_regime_target_performance_export, seasonal_addon_summary_export,
    seasonal_addon_predictions_export, seasonal_target_performance_export,
    seasonal_acf_export, seasonal_ljung_export,
    residual_regime_summary_export, corridor_regime_summary_export,
    corridor_year_summary_export, training_window_correction_estimates_export,
    post_2022_corridor_movie_diag_export, composition_shift_export,
    training_window_sufficiency_export, correction_stability_export,
    share_regime_actual_export, share_error_regime_export,
    diagnostic_acf_ljung_export, selected_acf_summary_export,
    seasonal_lag_scatter_export, training_window_validation_summary_export,
    training_window_validation_predictions_export, production_yearly_improvement_export,
    production_eb_corridor_share_error_export, production_r_corridor_dollar_error_export,
    production_error_distribution_export, production_interval_coverage_export,
    production_decision_summary_export, frozen_point_stack_summary_export,
    r_interval_quantiles_export, r_interval_coverage_recalibrated_export,
    r_interval_scored_export, guardrail_summary_export,
    guardrail_predictions_export, guardrail_corridor_summary_export,
    final_weekly_residual_audit_export, final_weekly_count_summary_export,
    final_seasonal_weekly_era_export, final_corridor_residual_summary_export,
    final_subseries_corridor_export, final_stl_decomposition_export,
    final_stl_component_summary_export, final_seasonally_adjusted_export,
    final_seasonality_acf_export, final_seasonality_ljung_export,
    final_seasonality_decision_export,
    seasonality_signoff_export, seasonality_point_decision_export,
    strict_guardrail_summary_export, strict_guardrail_corridor_export,
    apr_may_priority_check_export, strict_guardrail_predictions_export,
    christmas_interval_multiplier_summary_export, christmas_interval_multiplier_selection_export,
    christmas_interval_multiplier_rolling_scores_export, christmas_interval_multiplier_rolling_scored_export,
    final_strict_interval_quantiles_export, final_strict_interval_coverage_export,
    final_strict_interval_scored_export,
]
display(pd.DataFrame({
    "artifact": [p.name for p in export_paths],
    "path": [str(p) for p in export_paths],
}))
